# `Importing Libraries`


In [ ]:
# 🔥🔥 Check whether the list of PIDs remain same  AFTER updating the condition class_counts[(class_counts['class1'] > 0) & (class_counts['class0'] > 0)].copy()

from classification_models.keras import Classifiers
from collections import Counter
from datetime import datetime
import gc, gzip
from IPython.display import display, HTML
import matplotlib.pyplot as plt
from itertools import combinations
import numpy as np
import os

import platform
if platform.system() == 'Darwin': # That means Mac where I am converting the sensing data to an image
    from complexity_rqa import hrv_rqa

import pathlib
import pickle
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import pytz

import sys
# sys.path.append('... removed location for better privacy ...CommonCodes/') from f_scores import F1Score # F1Score of tensorflow_addons

import keras
from scipy.spatial.distance import jensenshannon
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, IsolationForest, GradientBoostingClassifier, AdaBoostClassifier, HistGradientBoostingClassifier, ExtraTreesClassifier, VotingClassifier, StackingClassifier
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, balanced_accuracy_score
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score

from tensorflow.keras import backend as K
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_io as tfio

from importlib.metadata import version # Version for reproducibility. April 10, 2025: tensorflow 2.17.1, keras:  3.7.0, scikit-learn 1.5.2
print('tensorflow', version('tensorflow'))
print('keras: ', version('keras'))
print('scikit-learn', version('scikit-learn'))

# `Strings & Locations`

In [ ]:
str_age = 'age'
str_anxiety = 'anxiety'
str_actual_class = 'Actual_class'
str_best_epoch = 'best_epoch'
str_difference = 'difference'

str_capital_l_level = 'Level' # :) strange name. This is for the column name of battery data in sapiens
str_category = 'category'
str_daily_anxiety = 'Daily Anxiety'
str_dataset = 'Dataset'
str_date = 'date'
str_dateTime = 'dateTime'
str_demograpnic_info = 'Demo_info'
str_device_id = 'DeviceID'
str_epoch = 'epoch'

str_features = 'Features'
str_filename = 'filename'
str_file_model_only_new_layers = 'model_new_layers.keras'
str_file_only_new_layers_predict_proba = 'predicted_proba_training_only_new_layers.xlsx'
str_file_predict_prob_fine_tuned_resnet = 'predicted_proba_fine_tuned_ResNet.xlsx'

str_hr = 'HR'
str_hour = 'Hour'
str_hrv = 'HRV'
HOUR_in_Sec = 60 * 60

str_light = 'Light'
str_median = 'median'
str_model_run_start_time = 'model_runing_start_time'
str_models = 'Models'
str_multi_modal = 'multi_modal'
str_magnitude = 'magnitude'

str_n_th_iter_of_train = 'n_th_iteration_of_training'
str_one_day_in_advance = 'One_day_in_advance'
str_on_the_same_day = 'on_the_same_day'

str_raw_sensor_data = 'Raw sensor data'
str_segment = 'segment'
str_predicted_proba = 'Predicted_probability'
str_predicted_class = 'Predicted_class'
str_p_id = 'pid'
str_performance = 'Performance'
str_rqa = 'RQA'

str_start_ts = 'start_ts'
str_sensus_submission_time = 'SubmissionTimestampUtc'
str_single_modal = 'single_modal'
str_splitted_into = 'splitted_into'
str_spectrogram = 'Spectrogram'
str_sapiens = 'SAPIENS'
str_state_anxiety = 'State Anxiety'
str_completed_ts = 'completed_ts'
str_count = 'Count'

str_tiles_18 = 'Tiles18'
str_timestamp = 'Timestamp'
str_test_fold = 'test_fold'
str_tiles_pid = 'participant_id'
str_us_eastern = 'US/Eastern'
str_us_pacific = 'US/Pacific'
str_weights_file = 'weights_file'

## Evaluation Metrics ###
str_f1_score = 'f1_score'
str_balanced_acc = 'bal_acc'
str_loss = 'loss'
str_val_loss = 'val_loss'
str_train_loss = 'train_loss'

## Sensors
str_sensor_battery = 'Smartwatch_BatteryDatum'
str_sensor_light = 'Smartwatch_LightDatum'
str_sensor_gravity = 'Smartwatch_GravityDatum'
str_sensor_heart_rate = 'Smartwatch_HeartRateDatum'
str_sensor_magnetometer = 'Smartwatch_MagnetometerDatum'
str_sensor_step = 'Smartwatch_StepCountDatum'
str_sensor_lin_acc = 'Smartwatch_LinearAccelerationDatum'

str_tiles_sensor_heart_rate = 'heart-rate'
str_tiles_sensor_step_count = 'step-count'
str_tiles_sensor_sleep_data = 'sleep-data'
list_sleep_stages = ['wake', 'rem', 'light', 'deep']
dict_sleep_stage_encode = dict(zip(list_sleep_stages, [1, 2, 3, 4]))
dict_tiles_sensor_timestamp_col = {str_tiles_sensor_heart_rate: str_timestamp, str_tiles_sensor_step_count: str_timestamp, str_tiles_sensor_sleep_data: str_dateTime}

str_HeartRatePPG = 'HeartRatePPG' # column name of tiles hr data
str_level = 'level' # column name for stages of sleep-data sensor data
str_seconds = 'seconds' # column name for duration of stages for sleep-data sensor data
str_StepCount = 'StepCount' # column name for step-count sensor data

str_1_vs_2_10_split = '1_vs_2_10_split'

# This is not used anywhere currently. However, it is used to get the data column name for a corresponding sensor to be assigned in the sensor_data_col variable.
dic_sensor_data_name = {'Smartwatch_BatteryDatum': [str_level], # 🔥🔥🔥🔥 Do NOT add any more items. It will cause severe error inside sensor_fusion_in_input_for_sapiens() where we are using the first data dic_sensor_data_name.get(sensor_comb[0])[0]
                     'Smartwatch_GravityDatum': [str_magnitude],
                     'Smartwatch_HeartRateDatum': [str_hr],
                     'Smartwatch_LinearAccelerationDatum': [str_magnitude], # we have data in x, y, and z axes as well
                     'Smartwatch_LightDatum': [str_light],
                     'Smartwatch_MagnetometerDatum': [str_magnitude],
                     'Smartwatch_OffbodyDatum': ['Offboddy'],
                     'Smartwatch_StepCountDatum': [str_count]}

list_IMU_sensor = ['Smartwatch_LinearAccelerationDatum', 'Smartwatch_MagnetometerDatum', 'Smartwatch_GravityDatum']
list_sensor_high_freq = ['Smartwatch_LightDatum'] # sampling frequency more than 1
list_sensor_high_freq.extend(list_IMU_sensor)

list_sensor_sampling_rate_1 = ['Smartwatch_BatteryDatum', 'Smartwatch_StepCountDatum']
list_sensor_relatively_LOWER_sampling_rate = ['Smartwatch_MagnetometerDatum', 'Smartwatch_LightDatum']
list_tiles_sensor_with_low_sampling = [str_tiles_sensor_sleep_data]

dict_sensor_sampling_rate = {# 'Smartwatch_GravityDatum': 33, # we received more samples than what we are supposed. Thus, set the sampling rate based on what we received.
                             'Smartwatch_HeartRateDatum': 1,
                             'Smartwatch_LightDatum': 5,
                             'Smartwatch_LinearAccelerationDatum': 33,
                             'Smartwatch_MagnetometerDatum': 6,
                             #  'Smartwatch_PPGDatum': 25,
                             'Smartwatch_StepCountDatum': 1} # it is basically depending on the chage of number of steps (so that the system consumes less resources). Thus, if there is no change in the number of steps since the last probe starting times, we will not receive any data.

list_sensors_combination = list(combinations(dict_sensor_sampling_rate.keys(), 4))

def get_fused_sensor_name(sensor_comb):
    fused_name = '_'.join([raw_sensor_name.replace(str_sensor_lin_acc, '_accel_').replace(str_sensor_magnetometer, '_magnet_').replace('Smartwatch_', '').replace('Datum', '')
                   for raw_sensor_name in sensor_comb])
    return str_sapiens +'_'+ fused_name

# Control Center

In [ ]:
######## Control Center - Start #######
bool_runing_first_time_after_restart = True # Using this for reproducibility. The model will stop running after a pre-defined number of iterations so that same process is used for each model development. But in titan which has a sufficient GPU memory and does not have memory leakage (unlike Mac), we do not need this much. But still, to make sure that in each iteration, we have exactly same configuration, we need this checker.
resnet_version = 18
base_model = 'ResNet_'+str(resnet_version)
current_day_status = str_on_the_same_day  # Will only be used if crt_anxiety_type == str_daily_anxiety
current_ds = str_sapiens
crt_anxiety_type = str_state_anxiety # crt: current; str_daily_anxiety is used to train the teacher model where we are using the data of the whole day (i.e., data from the start of the day upto before the survey response.).
crt_state_anxiety_classification_approach = str_1_vs_2_10_split # either median or str_1_vs_2_10_split. Remember, based on this variable's value, the target variable will be created in TILES regardless whether you are working on daily or state anxiety. In SAPIENS, this will only be used if you work on state anxiety
appended_conditions_to_be_matched = ''

window_size_in_HOUR = 2.5 # Will only be used if crt_anxiety_type == str_state_anxiety
n_segments_of_a_day = 3 # will be used regardless of anything else. a day's data will be divided into n segments and each of them will be labelled the same as the label of the whole day

save_fig_for_paper = False
str_modal_type = str_multi_modal # if you prepare image data for multi-modal system, keep it str_multi_modal. But if you develop a model solely using a sensor, name that sensor here
number_of_modals = 2 # 2: This will help us to add 4 modalities {(112 * 112) * 4} at a time. Change more than 2, only if you want to fuse sensors more than 4 in a single image :).
crt_sensor = str_sensor_gravity # str_multi_modal for multi-modal # WARNING: do not append .pkl at the end of the sensor name
sensor_data_col = str_magnitude # str_multi_modal if you develop multi-modal model; sensor_data_col is the column  name (e.g., column name of heart rate data in Tiles-18) of the data file. # check the data column available in dic_data_col_name to get the data column for the corresponding sensor 
top_n_teacher_models_for_SAPIENS = 3 # for SAPIENS data based model: will be used to get top n models based on b. accuracy after training only new layers and also top n ... fine tuning all layers
average_approach_for_ev_metric = 'weighted'
######## Control Center - End #######

if platform.system() == 'Darwin': # Mac == Darwin 🤣🤣🤣
    temp_loc = '... removed the location a bit for privacy .../Documents/SA39/'
elif platform.system() == 'Windows':
    temp_loc = 'C:... removed the location a bit for privacy .../sa39/'
else: # Ubuntu currently
    temp_loc = '/home/sabbir/Documents/sa39/'

loc_tiles_18 = os.path.join(temp_loc, 'Hridita/Data/Tiles18/')
root_loc = os.path.join(temp_loc, 'SAPIENS/')

if crt_anxiety_type == str_daily_anxiety:   
    loc_root_tl = os.path.join(root_loc, crt_anxiety_type, current_day_status)
elif crt_anxiety_type == str_state_anxiety:
    loc_root_tl = os.path.join(root_loc, crt_anxiety_type, str(window_size_in_HOUR)+ '_hours_'+crt_state_anxiety_classification_approach)

loc_root_tl = os.path.join(loc_root_tl, str_modal_type, str_splitted_into + str(n_segments_of_a_day), # do not rename splitted_into to segment. Otherwise, when we will work (inside evalulate_model() function) to have file name without the segment name, the root folder (e.g., segment_3 in case you do not keep splitted_into_3) will also be replaced by ''
                           current_ds+'_'+crt_sensor.replace('Smartwatch_', '').replace('Datum', '') +'_'+ sensor_data_col)
if (crt_sensor == str_multi_modal) and (current_ds == str_sapiens):
    name_multi_modal_sensor_comb = get_fused_sensor_name(list_sensors_combination[0])
    print(name_multi_modal_sensor_comb)
    loc_root_tl = os.path.join(pathlib.Path(loc_root_tl).parent, name_multi_modal_sensor_comb)


if n_segments_of_a_day == 3:
    loc_teacher = os.path.join(root_loc, 'Daily Anxiety/on_the_same_day/multi_modal/splitted_into3/Tiles18_multi_modal_multi_modal') # for SAPIENS data based model: Will be used to get the models for SAPIENS
elif n_segments_of_a_day == 2:
    loc_teacher = os.path.join(root_loc, 'Daily Anxiety/on_the_same_day/multi_modal/splitted_into2/Tiles18_multi_modal_multi_modal')
else:
    raise ValueError('Problem in split number!')

print('loc_teacher', loc_teacher)

loc_root_models = os.path.join(loc_root_tl, str_models, base_model)
loc_root_performances = os.path.join(loc_root_tl, str_performance, base_model)
loc_error_text_file = os.path.join(loc_root_models, current_ds +'_'+ base_model + current_ds +'_model_error_file.txt')

loc_tiles_sensor_data = os.path.join(loc_tiles_18, 'fitbit', crt_sensor)
loc_sapiens_ds = os.path.join(temp_loc, 'SAPIENS/All Data Spring\'24/Data of Spring\'24')
loc_sensor2image = os.path.join(loc_root_tl, 'Fig_for_Paper_'+str_dataset) if save_fig_for_paper else os.path.join(loc_root_tl, str_dataset)

loc_RAW_sensor_image = os.path.join(loc_root_tl, str_raw_sensor_data)
loc_sensus_survey = os.path.join(root_loc, 'Sensus/Survey data/Sensus Final/')

# Data quality
loc_data_quality = '... removed the location a bit for privacy .../Documents/SA39/SAPIENS/All Data Spring\'24/Figures/Data Quality/'

############# Dataset, Performance, and Models Directory Checking #############
for __directory in [loc_root_models, loc_root_performances, loc_sensor2image, loc_RAW_sensor_image]:
    if not os.path.exists(__directory):
        os.makedirs(__directory)




############# GPU selection and GPU memory allocation  #########################
#if you want to manually allocate the GPU.
# 1. first, assign the gpu number such as '1' to gpu_number as a string
# 2. reduce the indentation and align with the start of if block. Otherwise, the gpu assigning code will move to the for block (as above to create directories)
# 3. you are all set!
if platform.system() != 'Darwin': # means: I am using the Ubuntu to use our titan server
    gpu_number = input('Enter GPU number to be used (GPU 1 has a high VRAM): ')
    os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
    os.environ["CUDA_VISIBLE_DEVICES"] = gpu_number

    if gpu_number == '1':
        list_configuration = [tf.config.LogicalDeviceConfiguration(memory_limit = 32720)] # GPU 0 has a  GPU memory of 24564 MiB; keeping some GPU memory for other lab members
    else:
        list_configuration = [tf.config.LogicalDeviceConfiguration(memory_limit = 22564)] # total is 24564.  I tried by reducing 1 GB. But I could not allocate that memory as I saw the message something like "failed to allocate ..."

    list_gpus = tf.config.list_physical_devices('GPU')
    tf.config.set_logical_device_configuration(list_gpus[0], list_configuration)
    
############# Random State - For reproducibility  #########################
random_state = 1234

tf.keras.utils.set_random_seed(random_state)
tf.config.experimental.enable_op_determinism()

# `Common Functions`

In [ ]:
# classification performance report

from sklearn.metrics import average_precision_score

def clf_report(true_labels, predicted_val_list, bool_print, bool_return_columns = False, bool_auc_roc = False, bool_prec_recall = False, list_predicted_proba = []):
    n_digits = 2

    bal_acc = balanced_accuracy_score(y_true=true_labels, y_pred=predicted_val_list)
    precision = precision_score(y_true=true_labels, y_pred=predicted_val_list, average= average_approach_for_ev_metric)
    recall = recall_score(y_true=true_labels, y_pred=predicted_val_list, pos_label=1)
    specificity = recall_score(y_true=true_labels, y_pred=predicted_val_list, pos_label=0)
    f1 = f1_score(y_true=true_labels, y_pred=predicted_val_list, average= average_approach_for_ev_metric)
    acc = accuracy_score(y_true=true_labels, y_pred=predicted_val_list)

    list_columns = ['precision', 'recall', 'specificity', str_balanced_acc, str_f1_score, 'accuracy']
    # WARNING: if you change the order of the evaluation metric, please remember to change the order in the printed evaluation metric as well.
    list_performance = [round(precision*100, n_digits), round(recall*100, n_digits), 
                        round(specificity*100, n_digits), round(bal_acc*100, n_digits), 
                        round(f1*100, n_digits), round(acc*100, n_digits)] 
    
    if bool_auc_roc and bool_prec_recall:
        list_performance.extend([round(roc_auc_score(y_true=true_labels, y_score=list_predicted_proba, average=average_approach_for_ev_metric) * 100, 2),
                                 round(average_precision_score(y_true=true_labels, y_score=list_predicted_proba, average= average_approach_for_ev_metric)* 100, 2)])
        list_columns.extend(['ROC-AUC', '(Avg precision) Precision-Recall'])
    elif bool_auc_roc or bool_prec_recall:
        raise ValueError(f'Hey Sabbir, write codes to handle it! bool_auc_roc {bool_auc_roc}, bool_prec_recall {bool_prec_recall} :)')

    if bool_print:
        print(list_columns)
        print(list_performance)
    
    if bool_return_columns:
        return list_performance, list_columns
    else:
        return list_performance


def categorize_by_median(df, pid_col, variable_to_cal_median):
    medians_df = df.groupby(pid_col)[variable_to_cal_median].median().rename(str_median).reset_index()
    df[str_median] = df[pid_col].map(dict(zip(medians_df[pid_col].tolist(), medians_df[str_median].tolist())))
    df[str_category] = np.where(((df[str_median] == 1) & (df[variable_to_cal_median] > 1)) |
                                ((df[str_median] != 1) & (df[variable_to_cal_median] >= df[str_median])), 1, 0)
    df.drop(columns=str_median, inplace=True)
    
    return df


def get_TILES_18_cleand_anxiety_demo_df(local_anxiety_clf_approach):
    dt_format_tiles_anxiety = '%Y-%m-%dT%H:%M:%S%z' # 2018-03-05T06:06:11-08:00; 2018-07-13T22:40:40-07:00. Regarding format: https://docs.python.org/3/library/datetime.html#strftime-and-strptime-behavior

    df_anxiety = pd.read_csv(os.path.join(loc_tiles_18, 'surveys/scored/EMAs/anxiety.csv.gz'))
    df_demo_info = pd.read_csv(os.path.join(loc_tiles_18, 'surveys/raw/demographics/part_one-demographics.csv.gz'))
    df_demo_info[str_age].replace({0:21}, inplace=True) # the minimum age in TILES-18 dataset is 21. See the paper on the dataset.

    print('# of NA values in anxiety df: ', df_anxiety.isna().sum().sum(), df_anxiety.isna().sum())
    df_anxiety.dropna(inplace=True)
    df_anxiety = df_anxiety[df_anxiety['Finished'] != 0] # Finished: "Whether the survey was finished" (README.md)

    set_timezones = set([date_time.split('T')[1].split('-')[1] for date_time in df_anxiety[str_completed_ts].tolist()]) # 2019-03-06T06:06:11-08:00.example value of column. Changed the exact date-time to make sure not the timestamp public (due to privacy) in case  we make the codes publicly available
    if set_timezones != {'08:00', '07:00'}: # https://en.wikipedia.org/wiki/Pacific_Time_Zone {'08:00', '07:00'}
        raise ValueError("Can be due to a severe Problem. Check please!")
    
    df_anxiety[str_completed_ts] = pd.to_datetime(df_anxiety[str_completed_ts], format= dt_format_tiles_anxiety)
    df_anxiety[str_timestamp] = df_anxiety[str_completed_ts].apply(lambda x: x.timestamp())
    df_anxiety[str_date] = df_anxiety[str_completed_ts].apply(lambda single_dt: single_dt.date())
    df_anxiety[str_hour] = df_anxiety[str_completed_ts].apply(lambda single_dt: single_dt.hour)
    df_anxiety.sort_values(by=[str_tiles_pid, str_date, str_hour], inplace=True)
    
    df_anxiety[str_start_ts] = pd.to_datetime(df_anxiety[str_start_ts], format= dt_format_tiles_anxiety)
    df_anxiety[str_difference] = df_anxiety[str_completed_ts] - df_anxiety[str_start_ts]
    df_anxiety[str_difference] = df_anxiety[str_difference].apply(lambda diff: diff.total_seconds() / 3600)

    duplicated_df = df_anxiety.duplicated(subset=[str_tiles_pid, str_date], keep='last')
    # print(df_anxiety[duplicated_df][str_difference].mean(), df_anxiety[duplicated_df].shape, df_anxiety.shape)  # This and the following statements helped to make decision to drop the rows where the difference is high
    # display(HTML(df_anxiety[duplicated_df].to_html()))
    
    df_anxiety = df_anxiety[~duplicated_df]
    print('df_anxiety.shape',  df_anxiety.shape)

    if (local_anxiety_clf_approach == str_1_vs_2_10_split):
        df_anxiety[str_category] = (df_anxiety[str_anxiety] > 1).astype(int) ##### df_anxiety = compare_anxiety_label_days(df_anxiety, str_tiles_pid, str_date)
    elif (local_anxiety_clf_approach == str_median):
        df_anxiety = categorize_by_median(df= df_anxiety, pid_col= str_tiles_pid, variable_to_cal_median = str_anxiety)
    else:
        raise ValueError('Problem in the values of crt_anxiety_type and local_anxiety_clf_approach!')

    print(f'Shape of the final df_anxiety: {df_anxiety.shape},\n \
        # of participants: {len(df_anxiety[str_tiles_pid].unique())},\n \
        # Distribution of class: {Counter(df_anxiety[str_category].tolist())}')
    
    return {str_anxiety: df_anxiety[[str_tiles_pid, str_anxiety, str_date, str_timestamp, str_category]], str_demograpnic_info: df_demo_info}
    


def get_cleaned_state_anxiety_SAPIENS(local_state_anxiety_clf_approach): # clf: classification
    df_state_anxiety = pd.read_excel(os.path.join(loc_sensus_survey, 'cleaned_emotion_survey_ds.xlsx'))

    if sorted(df_state_anxiety['LocalTimeZone'].unique()) != ['-04:00', '-05:00']:
        raise ValueError('Significant Problem. All of the time zones are not in -04:00 and -05:00. Hence, you can not convert all timestamps to the time zone to US/Eastern')
    df_state_anxiety.drop(columns=['LocalTimeZone'], inplace=True)
    df_state_anxiety.rename(columns={'SubmissionTimestamp': str_sensus_submission_time, 'ParticipantId': str_p_id}, inplace=True)

    df_state_anxiety[str_timestamp] = df_state_anxiety[str_sensus_submission_time].tolist().copy()
    df_state_anxiety[str_sensus_submission_time] = pd.to_datetime(df_state_anxiety[str_sensus_submission_time], unit='s', utc=True).dt.tz_convert('US/Eastern')
    df_state_anxiety[str_date] = df_state_anxiety[str_sensus_submission_time].dt.date

    if local_state_anxiety_clf_approach == str_1_vs_2_10_split:
        df_state_anxiety[str_category] = (df_state_anxiety[str_state_anxiety] > 1).astype(int)
    elif local_state_anxiety_clf_approach == str_median:
        df_state_anxiety = categorize_by_median(df = df_state_anxiety, pid_col= str_p_id, variable_to_cal_median = str_state_anxiety)
    else:
        raise ValueError('value of local_state_anxiety_clf_approach is not correct!')

    df_state_anxiety[str_p_id] = df_state_anxiety[str_p_id].astype(str)
    
    return df_state_anxiety


def get_cleaned_eod_SAPIENS():
    df_eod_data = pd.read_excel(os.path.join(loc_sensus_survey, 'cleaned_EoD_survey_responses.xlsx'))
    df_eod_data.rename(columns={'EOD Question': str_category, 'ParticipantId': str_p_id, 'Date': str_date, str_sensus_submission_time: str_timestamp}, inplace=True)
    df_eod_data[str_category] = df_eod_data[str_category].map({1:1, 2:0})
    df_eod_data[str_p_id] = df_eod_data[str_p_id].astype(str)
    
    ######  🔥🔥🔥🔥🔥🔥    WARNING: by default, you can not consider timezone all timestamps to be US/Eastern
    # 🔥🔥🔥🔥🔥🔥
    # 🔥🔥🔥🔥🔥🔥
    # 🔥🔥🔥🔥🔥🔥
    # 🔥🔥🔥🔥🔥🔥 dates were calculated by considering all to be in us/eastern

    df_eod_data[str_date] =  pd.to_datetime(df_eod_data[str_date]).dt.date
    df_eod_data[str_hour] = df_eod_data[str_hour].astype(int)
    return df_eod_data


def get_cleaned_SAPIENS_demographic_info():
    df_parti_demographic_info = pd.read_csv(os.path.join(root_loc, 'SAPIENS_Baseline_Data.csv'))
    df_parti_demographic_info.rename(columns = {'age': str_age, 'id': str_p_id}, inplace=True)
    # df_parti_demographic_info = df_parti_demographic_info[df_parti_demographic_info[str_p_id].isin(df_parti_info[str_parti_id].tolist())]
    df_parti_demographic_info.loc[df_parti_demographic_info['StartDate'] == 'removed for better privacy of the P', str_p_id] = 443 # Mistakenly, the participant who started participating earlier (... removed for better privacy of the P... ) scanned the wrong QR code. Participant who participated later will have the p id 444
    df_parti_demographic_info[str_p_id] = df_parti_demographic_info[str_p_id].astype(str)

    return df_parti_demographic_info


# WARNING: 🔥🔥🔥🔥🔥 The following approach can not be applied.
# The reason is currently for daily label anxiety prediction we are taking data upto the submission timestamp.
# However, for in advance prediction, I think we can take all data of the previous day. Currently, I do not see any justification to use data upto a particular hour when you want to predict anxiety of the next day.

# def annotate_each_day_category_one_day_in_advance():
#     if str_tiles_18 in current_ds:
#         df_anxiety = get_TILES_18_cleand_anxiety_demo_df().get(str_anxiety)
#         df_anxiety.rename(columns={str_tiles_pid: str_p_id}, inplace=True)
#     elif str_sapiens in current_ds:
#         df_anxiety = get_cleaned_eod_SAPIENS()

#     final_df_anxiety = pd.DataFrame()

#     for pid in df_anxiety[str_p_id].unique():
#         sub_df = df_anxiety[df_anxiety[str_p_id] == pid].copy()
#         sub_df.sort_values(by = [str_date]).reset_index(drop=True, inplace = True)

#         sub_df[str_category] = sub_df[str_category].shift(-1)
#         sub_df = sub_df.iloc[:-1]
#         final_df_anxiety = pd.concat([final_df_anxiety, sub_df])
    
#     final_df_anxiety[str_p_id + str_timestamp] = final_df_anxiety[str_p_id].astype(str) + final_df_anxiety[str_timestamp].astype(str)
#     return final_df_anxiety


def get_model_name_from_performance_file(perf_file_name):
    iteration = 'iter_'+ perf_file_name.split('_')[0] +'__' # added '__' to prevent any potential coincidentally match with anything else
    model_type = perf_file_name.split('_')[-2] # it will return either new or tuned. WARNING 😡🔥; if you add "tuned" to new layers file name (while saving the file after model training), this condition will not work and you will get the wrong model
    file_extension = 'keras' # there is '.pkl' model saved as history. thus, to avoid..
    list_tiles_models = os.listdir(os.path.join(loc_teacher, str_models, base_model)) # used str_sapiens+'_' to differentiate from the root folder SAPIENS

    model_name = [tiles_model for tiles_model in list_tiles_models if (iteration in tiles_model) and (model_type in tiles_model) and (file_extension in tiles_model)]
    if len(model_name) > 1:
        raise ValueError('There can not be 2 models satisfying same conditions. Super severe problem! 🔥🔥🔥')
    return model_name[0]
    

def get_model_name_on_top_n_new_layers_top_n_fine_tuned_layers(top_n, bool_display_performance = False):    
    data_list_performances = []
    temp_loc_performances = os.path.join(loc_teacher, str_performance, base_model)
    
    for perf_file in os.listdir(temp_loc_performances):
        loc_perf_file = os.path.join(temp_loc_performances, perf_file)
        if ('DS_Store' in perf_file) or (str_file_predict_prob_fine_tuned_resnet == perf_file) or (str_file_only_new_layers_predict_proba == perf_file) or os.path.isdir(loc_perf_file) or not perf_file.endswith('.xlsx'):
            continue
        
        df_perf = pd.read_excel(loc_perf_file)
        df_perf[str_predicted_class] = np.where(np.array(df_perf[str_predicted_proba].tolist()) > 0.5, 1, 0) # WARNING: should it be greater than 0.5. Check and ensure consistency with the evaluation metric used for training.
        list_performance_metric_val, list_columns = clf_report(true_labels= df_perf[str_actual_class].tolist(), predicted_val_list= df_perf[str_predicted_class].tolist(), bool_print=False, bool_return_columns = True)
        list_performance_metric_val.extend([perf_file])
        data_list_performances.append(list_performance_metric_val)
    
    list_columns.extend([str_filename])
    df_performances = pd.DataFrame(data = data_list_performances, columns = list_columns)
    df_performances[str_models] = df_performances[str_filename].apply(get_model_name_from_performance_file)
    df_performances.sort_values(by=[str_balanced_acc], inplace=True, ascending=False)
    df_performances.reset_index(inplace=True, drop=True)
    if bool_display_performance:
        display(HTML(df_performances.to_html()))

    list_top_n_perf_new_models = df_performances[df_performances[str_filename].str.contains(str_file_only_new_layers_predict_proba)].iloc[:top_n][str_models].tolist()
    list_top_n_fine_tune_models = df_performances[df_performances[str_filename].str.contains(str_file_predict_prob_fine_tuned_resnet)].iloc[:top_n][str_models].tolist()
    return list_top_n_perf_new_models, list_top_n_fine_tune_models


def func_append_condition(new_string): # This function is to make sure that we created the dataset depending on the condition set in the Control Center. If we change the conditions there, we must have to run the dataset creation function.
    global appended_conditions_to_be_matched
    appended_conditions_to_be_matched = appended_conditions_to_be_matched +'__'+ new_string

# `Model Development`

## `Getting the Sensor2Image Data`

In [ ]:
def change_actual_class_depending_on_cat_criteria(df_anxiety, df_image_path): # cat: category 🤣. Criteria: e.g., median, 1 vs. 2-10 split
  df_anxiety[str_p_id + str_timestamp] = df_anxiety[str_p_id].astype(str) + df_anxiety[str_timestamp].astype(str)
  dict_pid_timestamp_with_category = dict(zip(df_anxiety[str_p_id + str_timestamp].tolist(), df_anxiety[str_category]))

  df_image_path[str_p_id + str_timestamp] = df_image_path[str_p_id].astype(str) + df_image_path[str_timestamp].astype(str)
  df_image_path[str_actual_class] = df_image_path[str_p_id + str_timestamp].map(dict_pid_timestamp_with_category)
  print('# of NaN values in the Image path: ', df_image_path.isna().sum().sum()) # For in advance, we are moving the things to back. As a result, there will be a NaN value per participant

  if df_image_path.isna().sum().sum() > 0:
    raise ValueError('June 28: 2025: I think there should not be any NaN value. Please, find out why there is NaN value, if there is any.')

  df_image_path.dropna(inplace=True)
  df_image_path[str_actual_class] = df_image_path[str_actual_class].astype(str) # DO NOT keep it before dropna since NaN values will be string and there will be no NaN value found.

  return df_image_path

def get_sensor2image_dataset(loc_sensor_2_image):
  data_list_df = []
  
  for image_file in os.listdir(loc_sensor_2_image):
    if 'DS_Store' not in image_file:
        segment_list_file_name = image_file.split('__')
        data_list_df.append([os.path.join(loc_sensor_2_image, image_file), segment_list_file_name[0], segment_list_file_name[1], segment_list_file_name[2], segment_list_file_name[3]])
        # example of segment_list_file_name: ['1', 'cc25830a-254a-487f-acec-c0afb3962679', '1527793364.0', 'segment3']

  df_image_path = pd.DataFrame(data= data_list_df, columns = [str_filename, str_actual_class, str_p_id, str_timestamp, str_segment])
  length_of_df_without_transformation = df_image_path.shape[0]

  func_append_condition(crt_state_anxiety_classification_approach)
  print('Shape of df_image_path: ', df_image_path.shape)

  

  if ((current_ds == str_sapiens) and (crt_anxiety_type == str_state_anxiety) and (crt_state_anxiety_classification_approach == str_median)) \
      or (( current_ds == str_tiles_18) and (crt_state_anxiety_classification_approach == str_median)):
     if current_ds == str_sapiens:
       df_anxiety = get_cleaned_state_anxiety_SAPIENS(crt_state_anxiety_classification_approach)
     elif current_ds == str_tiles_18:
       df_anxiety = get_TILES_18_cleand_anxiety_demo_df(crt_state_anxiety_classification_approach).get(str_anxiety)
       df_anxiety.rename(columns={str_tiles_pid: str_p_id}, inplace=True)

     df_image_path = change_actual_class_depending_on_cat_criteria(df_anxiety= df_anxiety, df_image_path= df_image_path)

     if length_of_df_without_transformation != df_image_path.shape[0]:
       raise ValueError('Seems there is a problem!')

     print('Final shape of df_image_path: ', df_image_path.shape)

  return df_image_path

df_image_path = get_sensor2image_dataset(loc_sensor2image)

## Model Utilities

In [ ]:
@tf.keras.utils.register_keras_serializable()
class F1Score(tf.keras.metrics.Metric):
    """Custom F1 Score metric for binary classification."""
    
    def __init__(self, name='f1_score', threshold=0.5, **kwargs):
        super(F1Score, self).__init__(name=name, **kwargs)
        self.threshold = threshold
        self.tp = self.add_weight(name='tp', initializer='zeros', dtype=tf.float32)
        self.fp = self.add_weight(name='fp', initializer='zeros', dtype=tf.float32)
        self.fn = self.add_weight(name='fn', initializer='zeros', dtype=tf.float32)
    
    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred_binary = tf.cast(y_pred >= self.threshold, tf.float32)
        y_true = tf.cast(y_true, tf.float32)
        
        self.tp.assign_add(tf.reduce_sum(y_true * y_pred_binary))
        self.fp.assign_add(tf.reduce_sum((1 - y_true) * y_pred_binary))
        self.fn.assign_add(tf.reduce_sum(y_true * (1 - y_pred_binary)))
    
    def result(self):
        precision = self.tp / (self.tp + self.fp + tf.keras.backend.epsilon())
        recall = self.tp / (self.tp + self.fn + tf.keras.backend.epsilon())
        return 2 * precision * recall / (precision + recall + tf.keras.backend.epsilon())
    
    def reset_states(self):
        for var in self.variables:
            var.assign(0.0)

    def get_config(self):
        base_config = super().get_config()
        return {**base_config,
                "threshold": self.threshold}
    


# https://stackoverflow.com/questions/64474463/custom-f1-score-metric-in-tensorflow

# class F1_Score(tf.keras.metrics.Metric):
#     def __init__(self, name='f1_score', **kwargs):
#         super().__init__(name=name, **kwargs)
#         self.f1 = self.add_weight(name='f1', initializer='zeros')
#         self.precision_fn = Precision(thresholds=0.5)
#         self.recall_fn = Recall(thresholds=0.5)

#     def update_state(self, y_true, y_pred, sample_weight=None):
#         p = self.precision_fn(y_true, y_pred)
#         r = self.recall_fn(y_true, y_pred)
#         # since f1 is a variable, we use assign
#         self.f1.assign(2 * ((p * r) / (p + r + 1e-6)))

#     def result(self):
#         return self.f1

#     def reset_states(self):
#         # we also need to reset the state of the precision and recall objects
#         self.precision_fn.reset_states()
#         self.recall_fn.reset_states()
#         self.f1.assign(0)
    


def select_pids_by_class_ratio(df, class_ratio, n, tolerance=10):
    df = df.copy()
    df[str_actual_class] = df[str_actual_class].astype(int)
   
    class_counts = df.groupby(['pid', 'Actual_class']).size().unstack(fill_value=0).reset_index()
    class_counts = class_counts.rename(columns={0: 'class0', 1: 'class1'})
    
    class_counts = class_counts[(class_counts['class1'] > 0) & (class_counts['class0'] > 0)].copy()
    
    if len(class_counts) < n:
        raise ValueError("It was not supposed to happen 🥴. Check the dataset!")
    
    class_counts['ratio'] = (class_counts['class1'] * 100) / class_counts['class0']
    class_counts['diff'] = abs(class_counts['ratio'] - class_ratio)
    class_counts = class_counts.sort_values('diff').reset_index(drop=True)
    
    selected_pids = []
    total_class0 = 0
    total_class1 = 0
    
    for idx, row in class_counts.iterrows():
        if len(selected_pids) >= n:
            break
        selected_pids.append(row['pid'])
        total_class0 += row['class0']
        total_class1 += row['class1']
    
    if total_class1 == 0:
        print("Selected pids have zero instances of class 1.")
        return []
    
    combined_ratio = (total_class1 * 100) / total_class0
    
    if (class_ratio - tolerance) <= combined_ratio <= (class_ratio + tolerance):
        return selected_pids
    else:
        print(f"\n\nCombined class ratio {combined_ratio:.2f} is outside the desired range.\n\n")
        return selected_pids
    

def save_list_to_text_file(list_of_data, filename):
    array = np.array(list_of_data)
    
    if not isinstance(array, np.ndarray):
        raise ValueError("Input must be a NumPy array.")

    try:
        with open(filename, 'a+') as f:
            np.savetxt(f, array.reshape(-1, array.shape[-1]) if array.ndim > 1 else array, fmt='%s')
    except Exception as e:
        print(f"An error occurred while saving the array: {e}")


@tf.keras.utils.register_keras_serializable()
class DataAugmentation(tf.keras.layers.Layer):
    def __init__(self, image_size=(224, 224, 3), flip_probability = 0.5, stretch_range = (1.001, 1.05), noise_stddev = 0.05, **kwargs):
        super().__init__(**kwargs)
        self.image_size = image_size
        self.flip_probability = flip_probability
        self.stretch_min, self.stretch_max = stretch_range
        self.noise_stddev = noise_stddev

    def call(self, inputs, training=None):
        if training:
            x = inputs
            x = self.random_flip_temporal(x)
            x = self.random_stretch_temporal(x)
            x = self.add_gaussian_noise(x)
            return x
        else:
            return inputs
        
    def random_flip_temporal(self, x):
        rand_val = tf.random.uniform(())
        return tf.cond(
            rand_val < self.flip_probability,
            lambda: tf.image.flip_left_right(x),
            lambda: x)
    

    def random_stretch_temporal(self, x):
        height = self.image_size
        width = self.image_size

        scale = tf.random.uniform([], self.stretch_min, self.stretch_max)
        new_width = tf.cast(scale * tf.cast(width, tf.float32), tf.int32)
        new_width = tf.maximum(new_width, 1)

        x_resized = tf.image.resize(x, [height, new_width], method=tf.image.ResizeMethod.BILINEAR)
        width_diff = tf.maximum(new_width - width, 0)
        offset = tf.random.uniform([], 0, tf.cast(width_diff + 1, tf.int32), dtype=tf.int32)
        x_cropped = tf.image.crop_to_bounding_box(x_resized, offset_height=0, offset_width=offset, target_height=height, target_width=width)
        return x_cropped

    def add_gaussian_noise(self, x):
        noise = tf.random.normal(shape=tf.shape(x), mean=0.0, stddev=self.noise_stddev)
        return x + noise
    
    def get_config(self):
        config = super().get_config()
        config.update({'image_size': self.image_size,
                       'stretch_range': (self.stretch_min, self.stretch_max),
                       'noise_stddev': self.noise_stddev})
        return config


@tf.keras.utils.register_keras_serializable()
class WeightedBinaryFocalLoss(tf.keras.losses.Loss):
    def __init__(self, w0, w1, gamma=2.0, from_logits=False, name="wbf", reduction='sum_over_batch_size', **kwargs):
        super().__init__(name=name, reduction=reduction, **kwargs)
        self.w0 = w0
        self.w1 = w1
        self.gamma = gamma
        self.from_logits = from_logits
        self._base = tf.keras.losses.BinaryFocalCrossentropy(gamma=gamma, from_logits=from_logits)

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        fl = self._base(y_true, y_pred)
        weight = y_true * self.w1 + (1. - y_true) * self.w0
        return tf.reduce_mean(weight * fl)

    def get_config(self):
        config = super().get_config()
        config.update({"w0": self.w0, "w1": self.w1,
                       "gamma": self.gamma, "from_logits": self.from_logits, 
                       "name": self.name})
        return config


def get_class_weight(df_train, target_column):
    list_classes = df_train[target_column].astype(int)
    neg, pos = np.bincount(list_classes)
    total = neg + pos
    weight_for_0 = (1 / neg) * (total / 2.0)
    weight_for_1 = (1 / pos) * (total / 2.0)
    class_weight = {0: weight_for_0, 1: weight_for_1}
    return class_weight


def residual_block(x, filters, kernel_size):
    y = layers.Conv2D(filters, kernel_size, padding='same')(x)
    y = layers.BatchNormalization()(y)
    y = layers.ReLU()(y)

    y = layers.Conv2D(filters, kernel_size, padding='same')(y)
    y = layers.BatchNormalization()(y)

    out = layers.Add()([x, y])
    out = layers.ReLU()(out)
    return out


def save_training_history(history_df, loc_in_keras, str_to_append):
    history_df.to_pickle(loc_in_keras.replace('.keras', str_to_append+'_.pkl'))


def conv1_1_residual_block(x, filters, kernel_size, use_1x1conv=False, strides=1):
    skip = x
    
    y = layers.Conv2D(filters, kernel_size, padding='same', strides=strides)(x)
    y = layers.BatchNormalization()(y)
    y = layers.ReLU()(y)

    y = layers.Conv2D(filters, kernel_size, padding='same')(y)
    y = layers.BatchNormalization()(y)

    if use_1x1conv:
        skip = layers.Conv2D(filters, 1, padding='same', strides=strides)(skip)
        skip = layers.BatchNormalization()(skip)

    out = layers.Add()([skip, y])
    out = layers.ReLU()(out)
    return out

class ConditionedModelCheckpointCompressed(tf.keras.callbacks.Callback):
    def __init__(self, save_dir, tolerance=0.05):
        """
        Parameters:
          save_dir: Directory to save epoch weights e.g. "./checkpoints"
          tolerance: e.g., 0.05 => 5% maximum difference allowed
        """
        super().__init__()
        self.save_dir = save_dir
        self.tolerance = tolerance
        self.epoch_data = []  # to store epoch stats, plus the file path
        self.best_epoch = None
        self.best_train_loss = None
        self.best_val_loss = None
        self.best_weight_file = None

        # Create directory if doesn't exist
        os.makedirs(self.save_dir, exist_ok=True)

    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            logs = {}
        train_loss = logs.get(str_loss)
        val_loss = logs.get(str_val_loss)

        if train_loss is not None and val_loss is not None:

            weights_file = os.path.join(self.save_dir, f"epoch_{epoch}_.weights.h5")
            self.model.save_weights(weights_file)

            # 3) record info
            self.epoch_data.append({
                str_epoch: epoch,
                str_train_loss: train_loss,
                str_val_loss: val_loss,
                str_difference: abs(val_loss - train_loss),
                str_weights_file: weights_file})

    def on_train_end(self, logs=None):
        valid = [d for d in self.epoch_data if ((d[str_difference] <= self.tolerance) and (d[str_val_loss] > d[str_train_loss]))]

        if valid:
            best = min(valid, key=lambda d: d[str_difference])
            print(f"Epochs meeting difference  <= {self.tolerance*100}%: {[v['epoch'] for v in valid]}")
            print(f"Restoring best among them => epoch={best['epoch']}, "
                  f"val_loss={best['val_loss']:.4f}, train_loss={best['train_loss']:.4f}, "
                  f"difference ={best['difference']:.4f}")
        else:
            best = min(self.epoch_data, key=lambda d: d["difference"])
            print(f"No epoch satisfies difference <= {self.tolerance*100:.1f}%.")
            print(f"Restoring epoch with smallest difference => epoch={best['epoch']}, "
                  f"val_loss={best['val_loss']:.4f}, train_loss={best['train_loss']:.4f}, "
                  f"difference ={best['difference']:.4f}")

        # Reload best epoch's weights
        best_file = best[str_weights_file]
        self.model.load_weights(best_file)

        self.best_epoch = best[str_epoch]
        self.best_train_loss = best[str_train_loss]
        self.best_val_loss = best[str_val_loss]
        self.best_weight_file = best_file

        print(f"Restored weights from file: {best_file}")



def squeeze_excite_block(x, ratio=16, name_prefix="se_block"):
    channels = x.shape[-1]
    se = layers.GlobalAveragePooling2D(name=f"{name_prefix}_gap")(x)
    # se = layers.Reshape((1,1,channels), name=f"{name_prefix}_reshape")(se)
    
    hidden_units = max(channels // ratio, 1)
    print('hidden_units ', hidden_units, channels)
    
    se = layers.Dense(hidden_units, use_bias=False, name=f"{name_prefix}_fc1", kernel_initializer = tf.keras.initializers.HeNormal())(se)
    se = layers.BatchNormalization()(se)
    se = layers.ReLU(name=f"{name_prefix}_relu")(se)
    se = layers.Dense(channels, use_bias=False, activation='sigmoid', name=f"{name_prefix}_fc2", kernel_initializer = tf.keras.initializers.HeNormal())(se)
    
    x = layers.multiply([x, se], name=f"{name_prefix}_scale")
    return x



list_metrics = [tf.keras.metrics.TruePositives(name='tp'), tf.keras.metrics.FalsePositives(name='fp'),
                tf.keras.metrics.TrueNegatives(name='tn'), tf.keras.metrics.FalseNegatives(name='fn'),
                F1Score(name=str_f1_score),
                tf.keras.metrics.AUC(name='auc_pr', curve='PR'), tf.keras.metrics.BinaryAccuracy(name='accuracy'),
                tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]

## Model Evaulation

In [ ]:
def individual_meta_learner(list_features, df_data_for_meta):
    # meta_learner(list_features, df_data_for_meta, AdaBoostClassifier(random_state=random_state, algorithm = 'SAMME'), 'AdaBoostClassifier') # had to set algorithm: ‘SAMME’ since I was getting warning "FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6....."
    # meta_learner(list_features, df_data_for_meta, DecisionTreeClassifier(random_state= random_state), 'DecisionTreeClassifier')
    # meta_learner(list_features, df_data_for_meta, ExtraTreesClassifier(random_state= random_state), 'ExtraTreesClassifier')
    # meta_learner(list_features, df_data_for_meta, GradientBoostingClassifier(random_state= random_state), 'GradientBoostingClassifier')
    # meta_learner(list_features, df_data_for_meta, HistGradientBoostingClassifier(random_state=random_state, class_weight='balanced'), 'HistGradientBoostingClassifier')
    # meta_learner(list_features, df_data_for_meta, KNeighborsClassifier(), 'KNeighborsClassifier')
    meta_learner(list_features, df_data_for_meta, NearestCentroid(), 'NearestCentroid')
    # meta_learner(list_features, df_data_for_meta, LogisticRegression(random_state= random_state), 'LogisticRegression')
    # meta_learner(list_features, df_data_for_meta, MLPClassifier(random_state= random_state, hidden_layer_sizes=3, early_stopping=True), 'MLPClassifier')
    # meta_learner(list_features, df_data_for_meta, RandomForestClassifier(random_state=random_state), 'RandomForestClassifier')
    # meta_learner(list_features, df_data_for_meta, LinearSVC(random_state= random_state), 'LinearSVC')
    # meta_learner(list_features, df_data_for_meta, XGBClassifier(random_state = random_state), 'XGBClassifier')


def meta_learner(list_features, df_data_for_meta, ML_model, model_name):
    list_actual_class = []
    list_predicted_class = []
    
    df_data_for_meta[str_timestamp] = df_data_for_meta[str_timestamp].astype(str)
    df_data_for_meta[str_p_id] = df_data_for_meta[str_p_id].astype(str)

    for pid in sorted(df_data_for_meta[str_p_id].unique()):
        test_df_meta_learner = df_data_for_meta[df_data_for_meta[str_p_id] == pid].copy()

        train_data_meta_learner = df_data_for_meta[df_data_for_meta[str_p_id] != pid].copy()
        list_test_timestamp_test = sorted(test_df_meta_learner[str_timestamp].tolist())

        for single_test_date in list_test_timestamp_test:
            single_test_data_instance = test_df_meta_learner[test_df_meta_learner[str_timestamp] == single_test_date]

            x_train = train_data_meta_learner[list_features].values.tolist()
            y_train = train_data_meta_learner[str_actual_class].tolist()

            x_test = single_test_data_instance[list_features].values.tolist()
            y_test = single_test_data_instance[str_actual_class].tolist()

            if len(y_test) > 1:
                print(pid, single_test_data_instance)
                print('\n🔥🔥🔥🔥Since meta learner is predicting class of each day seperately, there is suposed only 1 class per day.\n')
                raise ValueError('\nSince meta learner is predicting class of each day seperately, there is suposed only 1 class per day.\n')

            if len(np.unique(y_train)) == 1:
                print(f'\nNot sufficient data in this iteration. Date: {single_test_date}, PID: {pid}\n')
                continue

            ML_model.fit(x_train, y_train)
            list_actual_class.extend(y_test)
            list_predicted_class.extend(ML_model.predict(x_test))
            
            train_data_meta_learner = pd.concat([train_data_meta_learner, single_test_data_instance])
    
    print(model_name, clf_report(true_labels= list_actual_class, predicted_val_list= list_predicted_class, bool_print= True))


def get_file_names_without_segment_name(list_file_names):
    set_file_names_excl_segment_name = set()
    
    for temp_file_name in list_file_names:
        # We can optimize it later by removing the for loop :). Note: the codes are fine. A file can have maximum 1 segment name. Right? If yes, why do we need to iterate over each segment which will increase the time complexity n_segments_of_a_day times. Just look into the file name and replace just the segment name with ''
        for nth_segment in range(1, n_segments_of_a_day+1):
            temp_file_name = temp_file_name.replace(str_segment + str(nth_segment), '')
        set_file_names_excl_segment_name.add(temp_file_name)

    return set_file_names_excl_segment_name




def evalulate_model(list_predicted_probabilities, df_test, df_data_for_meta, n_th_iteration_of_training, training_type_file_name):
    global loc_performances, model_run_start_time
    print('Shape of df_data_for_meta: ', df_data_for_meta.shape[0])

    ############ Test dataframe including the predicted probabilities ############
    df_test = df_test.copy() # Copying it so that changes here does not reflect to the passed df_test :)
    df_test[str_actual_class] = df_test[str_actual_class].astype(int)
    df_test[str_predicted_proba] = list_predicted_probabilities
    df_test[str_n_th_iter_of_train] = np.repeat(n_th_iteration_of_training, df_test.shape[0])
    df_test[str_model_run_start_time] = np.repeat(model_run_start_time, df_test.shape[0])
    df_test.to_excel(os.path.join(loc_performances, str(n_th_iteration_of_training)+'__'+ training_type_file_name), index=False)

    print("Checking performances of the weakly labelled data")
    list_predicted_class = np.where(np.array(df_test[str_predicted_proba].tolist()) > 0.5, 1, 0)
    print(clf_report(true_labels= df_test[str_actual_class].tolist(), predicted_val_list= list_predicted_class, bool_print= True))
    

    dict_file_prob = dict(zip(df_test[str_filename].tolist(), df_test[str_predicted_proba].tolist()))
    set_file_names_excl_segment_name = get_file_names_without_segment_name(df_test[str_filename].tolist())
    pred_prob_data_list_segments_df = []

    # This loop is used to get the predicted probabilities from only the files (i.e., files of a date of a pid) for which there are data of all periods of the day
    # Finally, we will have a dataframe like probability_from_period_0_11_image - probability_from_period_0_11_image - ..... - class - pid - date - n_th_iteration_of_training (like fold) 
    for filename_without_seg_name in set_file_names_excl_segment_name:
        single_list_pred_prob = []
        list_segments = [str_segment + str(temp_nth_segment) for temp_nth_segment in range(1, n_segments_of_a_day+1)]
        bool_all_periods = True # Adding the data only when there are data of both periods (if there are at least min_number_of_samples_for_image (Jan. 5, 2024: 50 samples))
        
        for segment in list_segments:
            splitted_file_name = filename_without_seg_name.split('____') # Let's say, an image file name is "0__413__2024-03-06__segment1__Battery__Level.png". When you remove the segment name, the file name will become "0__413__2024-03-06____Battery__Level" (i.e., will contain ____)
            filename_with_segment_name = splitted_file_name[0] +'__'+  segment +'__'+ splitted_file_name[1]
            
            if filename_with_segment_name in dict_file_prob.keys() and bool_all_periods:
                single_list_pred_prob.append(dict_file_prob.get(filename_with_segment_name))
            else:
                print(f'This segment was not included during meta learner development: {filename_with_segment_name}')
                bool_all_periods = False

        if bool_all_periods:
            single_list_pred_prob.extend([df_test[df_test[str_filename] == filename_with_segment_name][str_actual_class].iloc[0], 
                                          df_test[df_test[str_filename] == filename_with_segment_name][str_p_id].iloc[0], 
                                          df_test[df_test[str_filename] == filename_with_segment_name][str_timestamp].iloc[0],
                                          n_th_iteration_of_training])
            pred_prob_data_list_segments_df.append(single_list_pred_prob)

    list_columns = list_segments.copy()
    list_columns.extend([str_actual_class, str_p_id, str_timestamp, str_n_th_iter_of_train])
    df_single_p_perform = pd.DataFrame(data= pred_prob_data_list_segments_df, columns = list_columns)    
    df_data_for_meta =  pd.concat([df_data_for_meta, df_single_p_perform])

    # if len(df_data_for_meta[str_p_id].unique()) > 1:
    #     individual_meta_learner(list_features= list_segments, df_data_for_meta= df_data_for_meta)
    # else:
    #     print(f'\n\nNot sufficient data to train a meta-learner. # of participants\' data currently I have: {len(df_data_for_meta[str_p_id].unique())}. The performance is based on weakly labelled data.\n\n')
    #     print(clf_report(true_labels= df_test[str_actual_class].tolist(), predicted_val_list= list_predicted_class, bool_print=True))
    
    return df_data_for_meta


## Tiles Model Point

In [ ]:
def train_val_test_save_model(df_temp_train, df_val, df_test, loc_new_layers_model_file, loc_resnet_fine_tuned_model_file, n_th_iteration_of_training):
    global loc_performances, loc_models
    global batch_size, last_n_layers, image_size
    global df_data_for_meta_only_new_layers, df_data_for_meta_all_layers
    
    ########## Control Center - Start ##########
    n_epoch_for_weights = 20
    n_epoch_for_fine_tune = 20
    n_times_patience_for_new_layers = 3
    n_times_patience_for_all_layers = 3
    residual_block_n_filters = 512
    input_shape = (image_size, image_size, 3)

    if resnet_version == 50:
        preprocess_input = tf.keras.applications.resnet.preprocess_input
        base_model = tf.keras.applications.ResNet50(input_shape=input_shape, include_top=False, weights="imagenet")
        residual_block_n_filters = 2048
    elif resnet_version == 18:
        ResNet_x, preprocess_input = Classifiers.get('resnet18')
        base_model = ResNet_x(input_shape=input_shape, weights='imagenet', include_top=False)
    elif resnet_version == 34:
        ResNet_x, preprocess_input = Classifiers.get('resnet34')
        base_model = ResNet_x(input_shape=input_shape, weights='imagenet', include_top=False)
    else:
        raise ValueError('Check the ResNet version, please. 😡😡😡')

    ### aug_layer = DataAugmentation(image_size= image_size, flip_probability=0.5, stretch_range=(1.001, 1.05), noise_stddev=0.05)
    ### x = aug_layer(inputs)

    ########## Creating Model ##########
    base_model.trainable = False
    inputs = tf.keras.Input(shape= input_shape)
    x = preprocess_input(inputs)
    x = base_model(x, training=False)

    x = residual_block(x, filters=residual_block_n_filters, kernel_size=3)
    x = layers.GlobalAveragePooling2D()(x)

    outputs = layers.Dense(1, activation='sigmoid', kernel_initializer = tf.keras.initializers.HeNormal())(x)
    tf_model = tf.keras.Model(inputs, outputs)

    ########## Train, test, val Data Preparation ##########
    datagen = ImageDataGenerator()
    train_gen = datagen.flow_from_dataframe(dataframe=df_temp_train, directory=None,  x_col=str_filename, y_col=str_actual_class, target_size=(image_size, image_size), color_mode='rgb', class_mode='binary', batch_size=batch_size, shuffle=True)
    val_gen = datagen.flow_from_dataframe(dataframe=df_val, directory=None,  x_col=str_filename, y_col=str_actual_class, target_size=(image_size, image_size), color_mode='rgb', class_mode='binary', batch_size=batch_size, shuffle=True)
    test_gen = datagen.flow_from_dataframe(dataframe=df_test, directory=None,  x_col=str_filename, y_col=str_actual_class, target_size=(image_size, image_size), color_mode='rgb', class_mode='binary', batch_size=batch_size, shuffle=False) # WARNING: do NOT set it to TRUE since the order of the data will get changed.

    train_steps = (len(df_temp_train) + batch_size - 1) // batch_size
    validation_steps = (len(df_val) + batch_size - 1) // batch_size
    test_steps = (len(df_test) + batch_size - 1) // batch_size
    class_weight = get_class_weight(df_temp_train, str_actual_class)

    ########## Training only newly added layers ##########
    # WARNING 🔥🔥🔥: If you change the value of the monitor paramneter in EarlyStopping, do NOT forget to change the mode parameter.
    callback_new_layers = tf.keras.callbacks.EarlyStopping(monitor=str_val_loss, patience=n_times_patience_for_new_layers, restore_best_weights=True) #  mode='max' 🔥 WARNING: do NOT forget to set mode='max' if you use a metric like f1 score to monitor and pick the best model based on that 
    lr_schedule_new_layers = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate=0.0001, decay_steps=train_steps, decay_rate=0.15, staircase=True) # lr: 0.0005 # 0.005
    optimizer_new_layer = tf.keras.optimizers.SGD(learning_rate=lr_schedule_new_layers)

    if (current_day_status == str_on_the_same_day) or (resnet_version != 50):
        decay_rate_for_all_layers = 0.4
        learning_rate_for_all_layers = 1e-8
    elif current_day_status == str_one_day_in_advance:
        decay_rate_for_all_layers = 0.40
        learning_rate_for_all_layers = 1e-7
    else:
        raise ValueError(f'Check current_day_status. Currently, it is {current_day_status}!')
    
    print('lr, decay for all layers training phase: ', learning_rate_for_all_layers, decay_rate_for_all_layers)
    callback_all_layers = tf.keras.callbacks.EarlyStopping(monitor=str_val_loss, patience=n_times_patience_for_all_layers, restore_best_weights=True) # mode='max'
    lr_schedule_all_layers = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate= learning_rate_for_all_layers, decay_steps=train_steps, decay_rate= decay_rate_for_all_layers, staircase=True)
    optimizer_all_layers = tf.keras.optimizers.SGD(learning_rate=lr_schedule_all_layers)

    tf_model.compile(loss=WeightedBinaryFocalLoss(w0=class_weight[0], w1=class_weight[1]), optimizer=optimizer_new_layer, metrics=list_metrics)
    history = tf_model.fit(train_gen, epochs = n_epoch_for_weights, steps_per_epoch= train_steps,  validation_data = val_gen, validation_steps= validation_steps, callbacks = [callback_new_layers], verbose = 1)
    save_training_history(pd.DataFrame(history.history), loc_new_layers_model_file, '')
    list_predict_proba = [arr[0] for arr in list(tf_model.predict(test_gen, steps = test_steps))]
    tf_model.save(loc_new_layers_model_file)

    print('\n\n\n Performance after training only the last layers 🙈😀')
    df_data_for_meta_only_new_layers = evalulate_model(list_predicted_probabilities=list_predict_proba, df_test= df_test, 
                                                       df_data_for_meta=df_data_for_meta_only_new_layers, n_th_iteration_of_training= n_th_iteration_of_training,
                                                       training_type_file_name = str_file_only_new_layers_predict_proba)
    df_data_for_meta_only_new_layers.to_excel(os.path.join(loc_performances, str_file_only_new_layers_predict_proba), index=False)

    ########## fine-tuning last n layers ##########
    base_model.trainable = True
    tf_model.compile(loss=WeightedBinaryFocalLoss(w0=class_weight[0], w1=class_weight[1]), optimizer= optimizer_all_layers, metrics=list_metrics)
    history = tf_model.fit(train_gen, epochs = n_epoch_for_fine_tune, steps_per_epoch= train_steps, validation_data = val_gen, validation_steps= validation_steps, callbacks = [callback_all_layers], verbose = 1)
    save_training_history(pd.DataFrame(history.history), loc_resnet_fine_tuned_model_file, '')
    list_predict_proba = [arr[0] for arr in list(tf_model.predict(test_gen, steps = test_steps))]
    tf_model.save(loc_resnet_fine_tuned_model_file)

    print('\n\n\n Performance After retrain all layers ')
    df_data_for_meta_all_layers = evalulate_model(list_predicted_probabilities=list_predict_proba, df_test=df_test, 
                                                  df_data_for_meta=df_data_for_meta_all_layers, n_th_iteration_of_training=n_th_iteration_of_training,
                                                  training_type_file_name = str_file_predict_prob_fine_tuned_resnet)
    df_data_for_meta_all_layers.to_excel(os.path.join(loc_performances, str_file_predict_prob_fine_tuned_resnet), index=False)

    ########## Clearing (memory is not feering (only in MAC though; there is a memory leakge in https://www.google.com/search?q=memory+leakage+issue+with+tf-metal+site%3A+forums.developer.apple.com) as much as I expected 😡😡😡😡)
    K.clear_session() # seems like implementation of K.clear_session() and keras.backend.clear_session a bit different and thus, using both. For example, I did not see the free_memory parameter in K.clear_session()
    gc.collect()
    del tf_model, train_gen, val_gen, test_gen, history, base_model

## SAPIENS Model Zone

In [ ]:
def get_teacher_models():
  list_teacher_models = []
  list_teacher_models_new_lyares, list_teacher_models_fine_tuned = get_model_name_on_top_n_new_layers_top_n_fine_tuned_layers(top_n_teacher_models_for_SAPIENS)
#   list_teacher_models.extend(list_teacher_models_new_lyares)
  list_teacher_models.extend(list_teacher_models_fine_tuned)

  return list_teacher_models


def SAPIENS_train_val_test_save_model(df_temp_train, df_val, df_test, loc_new_layers_model_file, loc_resnet_fine_tuned_model_file, n_th_iteration_of_training):
    global loc_performances, loc_models
    global batch_size, last_n_layers, image_size
    global df_data_for_meta_only_new_layers, df_data_for_meta_all_layers

    ########## Control Center - Start ##########
    n_epoch_for_weights = 10
    n_epoch_for_fine_tune = 10
    n_times_patience = 3

    ########## Train, test, val Data Preparation ##########
    datagen = ImageDataGenerator()
    train_gen = datagen.flow_from_dataframe(dataframe=df_temp_train, directory=None,  x_col=str_filename, y_col=str_actual_class, target_size=(image_size, image_size), color_mode='rgb', class_mode='binary', batch_size=batch_size, shuffle=True)
    val_gen = datagen.flow_from_dataframe(dataframe=df_val, directory=None,  x_col=str_filename, y_col=str_actual_class, target_size=(image_size, image_size), color_mode='rgb', class_mode='binary', batch_size=batch_size, shuffle=True)
    test_gen = datagen.flow_from_dataframe(dataframe=df_test, directory=None,  x_col=str_filename, y_col=str_actual_class, target_size=(image_size, image_size), color_mode='rgb', class_mode='binary', batch_size= batch_size, shuffle=False)  # WARNING: do NOT set it to TRUE since the order of the data will get changed in that case.
    
    ##### 🔥🔥🔥 WARNING: Do NOTTTTT call next(test_gen) since you are running that statement to predict


    train_steps = (len(df_temp_train) + batch_size - 1) // batch_size
    validation_steps = (len(df_val) + batch_size - 1) // batch_size
    class_weight = get_class_weight(df_temp_train, str_actual_class)
    input_shape = (image_size, image_size, 3)
    
    
    ########## Creating Model ############
    if resnet_version == 50:
        learning_rate_for_new_layer = 1e-6
        learning_rate_for_fine_tune = 1e-5
        residual_block_n_filters = 2048
    elif resnet_version == 18:
        learning_rate_for_new_layer = 1e-5
        learning_rate_for_fine_tune = 5e-7
        residual_block_n_filters = 512
    else:
        raise ValueError('Check the ResNet version, please. ☹️😡😡😡')
    
    # Do not move the following 3 lines of codes to the begining. The folloiwing codes will help to train based on multiple teacher models just creating loop along with a list of base models' location
    if not os.path.exists(os.path.join(loc_models)):
        os.makedirs(os.path.join(loc_models))
        os.makedirs(os.path.join(loc_performances))

    aug_layer = DataAugmentation(image_size= image_size, flip_probability=0.5, stretch_range=(1.001, 1.05), noise_stddev=0.05)
    loc_upto_day_status = '/'.join(loc_models.split('/')[:-2])
    tf_base_model = tf.keras.models.load_model(os.path.join(loc_teacher, str_models, base_model, base_model_name_for_SAPIENS),
                                            custom_objects={ "WeightedBinaryFocalLoss": WeightedBinaryFocalLoss, 'F1Score': F1Score, 'DataAugmentation': DataAugmentation})
    tf_base_model = tf.keras.Model(inputs=tf_base_model.input, outputs=tf_base_model.layers[-3].output)
    tf_base_model.trainable = False
        
    inputs = tf.keras.Input(shape= input_shape)
    x = tf_base_model(inputs, training=False)
    # x = residual_block(x, filters=residual_block_n_filters, kernel_size=3)
    #### x = conv1_1_residual_block(x= x, filters=384, kernel_size=3, use_1x1conv=True, strides=1)
    #### x = squeeze_excite_block(x, ratio=12, name_prefix="se_block")
    x = layers.GlobalAveragePooling2D()(x)
    
    #### x = layers.Dense(16, use_bias=False, kernel_initializer = tf.keras.initializers.HeNormal())(x)
    #### x = layers.BatchNormalization()(x)
    #### x = layers.ReLU()(x)
    
    outputs = layers.Dense(1, activation='sigmoid', kernel_initializer = tf.keras.initializers.HeNormal())(x)
    tf_model = tf.keras.Model(inputs, outputs)

    ########## Training only newly added layers ##########
    # WARNING 🔥🔥🔥: If you change the value of the monitor paramneter in EarlyStopping, do NOT forget to change the mode parameter.

    tolerance = 0.03
    temp_loc_models = loc_new_layers_model_file ## Do NOT keep the name as loc_models since it will conflict with the global loc_models variable

    
    loc_dir_weights = os.path.join(loc_model_weights, str(learning_rate_for_new_layer))
    callback = ConditionedModelCheckpointCompressed(save_dir=loc_dir_weights, tolerance= tolerance)
    optimizer =  tf.keras.optimizers.Nadam(learning_rate= learning_rate_for_new_layer, beta_1=0.9, beta_2=0.999)

    tf_model.compile(loss=WeightedBinaryFocalLoss(w0=class_weight[0], w1=class_weight[1]), optimizer = optimizer, metrics=list_metrics)
    history = tf_model.fit(train_gen, epochs = n_epoch_for_weights, steps_per_epoch= train_steps,  validation_data = val_gen, validation_steps= validation_steps, callbacks = [callback], verbose = 1)
    history.history[str_best_epoch] = callback.best_epoch; history.history[str_difference] = abs(callback.best_val_loss - callback.best_train_loss)
    save_training_history(pd.DataFrame(history.history), temp_loc_models, '__'+str(learning_rate_for_new_layer))
    
    print(callback.best_weight_file)
    tf_model.load_weights(callback.best_weight_file)
    tf_model.save(os.path.join(loc_models, os.path.basename(loc_new_layers_model_file)))

    test_steps = (len(df_test) + batch_size - 1) // batch_size
    list_predict_proba = [arr[0] for arr in list(tf_model.predict(test_gen, steps = test_steps))]

    # n_test_batches = int(np.ceil(df_test.shape[0] / batch_size)); print('n_test_batches', n_test_batches)
    # list_predict_proba = []
    # for i in range(n_test_batches):
    #     single_test_batch = next(test_gen)
    #     list_predict_proba.extend(np.array(tf_model.predict(single_test_batch)).flatten())
    
    print('\n\n\n Performance after training only the last layers 🙈😀')
    df_data_for_meta_only_new_layers = evalulate_model(list_predicted_probabilities = list_predict_proba, df_test= df_test, 
                                                       df_data_for_meta=df_data_for_meta_only_new_layers, n_th_iteration_of_training= n_th_iteration_of_training,
                                                       training_type_file_name = str_file_only_new_layers_predict_proba)
    df_data_for_meta_only_new_layers.to_excel(os.path.join(loc_performances, str_file_only_new_layers_predict_proba), index=False)









    #################### Fine-tuning last n layers ####################
    tf_base_model.trainable = True
    if str_file_model_only_new_layers in base_model_name_for_SAPIENS:
        print(f'Tuning only last {last_n_layers} layers 😀')
        fine_tune_at = len(tf_base_model.layers) - last_n_layers
        for layer in tf_base_model.layers[:fine_tune_at]:
            layer.trainable = False

    crt_lr = learning_rate_for_fine_tune
    temp_loc_models = loc_resnet_fine_tuned_model_file

    loc_dir_weights = os.path.join(loc_model_weights, str(crt_lr))
    callback = ConditionedModelCheckpointCompressed(save_dir=loc_dir_weights, tolerance= tolerance)
    optimizer =  tf.keras.optimizers.Nadam(learning_rate= crt_lr, beta_1=0.9, beta_2=0.999)

    tf_model.compile(loss=WeightedBinaryFocalLoss(w0=class_weight[0], w1=class_weight[1]), optimizer= optimizer, metrics=list_metrics)
    history = tf_model.fit(train_gen, epochs = n_epoch_for_fine_tune, steps_per_epoch= train_steps,  validation_data = val_gen, validation_steps= validation_steps, callbacks = [callback], verbose = 1)
    history.history[str_best_epoch] = callback.best_epoch; history.history[str_difference] = abs(callback.best_val_loss - callback.best_train_loss)
    save_training_history(pd.DataFrame(history.history), temp_loc_models, '__'+str(crt_lr))

    tf_model.load_weights(callback.best_weight_file)
    tf_model.save(os.path.join(loc_models, os.path.basename(loc_resnet_fine_tuned_model_file)))

    list_predict_proba = [arr[0] for arr in list(tf_model.predict(test_gen, steps = test_steps))]
    # n_test_batches = int(np.ceil(df_test.shape[0] / batch_size)); print('n_test_batches', n_test_batches)
    # list_predict_proba = []
    # for i in range(n_test_batches):
    #     single_test_batch = next(test_gen)
    #     list_predict_proba.extend(np.array(tf_model.predict(single_test_batch)).flatten())

    print('\n\n\n Performance After retrain all layers ')
    df_data_for_meta_all_layers = evalulate_model(list_predicted_probabilities = list_predict_proba, df_test=df_test, 
                                                  df_data_for_meta=df_data_for_meta_all_layers, n_th_iteration_of_training=n_th_iteration_of_training,
                                                  training_type_file_name = str_file_predict_prob_fine_tuned_resnet)
    df_data_for_meta_all_layers.to_excel(os.path.join(loc_performances, str_file_predict_prob_fine_tuned_resnet), index=False)

    ########## Clearing 😡😡😡😡 (not working as good as I expected) ##########
    K.clear_session()
    gc.collect()
    del tf_model, train_gen, val_gen, test_gen, history, tf_base_model

## Root Point of Running Models

In [ ]:
tf.keras.utils.set_random_seed(random_state)
tf.config.experimental.enable_op_determinism()

def get_last_valid_iter_to_drop_remaining_iteration(last_nth_iter, chunk_size_for_test, chunk_size_for_1_iteration):
   # We are using this function to get the last valid iteration after which all iterations will be discarded
   # last_nth_iter basically presents the last iterations including the chunk size for test

   nth_iter = int(last_nth_iter / chunk_size_for_test) + 1

   # The purpose of the int(nth_iter / chunk_size_for_1_iteration) is to drop the remainder.
   # subtracting chunk_size_for_test since in the first iteration, the value of n_th_iteration_of_training is 0. Thus, to precisely find which values of n_th_iteration_of_training is going to drop, I need to subtract.
   # e.g., let's say, values of last_nth_iter, chunk_size_for_test, chunk_size_for_1_iteration are 15, 3, and 5 respectively. 
   # We are supposed to get 12 as the last_valid_iter_to_drop_remaining_iteration
   return (chunk_size_for_test * chunk_size_for_1_iteration * int(nth_iter / chunk_size_for_1_iteration)) -  chunk_size_for_test


def develop_daily_anxiety_lebel_detect_model():
    global loc_performances, loc_models, model_run_start_time
    # WARNING
    # 1. Sometimes F1 score calculation can be incorrect

    global batch_size, last_n_layers, image_size
    global df_data_for_meta_only_new_layers, df_data_for_meta_all_layers, dict_alg_model
    dict_alg_model = dict() # WARNING: this is just a placeholder to run the train_predict_meta_leaner() inside evalulate_model()
    list_pid = sorted(df_image_path[str_p_id].unique())
    model_run_start_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    ##### Control Center - Start #####

    if str_tiles_18 in loc_root_models:
      loc_models = loc_root_models
      loc_performances = loc_root_performances
      chunk_size_for_test = 5
      chunk_size_for_val = 2
      batch_size = 32
    elif str_sapiens in loc_root_models:
      loc_models = os.path.join(loc_root_models, folder_base_model_for_SAPIENS)
      loc_performances = os.path.join(loc_root_performances, folder_base_model_for_SAPIENS)
      chunk_size_for_test = 5
      chunk_size_for_val = 2
      batch_size = 32
      
    last_n_layers = 9 # 🔥🔥🔥🔥 WARNING: I have to change it since I added Conv. layers.
    image_size = 224
    chunk_size_for_1_iteration = 5000 # After this # of iterations, the runtime will be restarted as there is a problem in the GPU of Mac . Setting it so big so that it does not cause any problem in Titan server
    iteration_counter = 0
    ##### Control Center - End #####

    # if os.path.exists(os.path.join(loc_performances, str_file_predict_prob_fine_tuned_resnet)):
    #   df_data_for_meta_only_new_layers = pd.read_excel(os.path.join(loc_performances, str_file_only_new_layers_predict_proba))
    #   df_data_for_meta_all_layers = pd.read_excel(os.path.join(loc_performances, str_file_predict_prob_fine_tuned_resnet))
    #   start_index_of_LNOCV = df_data_for_meta_all_layers.iloc[-1][str_n_th_iter_of_train] + chunk_size_for_test

    #   if df_data_for_meta_only_new_layers.shape[0] != df_data_for_meta_all_layers.shape[0]:
    #     df_data_for_meta_only_new_layers = df_data_for_meta_only_new_layers.iloc[:df_data_for_meta_all_layers.shape[0]] # Currently, we need to reset the file to remove the consumed memory. So, sometimes, only new layers have been trained and the corresponding fine tune all layers have not been executed.

    # else:
    
    start_index_of_LNOCV = 0
    df_data_for_meta_all_layers = pd.DataFrame()
    df_data_for_meta_only_new_layers = pd.DataFrame()


    # if platform.system() == 'Darwin': # Darwin means macOS 🤣🤣
    #     if current_ds == str_sapiens:
    #         chunk_size_for_1_iteration = 13
    #     else:
    #         chunk_size_for_1_iteration = 3
    
    # if (df_data_for_meta_only_new_layers.shape[0] > 0) and (platform.system() == 'Darwin'):
    #   iteration_number_to_drop = get_last_valid_iter_to_drop_remaining_iteration(last_nth_iter = df_data_for_meta_only_new_layers.iloc[-1][str_n_th_iter_of_train], chunk_size_for_test = chunk_size_for_test,
    #                                                                              chunk_size_for_1_iteration = chunk_size_for_1_iteration)
    #   df_data_for_meta_only_new_layers = df_data_for_meta_only_new_layers[df_data_for_meta_only_new_layers[str_n_th_iter_of_train] <= iteration_number_to_drop]
    #   df_data_for_meta_all_layers = df_data_for_meta_all_layers[df_data_for_meta_all_layers[str_n_th_iter_of_train] <= iteration_number_to_drop]
      
    #   start_index_of_LNOCV = df_data_for_meta_all_layers.iloc[-1][str_n_th_iter_of_train] + chunk_size_for_test
    #   print(f'\n\nRemember, we are dropping all iterations after iteration number {iteration_number_to_drop} to ensure consistency and also reproducibility.\n\n')

    for pid_index in range(start_index_of_LNOCV, len(list_pid), chunk_size_for_test):
        iteration_counter += 1
        test_pid_list = list_pid[pid_index:pid_index + chunk_size_for_test]
        loc_new_layers_model_file = os.path.join(loc_models, 'iter_'+ str(pid_index)+'__'+ test_pid_list[0]+'__chunk_test_'+str(chunk_size_for_test) +'_val_'+ str(chunk_size_for_val) +'__'+ str_file_model_only_new_layers)
        
        if str_tiles_18 in loc_models:
          loc_resnet_fine_tuned_model_file  = loc_new_layers_model_file.replace(str_file_model_only_new_layers, 'fine_tuned_all_layers.keras')
        else:
          loc_resnet_fine_tuned_model_file  = loc_new_layers_model_file.replace(str_file_model_only_new_layers, 'fine_tuned_last_'+str(last_n_layers)+ '_layers.keras')

        if os.path.exists(loc_new_layers_model_file) and os.path.exists(loc_resnet_fine_tuned_model_file):
            print(pid_index, 'Exists both new layers and fine tuned models\n')
            continue
      
        print('Started ', pid_index, ' iter_counter: ', iteration_counter, chunk_size_for_1_iteration)
        ################################# Seperating Training, Test, Validation Data ################################
        df_temp_train = df_image_path[~df_image_path[str_p_id].isin(test_pid_list)].copy()
        df_test = df_image_path[df_image_path[str_p_id].isin(test_pid_list)].copy()

        dict_train_count = dict(Counter(df_temp_train[str_actual_class].tolist()))
        train_1_to_0_class_ratio = dict_train_count.get('1') * 100 / dict_train_count.get('0')
        val_pid_list = select_pids_by_class_ratio(df= df_temp_train, class_ratio=train_1_to_0_class_ratio, n=chunk_size_for_val, tolerance=10)

        print(f'train_class_ratio_1_to_0: {train_1_to_0_class_ratio}, val_pid_list: {val_pid_list}')
        df_val = df_temp_train[df_temp_train[str_p_id].isin(val_pid_list)].copy()
        df_temp_train = df_temp_train[~ df_temp_train[str_p_id].isin(val_pid_list)]

        df_temp_train = shuffle(df_temp_train, random_state=random_state)
        df_temp_train.reset_index(inplace=True, drop=True)
        df_val = shuffle(df_val, random_state=random_state)
        df_test = shuffle(df_test, random_state=random_state)
        print('Classes in training dataset: ', Counter(df_temp_train[str_actual_class]), 'Classes in validation dataset: ', Counter(df_val[str_actual_class]), 'Classes in test dataset: ', Counter(df_test[str_actual_class]))
        ###### df_temp_train = df_temp_train.iloc[:250]
        
        if str_tiles_18 in loc_models:
          train_val_test_save_model(df_temp_train= df_temp_train, df_val= df_val, df_test = df_test, loc_new_layers_model_file = loc_new_layers_model_file, 
                                    loc_resnet_fine_tuned_model_file = loc_resnet_fine_tuned_model_file, n_th_iteration_of_training = pid_index)
        elif str_sapiens in loc_models:
           SAPIENS_train_val_test_save_model(df_temp_train= df_temp_train, df_val= df_val, df_test = df_test, loc_new_layers_model_file = loc_new_layers_model_file, 
                                             loc_resnet_fine_tuned_model_file = loc_resnet_fine_tuned_model_file, n_th_iteration_of_training = pid_index)
        
        print('Finished at ', datetime.now()) # Running this to make sure that a notebook (in case we schedule usig time) has started working after completing another one.
        if iteration_counter ==  chunk_size_for_1_iteration:
           os._exit(00)


def check_condition_based_strings_to_be_matched():
   bool_all_conditions__true = True
   for local_condition in [crt_state_anxiety_classification_approach]:   
      if local_condition not in appended_conditions_to_be_matched:
         bool_all_conditions__true = False
         break
   return bool_all_conditions__true

sample_file = df_image_path[str_filename].iloc[0]


if bool_runing_first_time_after_restart and (current_ds in sample_file.split('/')[-3]) and check_condition_based_strings_to_be_matched(): # to ensure that we have run the script containing image path. Otherwise, findings of one dataset will be added to another dataset.
  bool_runing_first_time_after_restart = False

  if current_ds == str_sapiens:
    list_teacher_models = get_teacher_models()

    for single_base_model_name in list_teacher_models:
      base_model_name_for_SAPIENS = single_base_model_name # do not replace single_base_model_name by base_model_name_for_SAPIENS since base_model_name_for_SAPIENS value will not be accessible from other functions in that case.
      folder_base_model_for_SAPIENS = single_base_model_name.replace('.keras', '')
      loc_model_weights = os.path.join(loc_root_models, 'Model_weights', folder_base_model_for_SAPIENS); print(folder_base_model_for_SAPIENS)
      develop_daily_anxiety_lebel_detect_model()
      os._exit(00) # to make sure that SAPIENS models based on every teacher model gets same resources, I am running this statement. This will help us to understand which teacher model is going to perform best and explore afterwards why
  else:
     develop_daily_anxiety_lebel_detect_model()

elif bool_runing_first_time_after_restart:
   print('Please, run the codes to create the dataframe containing both image data paths and labels and make sure all conditions are satisfied (e.g., if you plan to run the models based on median approach, codes regarding the changing the outcome variable also must have to run). The current image data is for a different dataset (e.g., image data is of tiles, but current approach to develop models is based on sapiens) :)')
else:
   print('Please, restart the kernel to ensure reproducibility of the findings.')
   os._exit(00)

# `SenseImage`

In [ ]:
from matplotlib.colors import BoundaryNorm
import matplotlib.pyplot as plt
import plotly.express as px
from scipy.signal import stft

# Initially, I thought to compare the previous day's data to decide whether the following day is more or NOT more anxious day.
# Later, visualizing several participants' anxiety data, I found anxiety data remains same over the day for several days (e.g., anxiety score 2). Thus, finding the difference may not show a true change (e.g., anxiety level 2-2 or 3-3) of anxiety level.
# Approach to compare anxiety level in compare_anxiety_label_days()
# 1. group by id
# 2. sort values by date
# 3. calculate the difference sequemtially. That is, (current day's anxiety score) - (previous day's anxiety score) # e.g., ot_temp_df = pd.DataFrame({'col1': [1, 3, 6, 10, 15]}); print(ot_temp_df['col1'].diff())
# 4. Since the for the first day, there is no previous day 😀. I am dropping that day.
# 5. If the current day has a higher score than previous day, the difference will be positive (i.e., > 0)
# 6. Concating the dataframe per participant

def compare_anxiety_label_days(df, pid_col, date_col):
    df_temp = pd.DataFrame()
    for pid, group in df.groupby(pid_col): # note: 94144e0e-7ea9-4b65-96a4-86e9741d269d  p is going to be removed in group.iloc[1:] due to having only 1 data. That is fine.
        group = group.sort_values(by=date_col)
        group[str_category] = group[str_anxiety].diff() # dff(): (current day's anxiety score) - (previous day's anxiety score)
        group = group.iloc[1:]
        group[str_category] = (group[str_category] > 0).astype(int)
        df_temp = pd.concat([df_temp, group])
        
    return df_temp

def visualize_each_participant_anxiety_data(df_anxiety):
    for pid, group in df_anxiety.groupby(str_tiles_pid):
        print(pid)
        px.scatter(x=group[str_date].tolist(), y = group[str_anxiety].tolist()).show()

In [ ]:
####### WARNING: Start of Control Center for Spectrogram generation ######
HR_min = 40
HR_max = 200
nperseg = 2
color_map = 'viridis'
color_shading = 'gouraud'
tiles_18_hr_sampling_rate = 0.2
actual_image_width = 224
actual_image_height = 224
####### WARNING: End of Control Center for Spectrogram generation ######

sample_data_min = np.full(10000, HR_min)
sample_data_max = np.full(10000, HR_max)
f_min, t_min, Zxx_min = stft(sample_data_min, fs= tiles_18_hr_sampling_rate, nperseg= nperseg)
magnitude_min = np.abs(Zxx_min) # WARNING: if you change the function (i.e., np.abs()) here, do not forget to change for magnitude_max and also inside create_spec_save()
f_max, t_max, Zxx_max = stft(sample_data_max, fs= tiles_18_hr_sampling_rate, nperseg= nperseg)
magnitude_max = np.abs(Zxx_max)

estimated_vmax = max(magnitude_min.max(), magnitude_max.max())
estimated_vmin = min(magnitude_min.min(), magnitude_max.min())

MIN_Number_of_Samples = 3
vmin = 40 # WARNING: even though estimated_vmin will be 0, manually, checking the values, I see only 2 (nperseg = 2) values are 0 (see the file ... removed location for better privacy ...SAPIENS/Anxiety/Notebooks/output.txt). I think it is due to this reason "For nperseg=2, Hann produces values [0,1][0,1]. The sample is multiplied by 00, yielding 0 in the windowed signal (thus showing up as zero in the STFT magnitude)."
vmax = estimated_vmax
print(f'vmin: {vmin}, vmax: {vmax}')

bin_width = 1
bins = np.arange(vmin, vmax + bin_width, bin_width)
num_bins = len(bins) - 1
cmap = plt.get_cmap(color_map, num_bins)
norm = BoundaryNorm(bins, cmap.N)

def create_spec_save(list_data, fs, loc_fig):
    f, t, Zxx = stft(list_data, fs=fs, nperseg=nperseg) # f: Array of sample frequencies, t: Array of segment times.
    absolute_magnitude = np.abs(Zxx) # WARNING: if you change the function (i.e., np.abs()) here, do not forget to change it while calculating magnitude_min and magnitude_max which are used for color map and normalization

    plt.ioff()
    plt.figure(figsize=(2.24, 2.24))

    pcm = plt.pcolormesh(t, f, absolute_magnitude, shading= color_shading, cmap=cmap, norm=norm) # t in x and f in y axes. Followed scipy doc mostly: plt.pcolormesh(t, f, np.abs(Zxx), vmin=0, vmax=amp, shading='gouraud'); https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.stft.html
    ###  plt.colorbar(pcm, label='')

    ax = plt.gca() # gca:"Get the current Axes" 😀
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(left=False, bottom=False)

    plt.subplots_adjust(left=0, right=1, top=1, bottom=0) # The reason for setting right=1, top=1: When I set top=0 and right=0: ValueError: left cannot be >= right & ValueError: bottom cannot be >= top
    plt.savefig(loc_fig, pad_inches=0)
    plt.close()

In [ ]:
from rqa_for_other_sensors import hrv_rqa_for_other_sensors


def get_rri_rri_time(list_sense_data):
    list_rri = 60 / np.array(list_sense_data)
    list_rri_time = np.nancumsum(list_rri)
    return list_rri, list_rri_time


def custom_diff_absolute_for_step_data(df_step): # Currently, we are saving the aggregated steps. Thus, to understand whether there is any change in the number of steps, we are subtracting.
    df_step.sort_values(by=['T'], ascending=True).reset_index(inplace=True)

    series = df_step[str_count].copy()
    diff_series = series.diff().abs() # taking abolute value since TILES-18 reports the number of steps. So, aligjing the base model and target task
    result = series.copy()
    
    # print(diff_series)

    # For several cases of several participants, the aggregated value became lower than the last probe starting time (which can be due to watch's internal issues or can be due to switching off the watch or can be due to resettinng)
    # The aggregated value of steps can not be lower in a probe starting time than the previous probe starting time
    # In those cases (series.iloc[i-1] > series.iloc[i]), we did not calculate the difference. Instead, we took the value as it is.
    # One reason is that by calculating the difference, we want to know how many steps there were since the last one.
    # By taking the value as it is, we may get # of steps from series.iloc[i] presenting the number of steps since the last probe starting time considering the lower value of steps can be due to resetting.
    for i in range(1, len(series)): # Do NOT set range (0, ) instead of 1. If you set 0, you will get the last value of the dataframe and comparision (if series.iloc[i-1] > series.iloc[i]:) with the last value is not meaningful.
        if series.iloc[i-1] > series.iloc[i]:
            result.iloc[i] = series.iloc[i]
        else:
            result.iloc[i] = diff_series.iloc[i]

    df_step[str_StepCount] = result
    return df_step.iloc[1:] # dropping the first row since we 


def process_heart_rate(df_sensor_data, df_demo, pid_col, pid, name_sensor_data_col):
    pid = pid.replace('.csv.gz', '').replace('P', '')
    max_hr = 220 - get_age_of_pid(df_demo= df_demo, pid_col= pid_col, pid= pid)
    df_sensor_data = df_sensor_data[(df_sensor_data[name_sensor_data_col] >= HR_min) & (df_sensor_data[name_sensor_data_col] <= max_hr)]
    return df_sensor_data


def get_age_of_pid(df_demo, pid_col, pid):
    pid_age_list = df_demo[df_demo[pid_col] == pid][str_age].tolist()
    if len(pid_age_list) > 1:
        raise ValueError('Severe problem in age data. Please, check')
    return pid_age_list[0]


def get_image_width_height():
    if save_fig_for_paper:  # for getting the ticks in x and y axis for save_fig_for_paper, a size higher than 2.24 works good
        image_width = 4.50
        image_height = 4.50
    elif (str_modal_type == str_multi_modal):
        image_width = actual_image_width / (number_of_modals * 100) # 224 / 100 = 2.24
        image_height = actual_image_height / (number_of_modals * 100)
    else:
        image_width = actual_image_width / 100
        image_height = actual_image_height / 100
        
    return image_width, image_height


# We are filtering to the nearest minute regardless whether we calculate the IV by resampling to minute or hour. The reason is filtering by minute will help us to be precise.
# However, filtering by seconds may not work since the date-time of the files (e.g., _minuteCaloriesNarrow_20230101_20240509) containing data per minute data are like this  3/22/2023  12:06:00 AM. Thus, if we focus to filetr by seconds, we have to ensure that we get data exactly at 12:06:00 which can be unlikely to happen due to not having exact sampling rate in Fitbit.
def filter_by_heart_rate_ref_as_wear_status(df_ref, df_target, datetime_col, dt_pattern_ref, dt_pattern_target):
    df_ref[datetime_col] = pd.to_datetime(df_ref[datetime_col], format = dt_pattern_ref)
    df_target[datetime_col] = pd.to_datetime(df_target[datetime_col], format = dt_pattern_target)

    df_ref['truncated'] = df_ref[datetime_col].dt.floor('T') # T: to floor to the nearest minute; step data was collected in minutes interval. TILES-18 paper: "In contrast to heart rate values, step count data is sampled with an interval of 1min, and reports the number of steps taken within that minute."
    df_target['truncated'] = df_target[datetime_col].dt.floor('T')

    valid_times = df_ref['truncated'].unique()
    filtered_df = df_target[df_target['truncated'].isin(valid_times)]
    filtered_df = filtered_df.drop(columns=['truncated'])

    return filtered_df


# Currently, we are relying on a fixed sampling rate (based on the first participant's all days' data) to ensure the consistency in the generated images
def calculate_sampling_rate_for_TILES_18(loc_sensor_data, pid, timestamp_col):
    df_sensor_data = pd.read_csv(os.path.join(loc_sensor_data, pid), compression='gzip')
    df_sensor_data[timestamp_col] = pd.to_datetime(df_sensor_data[timestamp_col])
    df_sensor_data.sort_values(by=timestamp_col, inplace=True)
    time_deltas = df_sensor_data[timestamp_col].diff().dropna().dt.total_seconds()

    return 1 / time_deltas.mean()


def get_segmented_data(list_data):
    number_of_samples_in_a_segment = int(len(list_data) / n_segments_of_a_day)

    list_segment_name = []
    list_segmented_data = []
    last_index = 0

    for n_th_segment in range(1, n_segments_of_a_day+1):
        list_segment_name.append(str_segment+ str(n_th_segment))
        list_segmented_data.append(list_data[last_index: last_index + number_of_samples_in_a_segment])
        last_index = last_index + number_of_samples_in_a_segment
    
    return zip(list_segment_name, list_segmented_data)



def plot_nested_list_with_boundaries(nested_list, loc_to_save_figure):
    flat_data = [item for sublist in nested_list for item in sublist]
    boundaries = []
    current_index = 0

    list_date_time = list(range(len(flat_data))) # if you want to show the real-time date time values, add a parameter in the function to pass the date-time and comment out this line.
    
    for sublist in nested_list[:-1]:  
        current_index += len(sublist)
        boundaries.append(current_index-1)

    fig = go.Figure()
    fig.add_trace(go.Scatter(y=flat_data, x=list_date_time, mode='markers+lines', line=dict(color='black', width=2), marker=dict(color='black', size=6), name=''))

    for b in boundaries:
        fig.add_shape(type='line', x0=list_date_time[b], x1=list_date_time[b], y0=min(flat_data), y1=max(flat_data), line=dict(color='red', width=2, dash='dash'))

    fig.update_layout(title='', xaxis_title = 'Sample number', yaxis_title = sensor_data_col.capitalize(), plot_bgcolor = '#FFFFFF', font_family = "Times New Roman")
    fig.update_layout(margin=dict(l=0, r=0, t=0, b=0),
                      xaxis=dict(tickfont=dict(size=20), constrain='domain'),
                      yaxis=dict(tickfont=dict(size=20), constrain='domain'),
                      width=800,
                      height=300)
    
    fig.write_image(os.path.join(loc_RAW_sensor_image, os.path.basename(loc_to_save_figure)))


def encode_sensing_data_for_each_p(sub_df_data, row, list_already_done_pid_date, name_sensor_data_col, pid_col, pid, column_to_make_the_file_name_unique, sampling_rate):
    global image_width, image_height, min_number_of_samples_for_image, dimension_for_rqa, n_day_missing, n_less_than_min_length_hr_data_list

    if sub_df_data.shape[0] == 0:
        n_day_missing += 1
        return

    if save_fig_for_paper:
        temp_figure_name_for_paper = str(row[str_category])+'__'+ row[pid_col] +'__'+ str(row[column_to_make_the_file_name_unique]) +'__'+crt_sensor.replace('Smartwatch_', '').replace('Datum', '') +'__'+name_sensor_data_col +'.png'
        temp_list_sensing_data_for_paper_fig = list(zip(*get_segmented_data(sub_df_data[name_sensor_data_col].tolist())))[1]

        if len(temp_list_sensing_data_for_paper_fig[0]) >= min_number_of_samples_for_image:
            plot_nested_list_with_boundaries(temp_list_sensing_data_for_paper_fig, os.path.join(loc_sensor2image, temp_figure_name_for_paper))
    
    for segment, list_sense_data in get_segmented_data(sub_df_data[name_sensor_data_col].tolist()):
        crt_id_date = str(row[str_category])+'__'+ row[pid_col] +'__'+ str(row[column_to_make_the_file_name_unique]) + '__'+segment+'__'+crt_sensor.replace('Smartwatch_', '').replace('Datum', '') +'__'+name_sensor_data_col +'.png'
        if crt_id_date in list_already_done_pid_date:
            continue
        
        # # # if len(list_sense_data) == 72289: # for P457 accelerometer split 2
        # # #     list_sense_data = list_sense_data[:30015] # I ran with the sensing data of 72289 samples in Mac, Windows, and Ubuntu with high amount o fresources. However, in none of the platforms, I was able to generate images. Every where, the system crashed.
            
        list_already_done_pid_date.append(crt_id_date)
        print(f"{pid}, # of data: {len(list_sense_data)}")
        list_sense_data = np.array(list_sense_data)

        if len(list_sense_data) < min_number_of_samples_for_image: # Checking numnber of samples of P413, I see there is at least 50 samples in almost all probe starting times. Thus, I wanted to create a spectrogram of a segment of a day even if there is data of a single probe satrting times. 
            n_less_than_min_length_hr_data_list += 1
            continue

        if 'heart' in crt_sensor.lower():
            # For spectrogram: Manually checking the file ... removed the location a bit for privacy .../Documents/SA39/SAPIENS/State Anxiety/Transfer_learning/ResNet/Others/Sensor2Image_Got_Good_Performance/0__0bf54591-e0ac-47ba-9b43-31ff54dfcee8__2018-06-05__0_11.png which was just an empy file, I found there was just 1 sample in list_sense_data ([70.])  and thus "nperseg = 2 is greater than input length  = 1," - UserWarning
            list_rri, list_rri_time = get_rri_rri_time(list_sense_data= list_sense_data)
            hrv_rqa({'RRI': list_rri * 1000, 'RRI_time': list_rri_time}, sampling_rate= sampling_rate, show=True,
                    image_length = image_width, image_height= image_height, loc_figure_to_save=os.path.join(loc_sensor2image, crt_id_date),
                    save_fig_for_paper = save_fig_for_paper)                            
        else:
            hrv_rqa_for_other_sensors(rri= list_sense_data, sampling_rate= sampling_rate, dimension=dimension_for_rqa, show=True, 
                                        image_length = image_width, image_height= image_height, loc_figure_to_save = os.path.join(loc_sensor2image, crt_id_date),
                                        save_fig_for_paper = save_fig_for_paper)
            

def get_sub_sensor_df_based_on_window(df_sensor_data, anxiety_data_row):
    if (crt_anxiety_type == str_daily_anxiety):
        column_to_make_the_file_name_unique = str_timestamp # Though currently, they are same in both conditions. They can be useful, if we focus on something like date.
        sub_df_data = df_sensor_data[(df_sensor_data[str_date] == anxiety_data_row[str_date]) & (df_sensor_data[str_timestamp] < anxiety_data_row[str_timestamp])].copy()
    elif (crt_anxiety_type == str_state_anxiety):
        column_to_make_the_file_name_unique = str_timestamp
        last_time_of_window = anxiety_data_row[str_timestamp]

        if (current_ds == str_tiles_18) and (crt_sensor == str_tiles_sensor_sleep_data): # This will run if we develop the TILES dataset based models selecting a window size for state anxiety
            df_sensor_data.sort_values(by=[str_timestamp], inplace=True, ascending = True)

            sleep_last_data_timestamp = df_sensor_data.iloc[-1][str_timestamp]
            if sleep_last_data_timestamp >= anxiety_data_row[str_timestamp]: # Think in this way: sleep_last_data_timestamp == 7 AM equivalent timestamp; anxiety_data_row[str_timestamp] == 6 AM timestamp
                last_time_of_window = anxiety_data_row[str_timestamp]
            else: # Think in this way: sleep_last_data_timestamp == 5 AM equivalent timestamp; anxiety_data_row[str_timestamp] == 6 PM timestamp
                last_time_of_window = sleep_last_data_timestamp
        
        start_time_of_window = last_time_of_window - (window_size_in_HOUR * HOUR_in_Sec)
        sub_df_data = df_sensor_data[(df_sensor_data[str_timestamp] >= start_time_of_window) & (df_sensor_data[str_timestamp] < last_time_of_window)].copy()
    
    return sub_df_data, column_to_make_the_file_name_unique


def convert_TILES_18_sensing_data_to_image(loc_sensor_data, loc_sensor2image, name_sensor_data_col, timestamp_col, pid_col, df_demo, df_anxiety):
    global image_width, image_height, min_number_of_samples_for_image, dimension_for_rqa, n_day_missing, n_less_than_min_length_hr_data_list
    
    list_pid_data = sorted(set(os.listdir(loc_sensor_data))) # Checked: with set() and without set() results in same length which is fine. There should be one file per pid
    list_already_done_pid_date = sorted(os.listdir(loc_sensor2image))
    sampling_rate = calculate_sampling_rate_for_TILES_18(loc_sensor_data, list_pid_data[0], timestamp_col)
    n_day_missing = 0
    n_less_than_min_length_hr_data_list = 0
    image_width, image_height = get_image_width_height()
    
    str_dt_format_in_step_data = '%Y-%m-%dT%H:%M:%S.%f'
    str_dt_format_in_hr_data = '%Y-%m-%dT%H:%M:%S.%f'

    for pid in list_pid_data:
        if ('README' in pid) or ('DS_Store' in pid):
            continue
        
        df_sensor_data = pd.read_csv(os.path.join(loc_sensor_data, pid), compression='gzip')
        if str_tiles_sensor_step_count in crt_sensor:
            df_hr = pd.read_csv(os.path.join(pathlib.Path(loc_sensor_data).parent, 'heart-rate', pid), compression='gzip')
            df_sensor_data = filter_by_heart_rate_ref_as_wear_status(df_ref= df_hr, df_target= df_sensor_data, datetime_col= str_timestamp, dt_pattern_ref= str_dt_format_in_hr_data, dt_pattern_target= str_dt_format_in_step_data)

        if str_tiles_sensor_sleep_data in crt_sensor:
            df_sensor_data = df_sensor_data[df_sensor_data[str_level].isin(list_sleep_stages)]
            df_sensor_data[str_level] = df_sensor_data[str_level].map(dict_sleep_stage_encode)
        
        if 'heart' in crt_sensor.lower():
            df_sensor_data = process_heart_rate(df_sensor_data, df_demo, pid_col, pid, name_sensor_data_col) ## In case of SAPIENS, there are 0 value in the first row.
        
        
        if crt_sensor in list_tiles_sensor_with_low_sampling:
            min_number_of_samples_for_image = MIN_Number_of_Samples # the min_number_of_samples_for_image is different for HR. I tried to ensure the consistency of the image quality. Having 3 samples for HR in an image will be an outlier since harrdly, there was a day (based on my observation on data quality) when we had 3 samples or number of samples closer to that.
            dimension_for_rqa = 2
        else:
            min_number_of_samples_for_image = 50 
            dimension_for_rqa = 7 # the default dimension in neurokit
        
        if df_sensor_data.shape[0] == 0:
            continue
        
        df_sensor_data[str_timestamp] = pd.to_datetime(df_sensor_data[timestamp_col]) # str_timestamp column contains values like 2018-04-09T00:00:02.000
        df_sensor_data[str_date] = df_sensor_data[str_timestamp].dt.date
        df_sensor_data[str_timestamp] = df_sensor_data[str_timestamp].apply(lambda x: x.timestamp())
        df_sensor_data.dropna(inplace=True)
        
        pid = pid.replace('.csv.gz', '')
        sub_df_anxiety = df_anxiety[df_anxiety[pid_col] == pid].copy()
        n_same_data_problem = 0 # same data for consecutive state anxiety. Due to not having variation, currently, we are discarding these data.

        for index, row in sub_df_anxiety.iterrows():
            sub_df_data, column_to_make_the_file_name_unique =  get_sub_sensor_df_based_on_window(df_sensor_data= df_sensor_data, anxiety_data_row= row)
            
            # # if len(set(sub_df_data[name_sensor_data_col].tolist())) == 1:
            # #     n_same_data_problem += 1
            # #     continue 
                       
            encode_sensing_data_for_each_p(sub_df_data= sub_df_data, row= row, list_already_done_pid_date = list_already_done_pid_date, name_sensor_data_col= name_sensor_data_col, 
                                           pid_col= pid_col, pid= pid, column_to_make_the_file_name_unique = column_to_make_the_file_name_unique, sampling_rate = sampling_rate)
    
    print(f'\n\n\n n_day_missing: {n_day_missing}, n_less_than_min_length_hr_data_list: {n_less_than_min_length_hr_data_list}') # Jan. 4, 2025:  n_day_missing: 526, n_less_than_min_length_hr_data_list: 843

## Encode Sensor Data in an Image

In [ ]:
# Updates:
# Took steps to resize the figure to match with the input shape (224 * 224; 2.24 * dpi 100) of ResNet
#   1. set dpi explicitly to 100 and removed bbox_inches='tight' which helped to get the image size 224*224 from 237 × 232. Setting the dpi to 100 may not be potential since the default dpi is 100
#   2. Removed the colorbar (plt.colorbar(pcm, label='')) since I am using the same colorbar and also normalizing the data using cmap and norm parameters. Due to using cmap and norm, the color bar will be same for all participants. I checked that manually as well. 
#       2.1. This helped to reduce the file size by ~50% (~6 KB to ~3/4KB) [WARNING: for both cases (6 vs. 3/4) sometimes the figure size can be significantly higher depending on the spectrogram].
#       2.2. Both step 1 (removed bbox_inches='tight') and 2 helped to reduce the file size at least 50% (~9 KB to ~3/4KB). This is supposed to reduce the training time and also may be model size.
#   3. set minimum number of required samples to 50 to create a spectrogram


def SAPIENS_pre_process_sensor_data(df_sensor_data, df_demo, name_sensor_data_col, timestamp_col, pid_col, pid):
    if 'Heart' in crt_sensor:
        df_sensor_data = process_heart_rate(df_sensor_data, df_demo, pid_col, pid, name_sensor_data_col) ## In case of SAPIENS, there are 0 value in the first row.
    elif crt_sensor in list_IMU_sensor:
        df_sensor_data[str_magnitude] = np.sqrt(df_sensor_data['X']**2 + df_sensor_data['Y']**2 + df_sensor_data['Z']**2)
    
    # if crt_sensor in list_sensor_high_freq:
    #     df_sensor_data = df_sensor_data[['T', sensor_data_col]]
    #     df_sensor_data.set_index('T', inplace=True)

    #     df_sensor_data = df_sensor_data.resample('1S').mean()
    #     df_sensor_data.reset_index(inplace=True)

    df_sensor_data[str_timestamp] = df_sensor_data[timestamp_col].apply(lambda x: x.timestamp())
    df_sensor_data[str_hour] = df_sensor_data[timestamp_col].dt.hour
    df_sensor_data[str_date] = df_sensor_data[timestamp_col].dt.date
    df_sensor_data.dropna(inplace=True)

    return df_sensor_data

def convert_SAPIENS_sensing_data_to_image(loc_sensor_data, loc_sensor2image, name_sensor_data_col, timestamp_col, pid_col, df_demo, df_anxiety):
    global image_width, image_height, min_number_of_samples_for_image, dimension_for_rqa, n_day_missing, n_less_than_min_length_hr_data_list

    list_pid_data = sorted(set(os.listdir(loc_sensor_data))) # Checked: with set() and without set() results in same length which is fine. There should be one file per pid
    list_already_done_pid_date = sorted(os.listdir(loc_sensor2image))
    ### sampling_rate = calculate_sampling_rate_for_TILES_18(loc_sensor_data, list_pid_data[-1], timestamp_col)
    ### sampling_rate = 1 # using the sampling rate of 1 Hz since we are resampling data of all the sensors having a sampling rate higher than 1 Hz. Hardly (maybe, less than 0.1% times, the )
    n_day_missing = 0
    n_less_than_min_length_hr_data_list = 0

    image_width, image_height = get_image_width_height()

    for pid in sorted(list_pid_data, reverse=True):
        if 'README' not in pid and 'DS_Store' not in pid:
            df_sensor_data = pd.read_pickle(os.path.join(loc_sensor_data, pid, crt_sensor+'.pkl'))
            df_sensor_data = SAPIENS_pre_process_sensor_data(df_sensor_data= df_sensor_data, df_demo= df_demo, name_sensor_data_col= name_sensor_data_col, 
                                                             timestamp_col= timestamp_col, pid_col= pid_col, pid= pid)
            
            #### 🔥🔥🔥🔥🔥🔥🔥
            #### WARNING: remember, I have to check the SAPIENS time-zone. Similarly, I have to check the time zone of SubmissionTimestamp of Sensus
            #### Also, in this statement (df_sensor_data[timestamp_col] = pd.to_datetime(df_sensor_data[timestamp_col])), you are not making the date-time tz aware. Have you already made it tz aware?
            ##### Also, think about the sampling rate. The sampling rate was 33 initially for Acc, but, later we changed it to 18 Hz or something like that.
            #### 🔥🔥🔥🔥🔥🔥🔥
            #### 🔥🔥🔥🔥🔥🔥🔥

            if crt_sensor in str_sensor_step:
                df_sensor_data = custom_diff_absolute_for_step_data(df_sensor_data)

            if (crt_sensor in list_sensor_sampling_rate_1) or ((crt_anxiety_type == str_state_anxiety) and (crt_sensor in list_sensor_relatively_LOWER_sampling_rate)):
                min_number_of_samples_for_image = MIN_Number_of_Samples # the min_number_of_samples_for_image is different for HR. I tried to ensure the consistency of the image quality. Having 3 samples for HR in an image will be an outlier since harrdly, there was a day (based on my observation on data quality) when we had 3 samples or number of samples closer to that.
                dimension_for_rqa = 2
            else:
                min_number_of_samples_for_image = 50 
                dimension_for_rqa = 7 # the default dimension in neurokit
            
            pid = pid.replace('P', '')
            sub_df_anxiety = df_anxiety[df_anxiety[pid_col] == pid].copy()
            n_same_data_problem = 0 # same data for consecutive state anxiety. Due to not having variation, currently, we are discarding these data.

            for index, row in sub_df_anxiety.iterrows():
                sub_df_data, column_to_make_the_file_name_unique=  get_sub_sensor_df_based_on_window(df_sensor_data= df_sensor_data, anxiety_data_row= row)
                
                # # if len(set(sub_df_data[name_sensor_data_col].tolist())) == 1:
                # #     n_same_data_problem += 1
                # #     continue
                
                encode_sensing_data_for_each_p(sub_df_data= sub_df_data, row= row, list_already_done_pid_date= list_already_done_pid_date, name_sensor_data_col= name_sensor_data_col, 
                                               pid_col= pid_col, pid= pid, column_to_make_the_file_name_unique= column_to_make_the_file_name_unique, sampling_rate = dict_sensor_sampling_rate.get(crt_sensor))



    print(f'\n\n\n n_day_missing: {n_day_missing}, n_less_than_min_length_hr_data_list: {n_less_than_min_length_hr_data_list}') # Jan. 4, 2025:  n_day_missing: 526, n_less_than_min_length_hr_data_list: 843

name_dataset_source = loc_sensor2image.split('/')[-2]

print(f'Dataset {name_dataset_source, crt_sensor}\n\n\n')

if str_tiles_18 in name_dataset_source:
    convert_TILES_18_sensing_data_to_image(loc_sensor_data=loc_tiles_sensor_data, loc_sensor2image=loc_sensor2image,name_sensor_data_col= sensor_data_col, 
                                           timestamp_col= dict_tiles_sensor_timestamp_col.get(crt_sensor), pid_col=str_tiles_pid, 
                                           df_demo = get_TILES_18_cleand_anxiety_demo_df(crt_state_anxiety_classification_approach).get(str_demograpnic_info),
                                           df_anxiety= get_TILES_18_cleand_anxiety_demo_df(crt_state_anxiety_classification_approach).get(str_anxiety))
elif str_sapiens in name_dataset_source:
    df_SAPIENS_demo = get_cleaned_SAPIENS_demographic_info()
    if crt_anxiety_type == str_daily_anxiety:
        df_anxiety = get_cleaned_eod_SAPIENS()
    elif crt_anxiety_type == str_state_anxiety:
        df_anxiety = get_cleaned_state_anxiety_SAPIENS(crt_state_anxiety_classification_approach)
    
    convert_SAPIENS_sensing_data_to_image(loc_sensor_data= loc_sapiens_ds, loc_sensor2image=loc_sensor2image, name_sensor_data_col= sensor_data_col, 
                                          timestamp_col='T', pid_col=str_p_id, df_demo= df_SAPIENS_demo, df_anxiety= df_anxiety)
else:
    print(f'Error. Check current_ds and also loc_sensor2image variables: {current_ds, loc_sensor2image}')

## `Sensor Fusion in Input Space`

In [ ]:
from PIL import Image

def combine_four_images(image_paths, output_path):
    if len(image_paths) != 4:
        raise ValueError("image_paths must be a list of exactly 4 image file locations.")
    
    imgs = [Image.open(path) for path in image_paths]
    for i, img in enumerate(imgs):
        if img.size != (112, 112):
            raise ValueError(f"Image at index {i} is not 112×112: {img.size}")

    # Create a blank 224×224 image for the 2×2 grid
    # Use the same mode as the first image (e.g., 'RGB')
    mode = imgs[0].mode
    combined_img = Image.new(mode, (224, 224))

    combined_img.paste(imgs[0], (0, 0)) # top-left
    combined_img.paste(imgs[1], (112, 0)) # top-right
    combined_img.paste(imgs[2], (0, 112)) # bottom-left
    combined_img.paste(imgs[3], (112, 112)) # bottom-right
    
    combined_img.save(output_path)

    print(f"Combined image saved to: {output_path}")


def sensor_fusion_in_input_for_tiles():
    list_sensors_to_fuse = ['Tiles18_'+ str_tiles_sensor_heart_rate +'_'+ str_HeartRatePPG ,
                            'Tiles18_'+ str_tiles_sensor_step_count +'_'+ str_StepCount, 
                            'Tiles18_'+ str_tiles_sensor_sleep_data +'_'+ str_level,
                            'Tiles18_'+ str_tiles_sensor_sleep_data +'_'+ str_seconds]
    name_fused_folder = '_'.join(list_sensors_to_fuse)

    hr_data_folder_name = 'Tiles18_heart-rate_HeartRatePPG'
    counter = 0

    for hr_file in os.listdir(os.path.join(pathlib.Path(loc_root_tl).parent, hr_data_folder_name, str_dataset)):
        list_images_to_fuse = []
        bool_FUSE = True

        for sensor_name in list_sensors_to_fuse:
            loc_1_sensor_to_fuse = os.path.join(os.path.join(pathlib.Path(loc_root_tl).parent, # Finding the root location so that we can get files of another sensor
                                                             sensor_name, str_dataset , hr_file). # hr_file presents the file of a segment of hr data
                                                             replace(str_tiles_sensor_heart_rate, sensor_name.split('_')[1]) # replacing heart rate sensor by the "name of the sensor" for which we are searching the file
                                                             ).replace(str_HeartRatePPG, sensor_name.split('_')[-1]) # replacing heart rate data name by the "data name of the sensor" for which we are searching the file
            if not os.path.exists(loc_1_sensor_to_fuse): # if the file is not available for a single sensor, we are not checking any more since currently, to have a fused input space for a segment of the data, we need data of all 4 sensors
                bool_FUSE = False
                break
            list_images_to_fuse.append(loc_1_sensor_to_fuse)
        
        if bool_FUSE:
            counter += 1
            print('Fuse', counter)
            combine_four_images(image_paths = list_images_to_fuse, output_path = os.path.join(loc_sensor2image, hr_file.replace(str_tiles_sensor_heart_rate, name_fused_folder).replace(str_HeartRatePPG, '')))

def clean_name(sensor_name):
    return sensor_name.replace('Smartwatch_', '').replace('Datum', '')

from itertools import combinations
def sensor_fusion_in_input_for_sapiens():
    for sensor_comb in list_sensors_combination:
        print(sensor_comb)

        list_sensors_to_fuse = [current_ds +'_'+ clean_name(sensor_comb[0]) +'_'+ dic_sensor_data_name.get(sensor_comb[0])[0],
                                current_ds +'_'+ clean_name(sensor_comb[1]) +'_'+ dic_sensor_data_name.get(sensor_comb[1])[0], 
                                current_ds +'_'+ clean_name(sensor_comb[2]) +'_'+ dic_sensor_data_name.get(sensor_comb[2])[0],
                                current_ds +'_'+ clean_name(sensor_comb[3]) +'_'+ dic_sensor_data_name.get(sensor_comb[3])[0]] # [current_ds +'_'+ clean_name(str_sensor_heart_rate) +'_'+ str_hr , current_ds +'_'+ clean_name(str_sensor_lin_acc) +'_'+ str_magnitude, current_ds +'_'+ clean_name(str_sensor_step) +'_'+ str_count, current_ds +'_'+ clean_name(str_sensor_battery) +'_'+ str_capital_l_level]
        
        name_fused_folder = get_fused_sensor_name(sensor_comb)
        loc_multi_modal_ds_save = os.path.join(pathlib.Path(pathlib.Path(loc_sensor2image).parent).parent, name_fused_folder, str_dataset)
        if not os.path.exists(loc_multi_modal_ds_save):
            os.makedirs(loc_multi_modal_ds_save)

        hr_data_folder_name = 'SAPIENS_HeartRate_HR'
        counter = 0

        for hr_file in os.listdir(os.path.join(pathlib.Path(loc_root_tl).parent, hr_data_folder_name, str_dataset)):
            list_images_to_fuse = []
            bool_FUSE = True

            for sensor_name in list_sensors_to_fuse:
                loc_1_sensor_to_fuse = os.path.join(os.path.join(pathlib.Path(loc_root_tl).parent, 
                                                                sensor_name, str_dataset , hr_file).
                                                                replace(clean_name(str_sensor_heart_rate), sensor_name.split('_')[1])).replace(str_hr, sensor_name.split('_')[-1])
                if not os.path.exists(loc_1_sensor_to_fuse):
                    bool_FUSE = False
                    break
                list_images_to_fuse.append(loc_1_sensor_to_fuse)
            
            if bool_FUSE:
                counter += 1
                print('Fuse', counter)
                combine_four_images(image_paths= list_images_to_fuse, output_path= 
                                    os.path.join(loc_multi_modal_ds_save, hr_file.replace(clean_name(str_sensor_heart_rate), name_fused_folder).replace(str_hr, '').replace('HeartRate', str_hr))) # first hr is to remove the HR data name. Calling .replace('HeartRate', str_hr) later will help to make sure that we get HR in the file name
                

if (crt_sensor == str_multi_modal) and (sensor_data_col == str_multi_modal):
    if current_ds == str_sapiens:
        sensor_fusion_in_input_for_sapiens()
    elif current_ds == str_tiles_18:
        sensor_fusion_in_input_for_tiles()
else:
    print('crt_sensor and sensor_data_col should be multi modal')

In [ ]:

# loc_sensor2image

In [ ]:
from itertools import combinations
combos = list(combinations(dict_sensor_sampling_rate.keys(), 4))

for test in combos:
    print(test)

# dict_sensor_sampling_rate.keys()


# Transforming Outcome Variable of TILES-18 Dataset

In [ ]:
import plotly.graph_objects as go

def sanity_check_for_eod():
    df_eod_anxiety = get_cleaned_eod_SAPIENS()
    print('Time zones in EOD Sensus', df_eod_anxiety['LocalTimeZone'].unique())

    if len(df_eod_anxiety['LocalTimeZone'].unique()) > 2:
        raise ValueError('Severe problem in the whole pipeline of analyzing and modeling since current approach converted all data in US/Eastern tz')

    number_of_eod_df = df_eod_anxiety.groupby(by=[str_p_id]).size().reset_index(name='count')
    if number_of_eod_df[number_of_eod_df['count'] > 10].shape[0] > 0:
        print(number_of_eod_df[number_of_eod_df['count'] > 10])
        raise ValueError('Ideally, there should not be more than 10 EoD surveys since participants participated for 10 days. But still, it can be possible in 1 way. Check the data of the problematic Ps manually.')
    go.Figure([go.Bar(x = number_of_eod_df[str_p_id],  y= number_of_eod_df['count'])]).show()

sanity_check_for_eod()

In [ ]:
str_us_eastern = 'US/Eastern'

def get_cleaned_state_anxiety():
    df_state_anxiety = pd.read_excel(os.path.join(loc_sensus_survey, 'cleaned_emotion_survey_ds.xlsx'))
    print('Time zones in state anxiety Sensus: ', df_state_anxiety['LocalTimeZone'].unique())

    if len(df_state_anxiety['LocalTimeZone'].unique()) > 2:
        raise ValueError('Severe problem in the whole pipeline of analyzing and modeling since current approach converted all data in US/Eastern tz')

    df_state_anxiety.rename(columns={'SubmissionTimestamp': str_sensus_submission_time, 'ParticipantId': str_p_id}, inplace=True)
    df_state_anxiety[str_sensus_submission_time] = pd.to_datetime(df_state_anxiety[str_sensus_submission_time], utc=True, unit='s').dt.tz_convert(pytz.timezone(str_us_eastern)) # e.g., 1709134954.892 is in seconds
    df_state_anxiety[str_hour] = df_state_anxiety[str_sensus_submission_time].dt.hour
    df_state_anxiety[str_date] = df_state_anxiety[str_sensus_submission_time].dt.date

    return df_state_anxiety

def analyze_state_anxiety_response():
    df_state_anxiety = get_cleaned_state_anxiety()
    px.histogram(df_state_anxiety[str_hour]).show()


analyze_state_anxiety_response()

In [ ]:
import plotly.express as px

def visualize_submission_hours_of_tiles_18_anxiety_data():
    df_anxiety = pd.read_csv(os.path.join(loc_tiles_18, 'surveys/scored/EMAs/anxiety.csv.gz'))
    print('# of NA values in anxiety df: ', df_anxiety.isna().sum().sum(), '\nBy columns: \n', df_anxiety.isna().sum())
    df_anxiety.dropna(inplace=True)
    df_anxiety = df_anxiety[df_anxiety['Finished'] != 0] # Finished: "Whether the survey was finished" (README.md)

    df_anxiety[str_date] = [date_time.split('T')[0] for date_time in df_anxiety[str_completed_ts].tolist()] # example data: 2018-09-13T23:40:40-07:00
    df_anxiety[str_hour] = [date_time.split('T')[1].split('-')[0] for date_time in df_anxiety[str_completed_ts].tolist()] 
    df_anxiety[str_date] = pd.to_datetime(df_anxiety[str_date]).dt.date
    df_anxiety[str_hour] = pd.to_datetime(df_anxiety[str_hour]).dt.hour

    hour_dist = px.histogram(df_anxiety[str_hour])
    hour_dist.show()

    print(f'Shape of the final df_anxiety: {df_anxiety.shape},\n \
            # of participants: {len(df_anxiety[str_tiles_pid].unique())}')


visualize_submission_hours_of_tiles_18_anxiety_data()

In [ ]:
def categorize_by_yesterday(subdf, global_mean, pid_col, date_col, target_col):

    subdf = subdf.sort_values(by=date_col)

    cat_list = []

    previous_score = None
    is_first_day = True  # signals we are on participant's first day

    for idx, row in subdf.iterrows():
        current_score = row[target_col]

        if is_first_day:
            cat_list.append(1 if current_score > global_mean else 0)
            is_first_day = False
        else:
            cat_list.append(1 if current_score > previous_score else 0)

        previous_score = current_score

    # Assign the new 'category' column
    subdf[str_transformed_state_anx] = cat_list
    return subdf

In [ ]:
np.random.seed(random_state)

def categorize_rows(subdf, global_mean, pid_col, date_col, target_col):
    subdf = subdf.sort_values(by=date_col)
    
    cat_list = []
    historical_values = []
    
    for idx, row in subdf.iterrows():
        if len(historical_values) < 2:
            if row[target_col] > global_mean:
                cat_list.append(1)
            else:
                cat_list.append(0)
        else:
            personal_mean = np.mean(historical_values)
            if row[target_col] > personal_mean:
                cat_list.append(1)
            else:
                cat_list.append(0)
        
        historical_values.append(row[target_col])
        
    subdf[str_transformed_state_anx] = cat_list
    return subdf

str_eod = 'EoD question'
str_transformed_state_anx = 'Transformed state anxiety'


def transform_state_anxiety_eod():
    df_state_anx = get_cleaned_state_anxiety()
    df_eod = get_cleaned_eod_SAPIENS()
    df_eod[str_p_id] = df_eod[str_p_id].astype(int)
    df_eod.rename(columns={str_category: str_eod}, inplace=True)

    df_state_eod = pd.merge(df_state_anx, df_eod, on=[str_p_id, str_date], how='inner')
    df_state_eod = df_state_eod.groupby(by=[str_p_id, str_date]).sample(n=1).reset_index(drop=True)
    print(df_state_eod.groupby(by=str_p_id)[str_state_anxiety].mean())

    global_mean = df_state_eod[str_state_anxiety].mean()
    df_state_eod = (df_state_eod.groupby(str_p_id, group_keys=False).apply(categorize_by_yesterday, global_mean=global_mean, pid_col=str_p_id, date_col=str_date, target_col=str_state_anxiety).reset_index(drop=True))

    print(jensenshannon(df_state_eod[str_transformed_state_anx].tolist(), df_state_eod[str_eod].tolist()))

    # for threshold in [2, 3, 4, 5, 6, 7, 8, 9, 10]:
    #      df_state_eod[str_transformed_state_anx] = (df_state_eod[str_state_anxiety] >= threshold).astype(int)
    #      print(threshold, jensenshannon(df_state_eod[str_transformed_state_anx].tolist(), df_state_eod[str_eod].tolist()), Counter(df_state_eod[str_transformed_state_anx].tolist()), Counter(df_state_eod[str_eod].tolist()))


    # print(df_state_eod[[str_p_id, str_date, str_state_anxiety, str_eod]])

transform_state_anxiety_eod()

In [ ]:
import pandas as pd
import numpy as np

data = {
    'Participant':  [101, 101, 101, 101, 102, 102, 102],
    'Day': [
        '2023-05-01','2023-05-02','2023-05-03','2023-05-04',
        '2023-05-01','2023-05-02','2023-05-03'
    ],
    'score': [10, 15, 20, 5, 12, 14, 16]
}
df = pd.DataFrame(data)

# Convert the Day column to an actual date object
df['Day'] = pd.to_datetime(df['Day']).dt.date

print("Original DataFrame:")
print(df)




In [ ]:
df_one_per_date.equals(df_one_per_date1)

# `Baseline Model`

In [ ]:
str_read_me = 'README.md'
b_loc_fitbit_features = '... removed the location a bit for privacy .../Documents/SA39/Hridita/Data/Tiles18/fitbit/daily-summary/' # b: baseline :)
b_fitbit_features_to_drop = [str_timestamp, 'Cardio_caloriesOut', 'Sleep1BeginTimestamp', 'Sleep1EndTimestamp', 'Sleep2BeginTimestamp', 'Sleep2Efficiency', 'Sleep2EndTimestamp', 'Sleep2MinutesAwake', 'Sleep2MinutesStageDeep', 'Sleep2MinutesStageLight', 'Sleep2MinutesStageRem', 'Sleep2MinutesStageWake', 'Sleep3BeginTimestamp', 'Sleep3Efficiency', 'Sleep3EndTimestamp', 'Sleep3MinutesAwake', 'Sleep3MinutesStageDeep', 'Sleep3MinutesStageLight', 'Sleep3MinutesStageRem', 'Sleep3MinutesStageWake', 'SleepPerDay']
b_df_dataset = pd.DataFrame()

def get_fitbit_features():
    df_features_all_p = pd.DataFrame()

    for pid in os.listdir(b_loc_fitbit_features):
        if pid in str_read_me:
            continue

        df_single_p_data = pd.read_csv(os.path.join(b_loc_fitbit_features, pid), compression='gzip')
        df_single_p_data[str_p_id] = np.repeat(pid, df_single_p_data.shape[0])
        df_features_all_p = pd.concat([df_features_all_p, df_single_p_data])
    
    df_features_all_p[str_date] = pd.to_datetime(df_features_all_p[str_timestamp]).dt.date
    df_features_all_p[str_p_id] = df_features_all_p[str_p_id].str.replace('.csv.gz', '')
    df_features_all_p.drop(columns= b_fitbit_features_to_drop, inplace=True)
    return df_features_all_p

# def get_acoustic_features():

b_df_dataset = get_fitbit_features()
b_df_anxiety = get_TILES_18_cleand_anxiety_demo_df(crt_state_anxiety_classification_approach).get(str_anxiety).rename(columns={str_tiles_pid: str_p_id})
b_df_dataset = pd.merge(b_df_dataset, b_df_anxiety[[str_p_id, str_date, str_category]], on=[str_p_id, str_date], how='inner')

In [ ]:
b_fitbit_features_to_drop = ['str_timestamp', 'Cardio_caloriesOut', 'Sleep1BeginTimestamp', 'Sleep1EndTimestamp', 'Sleep2BeginTimestamp', 'Sleep2Efficiency', 'Sleep2EndTimestamp', 'Sleep2MinutesAwake', 'Sleep2MinutesStageDeep', 'Sleep2MinutesStageLight', 'Sleep2MinutesStageRem', 'Sleep2MinutesStageWake', 'Sleep3BeginTimestamp', 'Sleep3Efficiency', 'Sleep3EndTimestamp', 'Sleep3MinutesAwake', 'Sleep3MinutesStageDeep', 'Sleep3MinutesStageLight', 'Sleep3MinutesStageRem', 'Sleep3MinutesStageWake', 'SleepPerDay']
ot_col_list = [Cardio_caloriesOut	Cardio_max	Cardio_min	Cardio_minutes	Fat Burn_caloriesOut	Fat Burn_max	Fat Burn_min	Fat Burn_minutes	NumberSteps	Out of Range_caloriesOut	Out of Range_max	Out of Range_min	Out of Range_minutes	Peak_max	Peak_min	RestingHeartRate	Sleep1Efficiency	Sleep1MinutesAwake	Sleep1MinutesStageLight	Sleep1MinutesStageRem	Sleep1MinutesStageWake	SleepMinutesAsleep	SleepMinutesInBed	SleepPerDay	F0final_sma	jitterLocal_sma	jitterDDP_sma	shimmerLocal_sma	logHNR_sma	voiceProb_sma	F0_sma	F0env_sma	audspec_lengthL1norm_sma	audspecRasta_lengthL1norm_sma	pcm_RMSenergy_sma	pcm_zcr_sma	pcm_intensity_sma	pcm_loudness_sma	pcm_fftMag_fband250-650_sma	pcm_fftMag_fband1000-4000_sma	pcm_fftMag_spectralRollOff25.0_sma	pcm_fftMag_spectralRollOff50.0_sma	pcm_fftMag_spectralRollOff75.0_sma	pcm_fftMag_spectralRollOff90.0_sma	pcm_fftMag_spectralFlux_sma	pcm_fftMag_spectralCentroid_sma	pcm_fftMag_spectralEntropy_sma	pcm_fftMag_spectralVariance_sma	pcm_fftMag_spectralSkewness_sma	pcm_fftMag_spectralKurtosis_sma	pcm_fftMag_spectralSlope_sma	pcm_fftMag_psySharpness_sma	pcm_fftMag_spectralHarmonicity_sma	BreathingRate	HeartRate	Intensity	AvgHeartRate	AvgXAccel_g	AvgYAccel_g	AvgZAccel_g	INRS_rmsdd	INRS_total_f	INRS_vlf	INRS_lf	INRS_hf	INRS_lf_hf	'INRS_lfnu'	'INRS_hfnu']

In [ ]:
print(b_df_dataset.columns[b_df_dataset.isna().any()])
print(b_df_dataset.isna().sum())

In [ ]:
columns_to_drop_for_features = [str_date, str_category, str_p_id]

def impute_and_scale_feature_values():
    columns_with_missing_values = b_df_dataset.columns[b_df_dataset.isna().any()].tolist()

    for feature_col in columns_with_missing_values:
        mean_value = b_df_dataset[feature_col].mean(skipna=True)
        b_df_dataset[feature_col].fillna(mean_value, inplace=True)
    
    if b_df_dataset.isna().sum().sum() > 0:
        raise ValueError('After imputation, there should not be any NaN values.')
    
    ##### Scaling
    scaler = StandardScaler()
    list_feature_columns = [feature_col for feature_col in b_df_dataset.columns.tolist() if feature_col not in columns_to_drop_for_features]
    b_df_dataset[list_feature_columns] = scaler.fit_transform(b_df_dataset[list_feature_columns])

def model_development_and_validation():
    list_predicted_class = []
    list_true_class = []

    for nth_p, pid in enumerate(sorted(b_df_dataset[str_p_id].unique())):
        df_train = b_df_dataset[b_df_dataset[str_p_id] != pid].copy()
        df_test = b_df_dataset[b_df_dataset[str_p_id] ==  pid].copy()

        model = RandomForestClassifier(random_state= random_state)
        model.fit(X=df_train.drop(columns=columns_to_drop_for_features), y=df_train[str_category])
        single_p_predicted_class = model.predict(X= df_test.drop(columns = columns_to_drop_for_features))

        list_true_class.extend(df_test[str_category].tolist())
        list_predicted_class.extend(single_p_predicted_class)
    
        print(nth_p+1, clf_report(true_labels= list_true_class, predicted_val_list= list_predicted_class, bool_print=False))


impute_and_scale_feature_values()
model_development_and_validation()

# `Evaluation of the Weak Learners`

In [ ]:
temp_ablation_perf = '... removed the location a bit for privacy .../Documents/SA39/SAPIENS/Daily Anxiety/on_the_same_day/multi_modal/splitted_into3/Tiles18_multi_modal_multi_modal/Performance'
for model_dev_strategy in os.listdir(temp_ablation_perf):
    if 'DS_Store' in model_dev_strategy:
        continue

    df_perf_ablation = pd.read_excel(os.path.join(temp_ablation_perf, model_dev_strategy, str_file_predict_prob_fine_tuned_resnet))
    list_segments = [col for col in df_perf_ablation if str_segment in col]
    list_all_segments_predicted_prob = []
    list_actual_classes = []
    for col_segment in list_segments:
        list_all_segments_predicted_prob.extend(df_perf_ablation[col_segment].tolist())
        list_actual_classes.extend(df_perf_ablation[str_actual_class].tolist())

    list_all_segments_predicted_prob = np.where(np.array(list_all_segments_predicted_prob) > 0.5, 1, 0)   
    print(model_dev_strategy, clf_report(true_labels= list_actual_classes, predicted_val_list= list_all_segments_predicted_prob, bool_print=False))

    df_perf_ablation[str_predicted_class] = ((df_perf_ablation[str_segment+'1'] > 0.5) & (df_perf_ablation[str_segment+'2'] > 0.5)).astype(int) | ((df_perf_ablation[str_segment+'1'] > 0.5) & (df_perf_ablation[str_segment+'3'] > 0.5)).astype(int) | ((df_perf_ablation[str_segment+'2'] > 0.5) & (df_perf_ablation[str_segment+'3'] > 0.5)).astype(int)
    print(clf_report(true_labels= df_perf_ablation[str_actual_class].tolist(), predicted_val_list= df_perf_ablation[str_predicted_class].tolist(), bool_print= False))    

# `Different Things for Paper`

## Descriptive Stats

In [ ]:
def get_pid_list():
    __ = pd.read_excel('... removed the location a bit for privacy .../Documents/SA39/SAPIENS/State Anxiety/2.5_hours_1_vs_2_10_split/multi_modal/splitted_into3/SAPIENS_HeartRate_Light__accel___magnet_/Performance/ResNet_18/iter_130__a9dfbe4d-4076-48c7-a72b-342fe4c12514__chunk_test_5_val_2__fine_tuned_all_layers/'+str_file_predict_prob_fine_tuned_resnet)
    return sorted(__[str_p_id].unique())

def HR_descriptive_stats_on_number_of_participant_days():
    list_n_days = []
    for pid in get_pid_list():
        pid = 'P'+str(pid)
        if 'DS_Store' not in pid:
            df_hr = pd.read_pickle(os.path.join(loc_sapiens_ds, pid, 'Smartwatch_HeartRateDatum.pkl'))
            df_hr[str_date] = df_hr['T'].dt.date
            list_n_days.append(len(df_hr[str_date].unique()))

    print(sum(list_n_days), round(np.mean(list_n_days), 2), round(np.std(list_n_days), 2))

# HR_descriptive_stats_on_number_of_participant_days()


def EMA_descriptive_stats_on_number_of_days():
    df_cleaned_state_anxiety = get_cleaned_state_anxiety_SAPIENS(str_1_vs_2_10_split)
    df_cleaned_state_anxiety[str_p_id] = df_cleaned_state_anxiety[str_p_id].astype(int)
    df_cleaned_state_anxiety = df_cleaned_state_anxiety[df_cleaned_state_anxiety[str_p_id].isin(get_pid_list())]

    print(round(df_cleaned_state_anxiety.groupby(by=[str_p_id, str_date]).size().mean(), 2), 
          round(df_cleaned_state_anxiety.groupby(by=[str_p_id, str_date]).size().std(), 2), 
          round(df_cleaned_state_anxiety.groupby(by=[str_p_id, str_date]).size().sum(), 2))

EMA_descriptive_stats_on_number_of_days()

In [ ]:

temp_ablation_perf = '... removed the location a bit for privacy .../Documents/SA39/SAPIENS/Daily Anxiety/on_the_same_day/multi_modal/splitted_into3/Tiles18_multi_modal_multi_modal/Performance'
for model_dev_strategy in os.listdir(temp_ablation_perf):
    if 'DS_Store' in model_dev_strategy:
        continue

    df_perf_ablation = pd.read_excel(os.path.join(temp_ablation_perf, model_dev_strategy, str_file_predict_prob_fine_tuned_resnet))
    list_segments = [col for col in df_perf_ablation if str_segment in col]
    list_all_segments_predicted_prob = []
    list_actual_classes = []
    for col_segment in list_segments:
        list_all_segments_predicted_prob.extend(df_perf_ablation[col_segment].tolist())
        list_actual_classes.extend(df_perf_ablation[str_actual_class].tolist())

    list_all_segments_predicted_prob = np.where(np.array(list_all_segments_predicted_prob) > 0.5, 1, 0)   
    print(model_dev_strategy, clf_report(true_labels= list_actual_classes, predicted_val_list= list_all_segments_predicted_prob, bool_print=False))

    df_perf_ablation[str_predicted_class] = ((df_perf_ablation[str_segment+'1'] > 0.5) & (df_perf_ablation[str_segment+'2'] > 0.5)).astype(int) | ((df_perf_ablation[str_segment+'1'] > 0.5) & (df_perf_ablation[str_segment+'3'] > 0.5)).astype(int) | ((df_perf_ablation[str_segment+'2'] > 0.5) & (df_perf_ablation[str_segment+'3'] > 0.5)).astype(int)
    print(clf_report(true_labels= df_perf_ablation[str_actual_class].tolist(), predicted_val_list= df_perf_ablation[str_predicted_class].tolist(), bool_print= False))    

## Number of Ps with 10 and 5 minutes interval

In [ ]:
from pandas import Timedelta
import pandas as pd
import os

def print_number_of_ps_with_10_5_minutes_interval():
    global dict_list_pid_with_interval
    n_5_min_interval = 0
    n_10_min_interval = 0

    dict_list_pid_with_interval = dict() 
    dict_list_pid_with_interval['5'] = []
    dict_list_pid_with_interval['10'] = []

    for pid in sorted(os.listdir(loc_sapiens_ds)):
        if 'DS' in pid:
            continue

        temp_df_sensor_start = pd.read_pickle(os.path.join(loc_sapiens_ds, pid, 'Smartwatch_SensorsStartingTimeDatum.pkl'))
        temp_df_sensor_start['T'] = pd.to_datetime(temp_df_sensor_start['T'])
        temp_df_sensor_start['T'] = temp_df_sensor_start['T'].dt.round('min')

        interval_counter_dict = Counter(temp_df_sensor_start['T'].diff().dropna().tolist())
        
        single_p_5_min_interval = interval_counter_dict.get(Timedelta('0 days 00:05:00'))
        single_p_10_min_interval = interval_counter_dict.get(Timedelta('0 days 00:10:00'))
        
        if (interval_counter_dict.most_common()[0][0] != Timedelta('0 days 00:05:00')) and (interval_counter_dict.most_common()[0][0] != Timedelta('0 days 00:10:00')):
            print('Papi', pid, interval_counter_dict.most_common())

        if single_p_10_min_interval == None:
            single_p_10_min_interval = 0
        if single_p_5_min_interval == None:
            single_p_5_min_interval = 0


        if single_p_5_min_interval > single_p_10_min_interval:
            n_5_min_interval += 1
            dict_list_pid_with_interval['5'].append(pid)
        elif single_p_10_min_interval > single_p_5_min_interval:
            n_10_min_interval += 1
            dict_list_pid_with_interval['10'].append(pid)
        else:
            raise ValueError(f"{pid}, 'Papi. Both are 0 which can not happen!")

    print("n_5_min_interval", n_5_min_interval, 'n_10_min_interval',  n_10_min_interval)

print_number_of_ps_with_10_5_minutes_interval()

In [ ]:
dict_list_pid_with_interval.get('5')

## Visualizing data quality

In [ ]:
import re
str_all = 'All'
str_data = 'Data'
str_date = 'Date'
str_data_name = 'Data_Name'
list_color = ['#FDE0D5', '#F5AE9D', '#df7863', '#CC4D38', '#ac2610', '#6d1002', '#FDE0D5', '#F5AE9D', '#778899', '#696969', '#2F4F4F']
list_data_files_to_explore = []


def sort_num_list_contain_str(list_str_num):
    return list(map(int, re.findall(r'-?\d+\.?\d*', list_str_num)))[0]

def get_unique_time_upto_min(df):
    df_temp = df.copy()
    df_differences = df_temp['T'].diff()
    expect_diff = pd.Timedelta(minutes=1)
    df_unique_times = df_temp.loc[df_differences > expect_diff]
    #df_temp.loc[(df_differences > pd.Timedelta(minutes=0)) & (df_differences <= expect_diff)]
    
    # df_temp['TimeDifference'] = df_temp['T'].diff().dt.total_seconds() / 60
    # visualize_time_diff_battery(df_temp)
    
    return df_unique_times


def get_num_of_samples_in_each_probe_start_time(df):
    df_temp = df.copy()
    df_temp.sort_values('T', inplace=True)
    
    unique_time_list = []
    unique_time_list.append(df_temp['T'].tolist()[0]) # for the first difference for which we get NaT while using diff() in get_unique_time_upto_min()
    # create a dataframe using the data: '2023-11-14T15:00:00', '2023-11-14T15:01:30', '2023-11-14T15:01:00', '2023-11-14T15:07:30', '2023-11-14T15:09:31', '2023-11-14T15:12:33', '2023-11-14T15:15:33'
    # after that pass the data to this function. You will understand the reason then
    unique_time_list.extend(sorted(get_unique_time_upto_min(df_temp)['T'].tolist()))
        
    min_timestamp_est = pd.Timestamp('2000-04-11T00:00:00.000-0500')
    max_timestamp_est = pd.Timestamp('2262-04-11T00:00:00.000-0500')
    
    bins = [min_timestamp_est.strftime("%Y-%m-%dT%H:%M:%S.%f%z")] + [ts - pd.Timedelta(microseconds=1) for ts in unique_time_list] + [max_timestamp_est.strftime("%Y-%m-%dT%H:%M:%S.%f%z")]
    bins = pd.to_datetime(bins, utc=True).tz_convert(pytz.timezone('US/Eastern')).tolist()

    df_temp['Interval'] = pd.cut(df_temp['T'], bins=bins, labels=range(len(unique_time_list) + 1))
    counts = df_temp.groupby('Interval').size()
    
    return unique_time_list, counts.tolist()[1:] # for the first instance, we will get 0 since our limit for counting the samples is [..., next_probe_starting_time - 1 microsecond]


# This function is used to divide (into 0 to 8 and 9 to 23 hours) any wearing duration if a user uses watch between 0 to 8
def divide_duration_night_morning(last_ts, crt_ts):
    if crt_ts.day == last_ts.day:
        return (crt_ts - last_ts).total_seconds(), 0
    else: # seperating the duration between 12 AM to 8 AM from the total duration
        max_last_ts = pd.Timestamp(year = last_ts.year, month = last_ts.month, day = last_ts.day,
                                           hour = 23, minute = 59, second = 59, microsecond=999999, tz=pytz.timezone(str_us_eastern)) # using microsecond will help us to avoid any potential very small negative difference since without specifying this will take the by default microsecond 0
        duration = (max_last_ts - last_ts).total_seconds()

        if (max_last_ts - last_ts).total_seconds() < 0:
            print('\n\n\nError. Negative duration 😒😡😠\n\n\n', crt_d_id, (max_last_ts - last_ts).total_seconds())

        min_crt_ts = pd.Timestamp(year = crt_ts.year, month = crt_ts.month, day = crt_ts.day,
                                  hour = 0, minute = 0, second = 0, microsecond=0, tz=pytz.timezone(str_us_eastern))
        _12_8_duration = (crt_ts - min_crt_ts).total_seconds()

        if (crt_ts - min_crt_ts).total_seconds() < 0:
            print('\n\n\nError. Negative duration 2nd case 😒😡😠\n\n\n', crt_d_id, (crt_ts - min_crt_ts).total_seconds())

        return duration, _12_8_duration
    

import warnings
warnings.filterwarnings('ignore')
def variance_by_sensor_per_participant(df_each_day_data, df_all_days_data, group_by_str, y_data, y_axis, bool_mean, folder_name):    
    fig = px.bar(df_each_day_data, x=str_data_name, y=y_data, color=str_date)
    fig.update_layout({
        "plot_bgcolor": "#FFFFFF",
        'yaxis':go.layout.YAxis(title=go.layout.yaxis.Title(text= y_axis,
        font=dict(family='Times New Roman', size=18, color='#000000')),
        tickmode = 'array', color="#000000", tickfont=dict(family='Times New Roman', size=18))})
    fig.write_html(loc_temp_data_quality + folder_name + '_by_day_per_parti.html')

    list_col_sub_df = group_by_str.split(',') # group_by_str: keeping variable number of columns
    list_col_sub_df.append(str_data)
    df_each_day_data = df_each_day_data[list_col_sub_df].copy() # copying to ensure that the processed dataframe inside the function does not reflect outside of the function.
    op_name = ''
    
    if bool_mean:
        op_name = 'mean'
        df_each_day_data = df_each_day_data.groupby(group_by_str.split(','))[y_data].mean().reset_index()
    else: # sum
        op_name = 'sum'
        df_each_day_data = df_each_day_data.groupby(group_by_str.split(','))[y_data].sum().reset_index()
    
    # df_each_day_data.to_excel('all_data_sum.xlsx', index=False)
    
    ordered_id_list = list(set(df_each_day_data[str_p_id].tolist()))
    ordered_id_list.sort(key=sort_num_list_contain_str)
    print(ordered_id_list)
    
    df_each_day_data[str_p_id] = pd.Categorical(df_each_day_data[str_p_id], categories=ordered_id_list, ordered=True)
    df_each_day_data.sort_values(by=str_p_id, inplace=True)
    
    fig = go.Figure()
    for p_id, color_code in zip(ordered_id_list, list_color):
        fig.add_trace(go.Box(
                        y= df_each_day_data[df_each_day_data[str_p_id] == p_id][y_data],
                        name= p_id,
                        marker_color= color_code,
                        boxmean='sd'))
        
    fig.update_layout(
        plot_bgcolor= "#FFFFFF",
        yaxis=go.layout.YAxis(title=go.layout.yaxis.Title(text= y_axis, font=dict(family='Times New Roman', size=18, color='#000000'))),
        xaxis=go.layout.XAxis(title=go.layout.xaxis.Title(text=str_data_name, font=dict(family='Times New Roman', size=21, color='#000000')), 
                              color="#000000", tickfont=dict(family='Times New Roman', size=21)))
    # fig.write_html(loc_temp_data_quality + op_name + folder_name + '_variance_by_sensor_per_parti.html')
    
    fig = px.bar(df_all_days_data, x=str_p_id, y=y_data,
                 color=str_data_name, barmode='group')
    fig.update_layout({
        "plot_bgcolor": "#FFFFFF",
        'yaxis':go.layout.YAxis(title=go.layout.yaxis.Title(text= y_axis,
        font=dict(family='Times New Roman', size=18, color='#000000')),
        tickmode = 'array', color="#000000", tickfont=dict(family='Times New Roman', size=18))})
    # fig.write_html(loc_temp_data_quality + op_name + folder_name + '_by_sensor_per_parti.html')
    
    df_all_days_data = df_all_days_data.loc[~df_all_days_data[str_data_name].isin(['Offbody', 'Wear dur 12 to 8', 'Non-wear dur since 1st data coll.'])].copy()
    df_all_days_data.replace({'Wear dur excl. 12 to 8 AM': 'Wear dur. (H)'}, inplace = True)
    df_all_days_data[y_data] = df_all_days_data[y_data].round(1)

    fig = px.bar(df_all_days_data, x=str_data_name, y=y_data, color=str_p_id, text=y_data, barmode='group')
    fig.update_layout(
        plot_bgcolor= "#FFFFFF",
        yaxis=go.layout.YAxis(title=go.layout.yaxis.Title(text= y_axis, font=dict(family='Times New Roman', size=20, color='#000000')), tickfont=dict(family='Times New Roman', size=20)),
        xaxis=go.layout.XAxis(title=go.layout.xaxis.Title(text=str_data, font=dict(family='Times New Roman', size=20, color='#000000')), 
                              color="#000000", tickfont=dict(family='Times New Roman', size=20)),
                              showlegend=False) 
    x_axis_data_order  = list(sorted(df_all_days_data[str_data_name].unique()))
    x_axis_data_order.remove('SensorsStart')
    x_axis_data_order.append('SensorsStart')
    
    fig.update_traces(textfont = dict(family="Times New Roman", size=20, color="white"))
    fig.update_xaxes(categoryorder='array', categoryarray= x_axis_data_order)

    fig.write_html(loc_temp_data_quality + op_name + folder_name + '_group_by_data_by_sensor_per_parti.html')
    fig.write_image(loc_temp_data_quality + op_name + folder_name + '_group_by_data_by_sensor_per_parti.png')


def get_total_duration_of_wearing_device(sub_df_p_id, include_consecutive_1, crt_date, crt_d_id, loc_sensor_starting_time, data_file_name):
    
    wear_status = 0
    last_wear_status = 0
    duration = 0
    _12_8_duration = 0
    HOUR = 60 * 60
    
    max_allowed_prob_dur = 16 # prob: problematic. This data (if you set True for include_consecutive_1, the data will be added) is problematic since the duration is based on consecutive event 1 
    # (means the participant wore the device) and we do not know exactly when the participant put off the device (when the event would be 0).
    # max_allowed_prob_dur is 16 since based on our current plan (Fall'23), the app can collect data maximum 16 hours a day. Additionally, to me (do not have evidence BTW), it is unlikely someone may keep wearing a device for more than 16 hours a day
    
    for index, row in sub_df_p_id.iterrows():
      wear_status = row['Offboddy']
      if wear_status == 0 and last_wear_status == 1:  # Do not worry about the first instance (when the wear_status is 0 and we do not have last_row data) since last_wear_status is initialized 0 and will not be changed until the first iteration is done.
        crt_ts = pd.Timestamp(row['T'])
        last_ts = pd.Timestamp(last_row['T'])
                
        temp_dur, temp_12_to_8_dur = divide_duration_night_morning(last_ts, crt_ts)
        duration += temp_dur
        _12_8_duration += temp_12_to_8_dur
        
      elif wear_status == 1 and last_wear_status == 1 and include_consecutive_1:
        consec_dur = (pd.Timestamp(row['T']) - pd.Timestamp(last_row['T'])).total_seconds() # consec: consecutive 1 (Offboddy==1 which means the participant wore the device)
        if (consec_dur / HOUR) <= max_allowed_prob_dur: # This data (if you set True for include_consecutive_1, the data will be added) is problematic since the duration is based on consecutive event 1 (means the participant wore the device) and we do not know exactly when the participant put off the device.
            duration += consec_dur
      
      last_wear_status = wear_status
      last_row = row
      
      if duration < 0: # This is not supposed to happen. However, it may happen in some cases. e.g., the data is not sorted by time
        print("\n\nError\n\n")
        break            
    
    df_sensor_start = pd.read_pickle(os.path.join(loc_sensor_starting_time, 'Smartwatch_SensorsStartingTimeDatum.pkl'))
    # print()
    df_sensor_start.rename(columns={'DeviceId': str_device_id}, inplace = True) # DeviceId is in AppTesting data of Spring
    df_sensor_start['T'] = pd.to_datetime(df_sensor_start['T'])
    df_sensor_start[str_date] = df_sensor_start['T'].dt.date
    
    if crt_date != str_all:
        df_temp = df_sensor_start[(df_sensor_start[str_date] == crt_date) & (df_sensor_start[str_device_id] == crt_d_id)].copy()
        df_temp.sort_values('T', inplace=True)
        mean_probe_start = 0  # WARNING: work so that you get the mean value in the case of day level analysis as well
        sd_probe_start = 0 # WARNING: work so that you get the sd value in the case of day level analysis as well
    else:
        df_temp = df_sensor_start[(df_sensor_start[str_device_id] == crt_d_id)].copy()
        df_temp.sort_values('T', inplace=True)
        df_temp['diff_in_start'] = df_temp['T'].diff().dt.total_seconds() / 60
        mean_probe_start =df_temp['diff_in_start'].mean()
        sd_probe_start =df_temp['diff_in_start'].std(ddof=1)
    
    # print(last_row)
    # print(str_date)
    
    if (last_row['Offboddy'] == 1) and (df_temp.shape[0] > 1): # df_temp.shape[0] == 0 but last_row['Offboddy'] == 1 means the participant used the watch on that day, but, no data was collected on that day since the partiicpant put off the watch before data collection
        last_row_start_time = df_temp.iloc[-1]
        if (pd.Timestamp(last_row_start_time['T']) > pd.Timestamp(last_row['T'])): # This means user started wearing (Offboddy==1) the watch in a day and kept wearing until the charge is 0%
            temp_dur, temp_12_to_8_dur = divide_duration_night_morning(pd.Timestamp(last_row['T']), 
                                                                       pd.Timestamp(last_row_start_time['T']))
            duration += temp_dur
            _12_8_duration += temp_12_to_8_dur
    
    if ('OffbodyDatum' not in data_file_name) | (crt_date == str_all):
        non_wear_dur = ((pd.Timestamp(df_temp.iloc[-1]['T']) - pd.Timestamp(df_temp.iloc[0]['T'])).total_seconds() - (duration + _12_8_duration)) / HOUR # non-wearing duration since the first probe starting time
    else:
        non_wear_dur = 0 # currently, we are not calculating the non-wearing duration per day since there can be some days when there is no sensor starting time (due to not wearing the watch for a minimum of ~10 minutes).
        # But we will be handling that in the near future
    
    # Do not change the order of returning values since the values are named as per this order: wearning duration (excluding 0 to 8), duration between 8 to 23, non-wearing duration since the first probe starting time
    return duration/HOUR, _12_8_duration/HOUR, non_wear_dur # mean_probe_start, sd_probe_start


def visualize_by_bar_chart(df_viz_data, y_data, y_axis, ):
    df_viz_data = df_viz_data[[str_p_id, y_data]].copy() # copying to ensure that the processed dataframe inside the function does not reflect outside of the function
    # df_viz_data = df_viz_data.groupby(str_p_id)[y_data].sum().reset_index()
    df_viz_data.sort_values(y_data, inplace=True)
    
    bar_fig = go.Bar(x=df_viz_data[str_p_id], y=df_viz_data[y_data])
    layout = go.Layout(
      title=go.layout.Title(text='', x=0),
      xaxis=go.layout.XAxis(title=go.layout.xaxis.Title(
      text="Participant", font=dict(family='Times New Roman', size=18, color='#000000')),
      tickmode = 'array', color="#000000", tickfont=dict(family='Times New Roman', size=18)),
      
      yaxis=go.layout.YAxis(title=go.layout.yaxis.Title(text= y_axis,
      font=dict(family='Times New Roman', size=18, color='#000000')),
      tickmode = 'array', color="#000000", tickfont=dict(family='Times New Roman', size=18)),
      legend=dict(orientation="h", x=0.3, y=1.1,  font=dict(family='Times New Roman', size=18), bgcolor='#FFFFFF'),
      
      plot_bgcolor = '#FFFFFF')
    fig = go.Figure(bar_fig, layout=layout)
    # fig.show()


def variance_per_sensor(df_viz_data, y_data, x_axis, y_axis):
    ordered_id_list = list(set(df_viz_data[str_p_id].tolist()))
    ordered_id_list.sort(key=sort_num_list_contain_str)
    
    data_name_list = list(set(df_viz_data[str_data_name].tolist()))
    
    for p_id in ordered_id_list:
        df_viz_data[str_p_id] = pd.Categorical(df_viz_data[str_p_id], categories=ordered_id_list, ordered=True)
        df_viz_data.sort_values(by=str_p_id, inplace=True)

        fig = go.Figure()
        for data_name, color_code in zip(data_name_list, list_color):
            fig.add_trace(go.Box(
                            x = df_viz_data[df_viz_data[df_viz_data[str_p_id] == p_id] == data_name][x_axis],
                            y = df_viz_data[df_viz_data[df_viz_data[str_p_id] == p_id] == data_name][y_data],
                            name= p_id,
                            marker_color=color_code,
                            boxmean='sd'))
        fig.update_layout(yaxis_title=y_axis, boxmode='group')
        fig.update_layout({
            "plot_bgcolor": "#FFFFFF",
            'yaxis':go.layout.YAxis(title=go.layout.yaxis.Title(text= y_axis,
            font=dict(family='Times New Roman', size=18, color='#000000')),
            tickmode = 'array', color="#000000", tickfont=dict(family='Times New Roman', size=18))})

        fig.write_html(loc_temp_data_quality+p_id+'_var_in_#_of_data_instance_in_each_probe_start_time.html')


def visualize_list_of_fig(p_id, plot_data_list, x_axis_title, y_axis_title, loc_file):

  layout = go.Layout(
  title=go.layout.Title(text='', x=0),
      xaxis=go.layout.XAxis(title=go.layout.xaxis.Title(
      text=x_axis_title, font=dict(family='Times New Roman', size=18, color='#000000')),
      tickmode = 'array', color="#000000", tickfont=dict(family='Times New Roman', size=18)),

      yaxis=go.layout.YAxis(title=go.layout.yaxis.Title(text= y_axis_title,
      font=dict(family='Times New Roman', size=18, color='#000000')),
      tickmode = 'array', color="#000000", tickfont=dict(family='Times New Roman', size=18)),
      legend=dict(orientation="h", x=0.3, y=1.1,  font=dict(family='Times New Roman', size=18), bgcolor='#FFFFFF'),

      plot_bgcolor = '#FFFFFF')

  fig = go.Figure(plot_data_list, layout=layout)
  fig.update_layout(legend=dict(
        orientation="h",
        x=0.5,                
        y=1.02,           
        xanchor="center",       
        yanchor="bottom",
        itemwidth=150,
        font=dict(size=20)
    ), width = 1000, height = 400)
  fig.write_html(loc_file)
  fig.write_image(loc_file.replace('html', 'png'))


In [ ]:
def process_visualize_data(loc_ds):
    global df_all_data
    
    for folder_name in sorted(os.listdir(loc_ds)):
        if 'DS_Store' in folder_name:
            continue

        loc_data_files = os.path.join(loc_ds, folder_name)        
        all_data_list_df = []
        fig_list = []
        
        for data_file in sorted(os.listdir(loc_data_files)):
            processed_data_file_name = data_file.replace('Smartwatch_', '').replace('.pkl', '').replace('Datum', '').replace('SensorsStartingTime', 'SensorsStart').replace('LinearAcceleration', 'Acceleration').replace('Count', '')

            if data_file not in ['Smartwatch_LinearAccelerationDatum.pkl', 'Smartwatch_BatteryDatum.pkl', 'Smartwatch_SensorsStartingTimeDatum.pkl', 'Smartwatch_LightDatum.pkl', 
                                 'Smartwatch_HeartRateDatum.pkl', 'Smartwatch_MagnetometerDatum.pkl', 'Smartwatch_GravityDatum.pkl', 'Smartwatch_StepCountDatum.pkl', 'Smartwatch_OffbodyDatum.pkl']:
                continue

            df_data = pd.read_pickle(os.path.join(loc_data_files, data_file))
            df_data['T'] = pd.to_datetime(df_data['T'])
            df_data[str_date] = df_data['T'].dt.date
            
            other_data = ''
            if 'DeviceId' in df_data.columns.tolist():
                df_data.rename(columns = {'DeviceId': str_device_id}, inplace=True)

            if len(df_data[str_device_id].unique()) > 1:
                # error_info_writer('\n\n🔥🔥🔥🔥Severe problem. One participant\'s data folder contains more than 1 device ID which is not supposed to happen\n\n.')
                raise Exception('🔥🔥🔥🔥Severe problem. One participant\'s data folder contains more than 1 device ID which is not supposed to happen. Check: '+folder_name +'_'+ data_file)
     
            d_id = df_data[str_device_id].unique().tolist()[0]
            df_data.sort_values('T', inplace=True) # Do not remove this statement since it will have impact. e.g., get_unique_time_upto_min() calculates the consecutive time difference. Not having the sorted data will create a misleading findings. Similarly, get_total_duration_of_wearing_device() will be impacted
            list_days = sorted(set(df_data[str_date]))
            list_days.append(str_all)
            for crt_date in list_days: # crt: current

                if crt_date != str_all:
                    sub_df_day = df_data[df_data[str_date] == crt_date].copy()
                else:
                    sub_df_day = df_data.copy()
                all_data_list_df.append([d_id, get_unique_time_upto_min(sub_df_day).shape[0] + 1, processed_data_file_name, crt_date]) # +1: for the first difference for which we get NaT while using diff().
                # create a dataframe using the data: '2023-11-14T15:00:00', '2023-11-14T15:01:30', '2023-11-14T15:01:00', '2023-11-14T15:07:30', '2023-11-14T15:09:31', '2023-11-14T15:12:33', '2023-11-14T15:15:33'
                # after that pass the data to this function. You will understand the reason then

                if 'Smartwatch_OffbodyDatum' in data_file:
                    other_data = 'Wear dur excl. 12 to 8 AM'
                    list_wearing_dur_related = get_total_duration_of_wearing_device(sub_df_day, False, crt_date, d_id, loc_data_files, data_file)
                    for dur_related_data, dur_data_name in zip(list_wearing_dur_related, [other_data, 'Wear dur 12 to 8', 'Non-wear dur since 1st data coll.']): # 'Mean probe starting time', 'SD starting time'
                        all_data_list_df.append([d_id, dur_related_data, dur_data_name, crt_date])

            columns_list = [str_p_id, str_data, str_data_name, str_date]
            df_sub_viz_data = pd.DataFrame(data = all_data_list_df, columns = columns_list)

            visualize_by_bar_chart(df_sub_viz_data[(df_sub_viz_data[str_data_name] == processed_data_file_name) & (df_sub_viz_data[str_date] == str_all)], str_data, '# of times data was collected')

            if '_BatteryDatum' in data_file:
                visualize_by_bar_chart(df_sub_viz_data[(df_sub_viz_data[str_data_name] == other_data) & (df_sub_viz_data[str_date] == str_all)], str_data, '# of times power saving mode was on')
            if 'Smartwatch_OffbodyDatum' in data_file:
                visualize_by_bar_chart(df_sub_viz_data[(df_sub_viz_data[str_data_name] == other_data) & (df_sub_viz_data[str_date] == str_all)], str_data, 'Hours the participant used the watch')
            
            unique_time_list, sample_list = get_num_of_samples_in_each_probe_start_time(df_data)

            if data_file not in 'Smartwatch_OffbodyDatum.pkl':
                fig = go.Scatter(x=unique_time_list, y=sample_list, mode='lines+markers', fill=None, name= processed_data_file_name, 
                                fillcolor=None, textfont=dict(family='Times New Roman', size=18, color='#000000')) # marker=dict(color = '#eb0b0b', size=8)
                fig_list.append(fig)
        
        df_all_data = pd.DataFrame(data = all_data_list_df, columns = columns_list)
        df_all_data.to_excel(os.path.join(loc_temp_data_quality, 'DF_for_Figures', folder_name  + '_data_for_viz.xlsx'), index=False)
        
        variance_by_sensor_per_participant(df_all_data[df_all_data[str_date] != str_all].copy(),
                                           df_all_data[df_all_data[str_date] == str_all].copy(),
                                           str_data_name+','+str_p_id, str_data, '# of probe starts with data', False, folder_name.replace('JSON_Files', '').replace('_Interval', ''))

        visualize_list_of_fig(folder_name, fig_list, 'Date-time', '# of Samples',
                             loc_temp_data_quality+folder_name+'_#_of_samples_in_each_time.html')


loc_app_test_spring_data = '... removed the location a bit for privacy .../Documents/SA39/SAPIENS/All Data Spring\'24/AppTestingSpring24/Processed Files/Raw_Data_Folders_Before_20th_Feb'
loc_figure_app_test_spring = '... removed the location a bit for privacy .../Documents/SA39/SAPIENS/All Data Spring\'24/AppTestingSpring24/Figures/Data_Folders_Before_20th_Feb/'

loc_temp_data_quality = loc_data_quality  # loc_data_quality for SAPIENS

if 'Testing' in loc_temp_data_quality:
    process_visualize_data(loc_app_test_spring_data)
else:
    process_visualize_data(loc_sapiens_ds)

In [ ]:
def SAPIENS_DS_get_overall_data_quality():
    for csv_excel_folder in os.listdir(loc_sapiens_ds):
        if 'DS_Store' in csv_excel_folder:
            continue
        if len(os.listdir(os.path.join(loc_sapiens_ds, csv_excel_folder))) == 0:
            print(csv_excel_folder)


    df_all_parti_data = pd.DataFrame()
    loc_fig_dataframe = '... removed the location a bit for privacy .../Documents/SA39/SAPIENS/All Data Spring\'24/Figures/Data Quality/DF_for_Figures'
    font_size = 20

    for parti_data in sorted(os.listdir(loc_fig_dataframe)):

        if ('DS_Store' in parti_data) or ('git' in parti_data) or ('ipynb_checkpoints' in parti_data):
            continue

        if int(parti_data.replace('_data_for_viz.xlsx', '').replace('P', '')) not in [413, 414, 415, 416, 417, 418, 420, 421, 423, 424, 428, 429, 430, 431, 432, 433, 434, 436, 437, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 480, 481, 482, 483, 485, 486, 487, 488, 490, 491, 492, 493, 494, 497, 498]:
            continue

        
        df_data = pd.read_excel(os.path.join(loc_fig_dataframe, parti_data))
        
        df_data[str_p_id] = np.repeat(parti_data.split('_')[0], df_data.shape[0])
        df_all_parti_data = pd.concat([df_all_parti_data, df_data])
    
    print(df_all_parti_data.head(5))
    df_all_parti_data = df_all_parti_data[df_all_parti_data[str_date] == str_all].copy()
    df_sum_by_sensor = df_all_parti_data.groupby(str_data_name)[str_data].sum().reset_index()

    fig = px.bar(df_all_parti_data, x=str_data_name, y=str_data, color=str_p_id)
    fig.update_layout({
        "plot_bgcolor": "#FFFFFF",
        'yaxis':go.layout.YAxis(title=go.layout.yaxis.Title(text= str_data,
        font=dict(family='Times New Roman', size=font_size, color='#000000')),
        tickmode = 'array', color="#000000", tickfont=dict(family='Times New Roman', size=font_size)),})
    fig.write_html(loc_data_quality + 'overall_data_quality_per_parti.html')
    fig.show()

    df_sum_by_sensor = df_sum_by_sensor[~df_sum_by_sensor[str_data_name].isin(['Offbody', 'Wear dur 12 to 8', 'Non-wear dur since 1st data coll.'])]
    df_sum_by_sensor.replace({'Wear dur excl. 12 to 8 AM': 'Wear dur. (H)'}, inplace = True)
    df_sum_by_sensor[str_data] = df_sum_by_sensor[str_data].round(1)

    df_sum_by_sensor = df_sum_by_sensor[(df_sum_by_sensor['Data_Name'] != 'Wear dur. (H)') & (df_sum_by_sensor['Data_Name'] != 'Battery')]
    df_sum_by_sensor["Category"] = df_sum_by_sensor["Data_Name"].apply(lambda x: "SensorsStart" if x == "SensorsStart" else "Other")
    color_map = {"Other": "#C5D4DE", "SensorsStart": "#CC4D38"}


    fig = px.bar(df_sum_by_sensor, x=str_data_name, y=str_data, color='Category', color_discrete_map=color_map, text=str_data, barmode='group')

    fig.update_layout({
        "plot_bgcolor": "#FFFFFF",
        'yaxis':go.layout.YAxis(title=go.layout.yaxis.Title(text= '# of probe starts with data', font=dict(family='Times New Roman', size=font_size, color='#000000')), tickmode = 'array', color="#000000", tickfont=dict(family='Times New Roman', size=font_size)),
        'xaxis': go.layout.XAxis(title=go.layout.xaxis.Title(text= 'Sensor', font=dict(family='Times New Roman', size=font_size, color='#000000')), tickmode = 'array', color="#000000", tickfont=dict(family='Times New Roman', size=font_size))})
    fig.update_traces(textfont = dict(family="Times New Roman", size=font_size, color="black"))
    
    x_axis_data_order  = list(sorted(df_sum_by_sensor[str_data_name].unique()))
    x_axis_data_order.remove('SensorsStart')
    x_axis_data_order.append('SensorsStart')
    fig.update_xaxes(categoryorder='array', categoryarray= x_axis_data_order)
    fig.update_layout(xaxis_tickangle=-15)

    fig.write_image(loc_data_quality + 'overall_data_quality_sum_by_sensor.png')
    fig.write_html(loc_data_quality + 'overall_data_quality_sum_by_sensor.html')
    fig.show()

    fig = go.Figure()
    for data_name, color_code in zip(df_all_parti_data[str_data_name].tolist(), list_color):
        fig.add_trace(go.Box(
                        x = df_all_parti_data[df_all_parti_data[str_data_name] == data_name][str_data_name],
                        y = df_all_parti_data[df_all_parti_data[str_data_name] == data_name][str_data],
                        name= data_name,
                        # marker_color=color_code,
                        boxmean='sd'))
    fig.update_layout(yaxis_title=str_data, boxmode='group')
    
    fig.update_layout({
        "plot_bgcolor": "#FFFFFF",
        'yaxis':go.layout.YAxis(title=go.layout.yaxis.Title(text= str_data,
        font=dict(family='Times New Roman', size=font_size, color='#000000')),
        tickmode = 'array', color="#000000", tickfont=dict(family='Times New Roman', size=font_size))})

    fig.write_html(loc_data_quality + 'overall_data_quality_variance_by_parti_by_sensor.html')
    fig.show()

SAPIENS_DS_get_overall_data_quality()

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from rqa_for_other_sensors import hrv_rqa_for_other_sensors


start_index = 0
end_index = 600
segment_number = -1

random_df_acc = pd.read_pickle('... removed the location a bit for privacy .../Documents/SA39/SAPIENS/All Data Spring\'24/Data of Spring\'24/P413/Smartwatch_HeartRateDatum.pkl')
random_df_acc = random_df_acc[random_df_acc['HR'] != 0]
list_sense_data = random_df_acc['HR'].tolist()[start_index: end_index]

random_fig = go.Figure([go.Scatter(x=random_df_acc.iloc[start_index:end_index]['T'], y=random_df_acc.iloc[start_index: end_index]['HR'], mode='markers+lines')])
random_fig.update_layout({'plot_bgcolor': '#FFFFFF'}).show()
random_fig.write_image('... removed the location a bit for privacy .../Documents/SA39/SAPIENS/Presentation/plot_rqa_hr_'+str(segment_number)+'.jpeg')


# list_rri, list_rri_time = get_rri_rri_time(list_sense_data= list_sense_data)
# hrv_rqa({'RRI': list_rri * 1000, 'RRI_time': list_rri_time}, sampling_rate= 1, show=True,
#         image_length = 224, image_height= 224, loc_figure_to_save= '... removed the location a bit for privacy .../Documents/SA39/SAPIENS/Presentation/rqa_hr_'+str(segment_number)+'.jpeg')

In [ ]:
import os
from PIL import Image

def compress_under_size(size, file_path):
    '''file_path is a string to the file to be custom compressed
    and the size is the maximum size in bytes it can be which this 
    function searches until it achieves an approximate supremum'''

    quality = 90 #not the best value as this usually increases size

    current_size = os.stat(file_path).st_size

    print(current_size)

    while current_size > size or quality == 0:
        if quality == 0:
            os.remove(file_path)
            print("Error: File cannot be compressed below this size")
            break

        compress_pic(file_path, quality)
        current_size = os.stat(file_path).st_size
        quality -= 5


def compress_pic(file_path, qual):
    '''File path is a string to the file to be compressed and
    quality is the quality to be compressed down to'''
    picture = Image.open(file_path)
    dim = picture.size

    picture.save(file_path,"JPEG", optimize=True, quality=qual) 

    processed_size = os.stat(file_path).st_size

    return processed_size


compress_under_size(3445951, '... removed the location a bit for privacy .../Documents/SA39/SAPIENS/Presentation/rqa_hr_2.jpeg')

## `Presenting Models' Performances`

In [ ]:
if platform.system() == 'Darwin': # Mac == Darwin 🤣🤣🤣
    temp_loc = '... removed the location a bit for privacy .../Documents/SA39/'
else: # Ubuntu currently
    temp_loc = '... removed location a bit for privacy ...Documents/sa39/'
loc_chi_25_findings = os.path.join(temp_loc, 'SAPIENS/Conference/CHI\'26/Findings')
custom_order_of_hours = {'1 hour': 1, '1.5 hours': 2, '2 hours': 3, '2.5 hours': 4, '3 hours': 5}

str_accuracy = 'Accuracy'
str_algorithm = 'Algorithm'
str_accuracy_balanced = 'Balanced accuracy'
str_modalities = 'Modalities'
str_n_closest_sample = 'Closest n'
str_precision = 'Precision'
str_sensitivity = 'Sensitivity'
str_speicificity = 'Specificity'
str_window = 'Window'

def df_to_full_latex_table(df, caption="", label="", round_upto_n_decimal=2):
    float_format_latex = "{:."+str(round_upto_n_decimal)+"f}"
    tabular = df.to_latex(index=False, float_format=float_format_latex.format, column_format = "c" * len(df.columns))
    return (
        "\\begin{table}[htbp]\n"
        "\\centering\n"
        + tabular +
        (f"\\caption{{{caption}}}\n")
        + (f"\\label{{{label}}}\n")
        + "\\end{table}"
        )

def update_plotly_layout(fig, y_axis_title, x_axis_title, legend_orientation = 'h',  legend_x_pos=0.3, legend_y_pos=1.1):
    fig.update_layout(plot_bgcolor= "#FFFFFF",
                      yaxis=go.layout.YAxis(title=go.layout.yaxis.Title(text= y_axis_title, font=dict(family='Times New Roman', size=21, color='#000000')),
                                            color="#000000", tickfont=dict(family='Times New Roman', size=21)),
                      xaxis=go.layout.XAxis(title=go.layout.xaxis.Title(text= x_axis_title, font=dict(family='Times New Roman', size=21, color='#000000')), 
                                            color="#000000", tickfont=dict(family='Times New Roman', size=21)),
                      legend=dict(orientation=legend_orientation, x=legend_x_pos, y=legend_y_pos,  font=dict(family='Times New Roman', size=18), bgcolor="rgba(0,0,0,0)"))
    return fig


def plot_category_scatter(df, date_col, category_col, title, x_axis_title, y_axis_title, bool_contains_date_time = True):
    df = df.copy()

    if bool_contains_date_time:
        df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(by=date_col)
    fig = px.scatter(df, x=date_col, y=category_col, title=title)

    fig.update_traces( mode = 'markers+lines', line=dict(color="#CC4D38"), marker=dict(color="#000000"))
    fig.update_layout(plot_bgcolor="white", font=dict(size=18), margin=dict(l=20, r=20, t=40, b=40))
    fig.update_layout(
        plot_bgcolor= "#FFFFFF",
        yaxis=go.layout.YAxis(title=go.layout.yaxis.Title(text= y_axis_title, font=dict(family='Times New Roman', size=21, color='#000000'))),
        xaxis=go.layout.XAxis(title=go.layout.xaxis.Title(text= x_axis_title, font=dict(family='Times New Roman', size=21, color='#000000')), 
                              color="#000000", tickfont=dict(family='Times New Roman', size=21)))
    fig.show()


def clean_performance_df(df_perf):
    df_perf.rename(columns={str_balanced_acc: str_accuracy_balanced, 'recall': str_sensitivity, str_speicificity.lower(): str_speicificity, 
                                     'precision': str_precision, 'F1_score': 'F1', 'accuracy': str_accuracy, 
                                     'window_folder': str_window, 'multi_mode_comb': str_modalities,
                                     'n_closest_instances': str_n_closest_sample, 'model_name': str_algorithm}, inplace=True)
    
    if str_algorithm in df_perf.columns.tolist():
        df_perf[str_algorithm] = df_perf[str_algorithm].str.replace('Classifier', '').replace('LogisticRegression', 'Logit').replace('KNeighbors', 'KNN')

    return df_perf

def prepare_performance_dataframe():
    df_all_models_perf_1 = pd.read_excel(os.path.join(loc_chi_25_findings, 'Models_performance/1st_SAPIENS_fine_tuned_layers_incl_tiles_pred_False_explore_each_k_foldFalse_incl_predicted_instance_True_splitted_into3__first_few_windows_findings.xlsx'))
    df_all_models_perf_2 = pd.read_excel(os.path.join(loc_chi_25_findings, 'Models_performance/2nd_SAPIENS_fine_tuned_layers_incl_tiles_pred_False_explore_each_k_foldFalse_incl_predicted_instance_True_splitted_into3__first_few_windows_findings.xlsx'))
    df_all_models_perf_3 = pd.read_excel(os.path.join(loc_chi_25_findings, 'Models_performance/3rd_SAPIENS_fine_tuned_layers_incl_tiles_pred_False_explore_each_k_foldFalse_incl_predicted_instance_True_splitted_into3__first_few_windows_findings.xlsx'))
    df_all_models_perf_4 = pd.read_excel(os.path.join(loc_chi_25_findings, 'Models_performance/4th_SAPIENS_fine_tuned_layers_incl_tiles_pred_False_explore_each_k_foldFalse_incl_predicted_instance_True_splitted_into3__first_few_windows_findings.xlsx'))
    df_all_models_perf_5 = pd.read_excel(os.path.join(loc_chi_25_findings, 'Models_performance/5th_SAPIENS_fine_tuned_layers_incl_tiles_pred_False_explore_each_k_foldFalse_incl_predicted_instance_True_splitted_into3__first_few_windows_findings.xlsx'))

    df_meta_learners_peformance = pd.concat([df_all_models_perf_1, df_all_models_perf_2, df_all_models_perf_3, df_all_models_perf_4, df_all_models_perf_5], ignore_index=True)
    df_meta_learners_peformance.rename(columns={str_balanced_acc: str_accuracy_balanced, 'recall': str_sensitivity, str_speicificity.lower(): str_speicificity, 
                                                'precision': str_precision, 'F1_score': 'F1', 'accuracy': str_accuracy, 
                                                'window_folder': str_window, 'multi_mode_comb': str_modalities, 
                                                'n_closest_instances': str_n_closest_sample, 'model_name': str_algorithm}, inplace=True)

    df_meta_learners_peformance[str_window] = df_meta_learners_peformance[str_window].str.replace('_1_vs_2_10_split', '').str.replace('1_hours', '1 hour').str.replace('_', ' ')
    df_meta_learners_peformance[str_modalities] = df_meta_learners_peformance[str_modalities].str.replace('SAPIENS', '').str.replace('HeartRate', 'HR').str.replace('accel', 'Acc').str.replace('magnet', 'Magnet').str.replace('StepCount', 'Step')
    df_meta_learners_peformance[str_modalities] = df_meta_learners_peformance[str_modalities].str.replace('__', '_').str.replace('__', '_').str.replace(r"^_|_$", "", regex=True)
    df_meta_learners_peformance[str_algorithm] = df_meta_learners_peformance[str_algorithm].str.replace('Classifier', '').replace('LogisticRegression', 'Logit').replace('KNeighbors', 'KNN')
    print('Before removing duplicate. Currently, there should not be any duplicate', df_meta_learners_peformance.shape)
    df_meta_learners_peformance.drop_duplicates(subset=[str_window, str_modalities, str_algorithm, str_n_closest_sample], inplace=True) # to get the findings faster, sometimes, I ran the codes in different notebooks seperating each window. To make sure, even if same window is explored in multiple notebooks, those are removed, I set this statement. 
    print('After removing duplicate. Currently, there should not be any duplicate', df_meta_learners_peformance.shape)
    df_meta_learners_peformance.sort_values(by=str_accuracy_balanced, ascending=False, inplace=True)
    # df_meta_learners_peformance.head(10)

    return df_meta_learners_peformance

In [ ]:
from pandas.api.types import is_numeric_dtype

def create_table_presenting_best_models_performance_in_each_window():
    df_meta_learners_peformance = prepare_performance_dataframe()
    set_windows = df_meta_learners_peformance[str_window].unique()
    if sorted(set_windows) != sorted(custom_order_of_hours.keys()):
        raise ValueError('Seems like you did not explore the 5 windows: 1, 1.5, 2, 2.5, 3 hours. Check please.')
    
    df_best_model_in_each_window = pd.DataFrame()
    for window in custom_order_of_hours.keys():
        df_best_model_in_each_window = pd.concat([df_best_model_in_each_window, 
                                                  pd.DataFrame([df_meta_learners_peformance[df_meta_learners_peformance[str_window] == window].sort_values(by=[str_accuracy_balanced], ascending=False).iloc[0]])])
    df_best_model_in_each_window.reset_index(inplace=True, drop=True)
    df_best_model_in_each_window.drop(columns=['n_segments_of_a_day', 'global_n_minor'], inplace=True)
    list_cols = df_best_model_in_each_window.columns.tolist()
    list_cols.remove(str_window); list_cols.remove(str_modalities)
    list_cols = [str_window] + list_cols + [str_modalities]
    df_best_model_in_each_window[str_modalities] = df_best_model_in_each_window[str_modalities].str.replace('_', '\_')


    display(HTML(df_best_model_in_each_window.to_html()))
    print(df_best_model_in_each_window[list_cols].to_latex(index=False, float_format="{:.2f}".format) )

create_table_presenting_best_models_performance_in_each_window()

In [ ]:
import plotly.express as px

def plot_model_performance(df, column_to_melt, y_axis='Percentage'):
    list_colors = ['#FFCCCC', '#FFA49A', '#FA7861', '#F04C2D', '#DB2800']
    df_long = df.melt(id_vars=[column_to_melt], var_name="Metric", value_name="Value")    
    fig = px.bar(df_long, x="Metric", y='Value', color=column_to_melt, barmode="group", color_discrete_sequence= list_colors, text="Value", title="")
    fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
    
    fig.update_layout(legend=dict(
        orientation="h",           # make legend horizontal
        yanchor="bottom",
        y=95,                   # move legend below plot
        xanchor="center",
        x=0.5,
        font=dict(family="Times New Roman", size=16)
    ))
    fig.update_layout(
        plot_bgcolor= "#FFFFFF",
        yaxis=go.layout.YAxis(title=go.layout.yaxis.Title(text= y_axis, font=dict(family='Times New Roman', size=21, color='#000000')),
                              tickfont=dict(family='Times New Roman', size=21)),
        xaxis=go.layout.XAxis(title=go.layout.xaxis.Title(text= "Performance Metric", font=dict(family='Times New Roman', size=21, color='#000000')), 
                            color="#000000", tickfont=dict(family='Times New Roman', size=21)))
    fig.update_traces(textfont = dict(family='Times New Roman', size=21, color='#000000'))

    fig.show()
    return fig


def visualize_different_windows_peformances():
    df_meta_learners_peformance = prepare_performance_dataframe()
    best_models_perf = df_meta_learners_peformance[ (df_meta_learners_peformance[str_n_closest_sample] == 5) & (df_meta_learners_peformance[str_algorithm] == 'KNN') 
                                                   & (df_meta_learners_peformance[str_modalities] == 'HR_Light_Acc_Magnet')] # (df_meta_learners_peformance[str_window] == '2.5_hours_1_vs_2_10_split')
    best_models_perf = best_models_perf[[ str_accuracy_balanced, str_sensitivity, str_window]]
    best_models_perf.sort_values(by=[str_window], key=lambda x: x.map(custom_order_of_hours), inplace=True)
    plot_model_performance(best_models_perf, str_window)


def visualize_different_meta_learners_performance():
    df_meta_learners_peformance = prepare_performance_dataframe()
    best_models_perf = df_meta_learners_peformance[ (df_meta_learners_peformance[str_n_closest_sample] == 5) & (df_meta_learners_peformance[str_window] == '2.5 hours') 
                                                   & (df_meta_learners_peformance[str_modalities] == 'HR_Light_Acc_Magnet')] # (df_meta_learners_peformance[str_window] == '2.5_hours_1_vs_2_10_split')
    best_models_perf = best_models_perf[[str_precision, str_sensitivity, str_speicificity, str_accuracy_balanced, 'F1', str_algorithm]]
    best_models_perf = best_models_perf[best_models_perf[str_algorithm] != 'LinearSVC'] # removed the Linear SVC as I used it somewhat inappropriately. Linear SVC is used for large dataset. Since there were 2700+ samples only in our dataset, I am not sure whether we can call it as a large dataset. And, also our meta learner uses <10 samples to be trained with. So, there we NEVER can call it large dataset. Also, LinearSVC does not support to predict probability which is causing problem to find out ROC-AUC and Precision-Recall curve. Hence, to keep the analysis consistent, I removed it.
    best_models_perf.sort_values(by=[str_algorithm], inplace=True)
    plot_model_performance(best_models_perf, str_algorithm)

def visualize_performances_of_different_modalities():
    df_meta_learners_peformance = prepare_performance_dataframe()
    best_models_perf = df_meta_learners_peformance[(df_meta_learners_peformance[str_n_closest_sample] == 5) & (df_meta_learners_peformance[str_algorithm] == 'KNN') & 
                                                   (df_meta_learners_peformance[str_window] == '2.5 hours')]
    best_models_perf = best_models_perf[[ str_accuracy_balanced, str_sensitivity, str_modalities]]
    best_models_perf.sort_values(by=[str_modalities], inplace=True)
    plot_model_performance(best_models_perf, str_modalities)

visualize_different_windows_peformances()
visualize_different_meta_learners_performance()
visualize_performances_of_different_modalities()

## `Ablation study`

In [ ]:
def present_performance_based_on_ablation_study():
    df_ablation_files = pd.DataFrame()

    for perf_file in os.listdir(os.path.join(loc_chi_25_findings, 'Models_performance')):
        if 'ablation' in perf_file:
            __ = pd.read_excel(os.path.join(loc_chi_25_findings, 'Models_performance', perf_file))
            __[str_filename] = np.repeat('.'.join(perf_file.split('_')[:5]), __.shape[0])
            df_ablation_files = pd.concat([df_ablation_files, __])

    df_ablation_files.drop(columns=['multi_mode_comb', 'global_n_minor', 'window_folder', 'model_name', 'n_segments_of_a_day', 'n_closest_instances'], inplace=True)
    df_ablation_files.rename(columns={str_balanced_acc: 'BA', 'recall': str_sensitivity, str_speicificity.lower(): str_speicificity, 
                                                'precision': str_precision, 'F1_score': 'F1', 'accuracy': str_accuracy, 
                                                'window_folder': str_window, 'multi_mode_comb': str_modalities, 
                                                'n_closest_instances': str_n_closest_sample, 'model_name': str_algorithm}, inplace=True)
    print(df_ablation_files)
    print(df_ablation_files.to_latex(index=False, float_format="{:.2f}".format))

present_performance_based_on_ablation_study()

## `Performance in low sampling rate`

In [ ]:
def present_performance_on_low_sampling_rate():
    df_low_sampling_perf = pd.DataFrame()

    for perf_file in os.listdir(os.path.join(loc_chi_25_findings, 'Models_performance')):
        if 'low_sample_' in perf_file:
            __ = pd.read_excel(os.path.join(loc_chi_25_findings, 'Models_performance', perf_file))
            __[str_filename] = np.repeat('.'.join(perf_file.split('_')[:5]), __.shape[0])
            df_low_sampling_perf = pd.concat([df_low_sampling_perf, __])

    df_low_sampling_perf.drop(columns=['multi_mode_comb', 'global_n_minor', 'window_folder', 'model_name', 'n_segments_of_a_day', 'n_closest_instances'], inplace=True)
    df_low_sampling_perf.rename(columns={str_balanced_acc: 'BA', 'recall': str_sensitivity, str_speicificity.lower(): str_speicificity, 
                                                'precision': str_precision, 'F1_score': 'F1', 'accuracy': str_accuracy, 
                                                'window_folder': str_window, 'multi_mode_comb': str_modalities, 
                                                'n_closest_instances': str_n_closest_sample, 'model_name': str_algorithm}, inplace=True)
    print(df_low_sampling_perf.to_latex(index=False, float_format="{:.2f}".format))

present_performance_on_low_sampling_rate()

## Exploring the resource consumption

In [ ]:
import plotly.express as px

def visualize_resource_consumption():
    str_peak_memory = 'Peak memory'
    str_time_in_sec = 'Time (in sec)'

    loc_resource_consump = os.path.join(loc_chi_25_findings, 'Resource_consumption')
    for res_consump_file in sorted(os.listdir(loc_resource_consump)):
        if 'DS_' in res_consump_file or '0.5' in res_consump_file:
            continue
        
        if 'time' in res_consump_file:
            x_axis = 'Time (in seconds)'
            data_column = str_time_in_sec
        else:
            x_axis = 'RAM (in GB)'
            data_column = str_peak_memory

        df_res = pd.read_excel(os.path.join(loc_resource_consump, res_consump_file.replace('~$', '')))
        if 'time' not in res_consump_file:
            df_res[data_column] = df_res[data_column]/1024

        fig = px.histogram(df_res[data_column], title=res_consump_file, width=600, height=400)
        fig.update_layout(xaxis_title=x_axis, yaxis_title='Count',  legend_title="",
                          uniformtext_minsize=8, uniformtext_mode='hide', plot_bgcolor = '#FFFFFF')

        fig.add_vline(
            x=df_res[data_column].mean(),
            line_dash="dot",   # dotted line
            line_color="black",
            line_width = 4,
            annotation_text=f"Mean = {df_res[data_column].mean():.4f}",
            annotation_position="top"
        )
                          
        fig.show()
        

visualize_resource_consumption()

## `Variation of state anxiety within the day`

In [ ]:
def visualize_variation_of_state_anxiety_within_the_day():
    df_state_anxiety = get_cleaned_state_anxiety_SAPIENS(crt_state_anxiety_classification_approach)
    for pid in df_state_anxiety[str_p_id].unique():
        if pid != "465": # explored and found it to be great to present
            continue

        sub_df_anxiety = df_state_anxiety[df_state_anxiety[str_p_id] == pid].copy()

        plot_category_scatter(df = sub_df_anxiety, date_col= str_sensus_submission_time, x_axis_title='Date-time', y_axis_title='State anxiety', category_col= str_category, title = pid)
visualize_variation_of_state_anxiety_within_the_day()

## `Time Difference Among the consescutive EMAs`

In [ ]:
import pandas as pd
import plotly.express as px

import pandas as pd
import plotly.express as px

def plot_time_gap_hist(date_col, pid_col):
    df = get_cleaned_state_anxiety_SAPIENS(crt_state_anxiety_classification_approach)

    data = df[[pid_col, date_col]].copy()
    data[date_col] = pd.to_datetime(data[date_col])
    data = data.sort_values(by=[pid_col, date_col])

    gaps = data.groupby(pid_col)[date_col].diff()
    gaps_hours = gaps.dt.total_seconds() / 3600.0

    if gaps_hours[gaps_hours > 50].shape[0] == 1: # there is just a single observation
        gaps_hours = gaps_hours[gaps_hours <= 50]

    gaps_hours = gaps_hours.dropna()

    fig = px.histogram(x=gaps_hours, color_discrete_sequence=['#CC4D38'],title="Distribution of consecutive time gaps (hours) among state anxiety responses")

    for h in [1, 2, 3]:
        fig.add_vline(x=h, line_dash="dot", line_color="black", line_width=2, annotation_text=f"{h}h", annotation_position="top",
                      annotation_font=dict(family="Times New Roman", size=21, color="black"))
    fig.update_layout(plot_bgcolor="white", font=dict(size=18))
    fig.update_layout( plot_bgcolor= "#FFFFFF",
                      yaxis=go.layout.YAxis(title=go.layout.yaxis.Title(text= 'Count', font=dict(family='Times New Roman', size=21, color='#000000'))),
                      xaxis=go.layout.XAxis(title=go.layout.xaxis.Title(text='Time difference (hours)', font=dict(family='Times New Roman', size=21, color='#000000')), 
                                            color="#000000", tickfont=dict(family='Times New Roman', size=21)))
    fig.show()

plot_time_gap_hist(date_col= str_sensus_submission_time, pid_col= str_p_id)

## `Generalization`

In [ ]:
def present_generalization_findings():
    str_criteria = 'Criteria'

    df_tiles_18_perf = pd.read_excel('... removed the location a bit for privacy .../Documents/SA39/SAPIENS/Conference/CHI\'26/Findings/Models_performance/TILES_18_fine_tuned_layers_incl_tiles_pred_False_explore_each_k_foldFalse_incl_predicted_instance_True_splitted_into3__first_few_windows_findings.xlsx')
    df_tiles_18_perf.rename(columns={str_balanced_acc: str_accuracy_balanced, 'recall': str_sensitivity, str_speicificity.lower(): str_speicificity, 
                                     'precision': str_precision, 'F1_score': 'F1', 'accuracy': str_accuracy, 
                                     'window_folder': str_window, 'multi_mode_comb': str_modalities,
                                     'n_closest_instances': str_n_closest_sample, 'model_name': str_algorithm}, inplace=True)
    df_tiles_18_perf[str_algorithm] = df_meta_learners_peformance[str_algorithm].str.replace('Classifier', '').replace('LogisticRegression', 'Logit').replace('KNeighbors', 'KNN')


    df_best_models_tiles = pd.concat([df_tiles_18_perf.sort_values(by=str_accuracy_balanced, ascending=False).head(1), # do NOT change the order. I mean the first row should be based on BA. Otherwise the values in Criteria column will be misleading
                                      df_tiles_18_perf.sort_values(by=str_sensitivity, ascending=False).head(1)])
    df_best_models_tiles[str_criteria] = [str_accuracy_balanced, str_sensitivity]
    df_best_models_tiles.drop(columns=[str_window, str_modalities, 'n_segments_of_a_day', 'global_n_minor'], inplace=True)
    list_columns = df_best_models_tiles.columns.tolist()
    list_columns.remove(str_criteria)
    list_columns = [str_criteria] + list_columns    

    print(df_to_full_latex_table(df_best_models_tiles[list_columns], caption='Generalization of the meta-learning approach on a public dataset TILES-18.',
                                 label= 'generalization_on_tiles18'))

present_generalization_findings()

In [ ]:
# # for a strange unknown reason, I can not unzip the large data file in mac. thus, program...
import zipfile
path_to_zip_file = '... removed the location a bit for privacy .../Documents/SA39/Katha/Social Interaction/Model dev/mm.zip'
with zipfile.ZipFile(path_to_zip_file, 'r') as zip_ref:
    zip_ref.extractall('... removed the location a bit for privacy .../Documents/SA39/Katha/Social Interaction/Model dev/')

## Ablation for the meta-learner

In [ ]:
def ablation_for_meta_learner():
    str_criteria = 'Criteria'
    df_meta_learner_ablation = pd.DataFrame()

    for meta_abl_perf_file in sorted(os.listdir(os.path.join(loc_chi_25_findings, 'Models_performance'))):
        crt_criteria_name = ''

        if ('_incl_predicted_instance_True_' in meta_abl_perf_file) and ('__global_person_False_add_prev_days_False' in meta_abl_perf_file): # _incl_predicted_instance_True_ means personalization module is activated. To be aligned with previous file names, I have not changed "_incl_predicted_instance_True_" it to something like personalziation_module
            crt_criteria_name = 'Our meta-learner w/o addition of past test instances'
        elif ('_incl_predicted_instance_True_' in meta_abl_perf_file) and ('__global_person_True_add_prev_days_True' in meta_abl_perf_file):
            crt_criteria_name = 'Personalize from global instances for each test (adding past test instances)'
        elif ('_incl_predicted_instance_True_' in meta_abl_perf_file) and ('__global_person_True_add_prev_days_False' in meta_abl_perf_file):
            crt_criteria_name = 'Personalize from global instances for each test (w/o adding past test instances)'
        elif ('_incl_predicted_instance_False_' in meta_abl_perf_file) and ('__global_person_False_add_prev_days_False' in meta_abl_perf_file):
            crt_criteria_name = 'Non-personalize'
        elif ('_incl_predicted_instance_True_' in meta_abl_perf_file) and ('__global_person_False_add_prev_days_True' in meta_abl_perf_file):
            crt_criteria_name = 'AnxietySense'
        elif '_incl_predicted_instance_False_' in meta_abl_perf_file:
            raise ValueError(f'This means \"__global_person_False_add_prev_days_False in in meta_abl_perf_file\" is not True which is not supposed to happen. Check that performance file.{meta_abl_perf_file}')
        
        if crt_criteria_name != '':
            print(meta_abl_perf_file)
            df_meta_abl_perf = pd.read_excel(os.path.join(loc_chi_25_findings, 'Models_performance', meta_abl_perf_file))
            df_meta_abl_perf[str_criteria] = crt_criteria_name
            df_meta_learner_ablation = pd.concat([df_meta_learner_ablation, df_meta_abl_perf])
    
    df_meta_learner_ablation  = clean_performance_df(df_meta_learner_ablation)
    df_meta_learner_ablation.drop(columns=[str_window, str_modalities, 'n_segments_of_a_day', 'global_n_minor', 'Closest n', '# of EMAs', 'Algorithm'], inplace=True)
    display(HTML(df_meta_learner_ablation.to_html()))
    print(df_to_full_latex_table(df= df_meta_learner_ablation, caption='Ablation of our meta learning algorithm.', label='ablation_of_meta'))

ablation_for_meta_learner()

## Variation in performance with the variation in closest number of samples

In [ ]:
def visualize_variation_in_perform_with_closest_n():
    df_meta_perf = prepare_performance_dataframe()
    df_meta_perf = df_meta_perf[(df_meta_perf[str_modalities] == 'HR_Light_Acc_Magnet') & (df_meta_perf[str_window] == '2.5 hours') & (df_meta_perf[str_algorithm] == 'KNN')]
    plot_category_scatter(df=df_meta_perf, date_col= 'Closest n', category_col= str_accuracy_balanced, title= '', x_axis_title='Closet n', y_axis_title= str_accuracy_balanced, bool_contains_date_time= False)

visualize_variation_in_perform_with_closest_n()

## Number of samples required

In [ ]:
from pandas.api.types import is_numeric_dtype

def visualize_the_number_of_samples_required():
    list_df_perf_based_on_interval = []

    loc_low_sampling_rate_info = '... removed the location a bit for privacy .../Documents/SA39/SAPIENS/Conference/CHI\'26/Low_sampling_rate'
    dic_double_safe_guard_n_ema = {0.5: 2586, 1: 2713}
    str_n_samples = '# of samples from 4 sensors'
    str_correct_detection = 'Correct detection'
    str_nth_ema = 'EMA Index'

    for low_sample_rate in [0.5, 1]:
        list_df_n_samples_correct = []

        perf_filename = 'ResNet_18_RQA__fine_tuned_layers__SAPIENS__on_the_same_day__KNeighborsClassifier_incl_predicted_instance_True_global_person_False_add_prev_days_True_including_predicted_class_proba.xlsx'
        df_fine_tuned_perf = pd.read_excel(os.path.join('... removed the location a bit for privacy .../Documents/SA39/SAPIENS/State Anxiety/2.5_hours_1_vs_2_10_split/multi_modal/splitted_into3/SAPIENS_HeartRate_Light__accel___magnet_/Performance/sampling_rate_'+str(low_sample_rate)+'_ResNet_18/iter_130__a9dfbe4d-4076-48c7-a72b-342fe4c12514__chunk_test_5_val_2__fine_tuned_all_layers', perf_filename))

        if df_fine_tuned_perf.shape[0] != dic_double_safe_guard_n_ema.get(low_sample_rate):
            raise ValueError(f'Should not there be {dic_double_safe_guard_n_ema.get(low_sample_rate)} rows! Check please!')
        
        with open(os.path.join(loc_low_sampling_rate_info, str(low_sample_rate)+'Hz__SAPIENS.pkl'), 'rb') as f:
            dict_sensor_ema_number_of_sample = pickle.load(f)
        
        for index, perf_row in df_fine_tuned_perf.iterrows():
            list_n_samples_for_each_ema = []
            for temp_sensor in ['Smartwatch_LinearAccelerationDatum', 'Smartwatch_MagnetometerDatum', 'Smartwatch_LightDatum', 'Smartwatch_HeartRateDatum']:
                file_name_without_segment = str(int(perf_row[str_actual_class]))+'__'+ str(int(perf_row[str_p_id]))+ '__'+ str(perf_row[str_timestamp])+'____'+temp_sensor.replace('Smartwatch_', '').replace('Datum', '')+'__'+dic_sensor_data_name.get(temp_sensor)[0] +'.png'
                list_n_samples_for_each_ema.extend(dict_sensor_ema_number_of_sample[temp_sensor][file_name_without_segment])
            list_df_n_samples_correct.append([sum(list_n_samples_for_each_ema), int(perf_row[str_actual_class] == perf_row[str_predicted_class]), perf_row[str_actual_class], perf_row[str_predicted_class], perf_row[str_predicted_proba], perf_row[str_timestamp], 'P'+str(int(perf_row[str_p_id]))])
        
        df_n_samples_correct = pd.DataFrame(list_df_n_samples_correct, columns=[str_n_samples, str_correct_detection, str_actual_class, str_predicted_class, str_predicted_proba, str_timestamp, str_p_id])
        df_n_samples_correct = df_n_samples_correct.sort_values(by=str_timestamp).reset_index(drop=True)
        df_n_samples_correct[str_nth_ema] = list(range(1, df_n_samples_correct.shape[0]+1))
        df_n_samples_correct[str_correct_detection] = df_n_samples_correct[str_correct_detection].map({0: 'No', 1: 'Yes'})

        print('950 <<<<', low_sample_rate, set(df_n_samples_correct[df_n_samples_correct[str_nth_ema] <= 900][str_p_id].unique())- set(dict_list_pid_with_interval.get('10')))

        for interval in ['5', '10']:
            df_n_min_interval_n_samples_correct = df_n_samples_correct[df_n_samples_correct[str_p_id].isin(dict_list_pid_with_interval.get(interval))].copy()
            temp_perf, temp_perf_columns = clf_report(df_n_min_interval_n_samples_correct[str_actual_class].tolist(), df_n_min_interval_n_samples_correct[str_predicted_class].tolist(), 
                                                      bool_print=False, bool_return_columns=True, bool_auc_roc=True, bool_prec_recall=True, list_predicted_proba= df_n_min_interval_n_samples_correct[str_predicted_proba])
            temp_perf.extend([df_n_min_interval_n_samples_correct[str_n_samples].mean(), df_n_min_interval_n_samples_correct[str_n_samples].std(ddof=1),
                              interval, low_sample_rate, df_n_min_interval_n_samples_correct.shape[0], len(df_n_min_interval_n_samples_correct[str_p_id].unique()), Counter(df_n_min_interval_n_samples_correct[str_actual_class].tolist())])
            temp_perf_columns.extend(['# of samples (Mean)', '# of samples (SD)', 'Interval', 'Sampling rate', '# of EMAs', '# of Ps', 'Class Distr.'])
            list_df_perf_based_on_interval.append(temp_perf)

            ### print(stats.spearmanr(df_n_samples_correct[str_n_samples], df_n_samples_correct[str_predicted_proba]))
            ### print(df_n_samples_correct[(df_n_samples_correct[str_nth_ema] < 919) & (df_n_samples_correct[str_correct_detection] == 'Yes')][str_p_id].unique())

        for temp_minute in [10, 20, 30]:
            temp_df_data_of_certain_minutes = df_n_samples_correct[(df_n_samples_correct[str_n_samples] < (low_sample_rate * 60 * temp_minute * 4))].copy()
            print(f'# of EMAs within {temp_minute} minutes where the number of samples is less than {low_sample_rate * 60 * temp_minute * 4}: ', temp_df_data_of_certain_minutes.shape, df_n_samples_correct.shape)
            print(f'Performance in {temp_minute}: ', clf_report(true_labels= temp_df_data_of_certain_minutes[str_actual_class].tolist(),
                                                                predicted_val_list= temp_df_data_of_certain_minutes[str_predicted_class].tolist(), bool_print= False))

        fig = px.scatter(df_n_samples_correct, x = str_nth_ema, y = str_n_samples, color= str_correct_detection, color_discrete_map={'Yes': '#D3D3D3', 'No': '#CC4D38'})
        fig.update_layout(plot_bgcolor = '#FFFFFF')
        fig.update_traces(marker=dict(size=12))
        fig.for_each_trace(lambda trace: trace.update(opacity=0.6) if trace.name == "No" else ())

        fig.add_hline(y=(low_sample_rate * 60 * 10 * 4), x0=0.05, x1=0.95, line_dash="dot", line_color="black", line_width=2.5, annotation_text='10 minutes', annotation_position="right",
                      annotation_font=dict(family="Times New Roman", size=21, color="black"))
        
        fig.add_hline(y=(low_sample_rate * 60 * 20 * 4), x0=0.05, x1=0.95, line_dash="dot", line_color="black", line_width=2.5, annotation_text='20 minutes', annotation_position="right",
                      annotation_font=dict(family="Times New Roman", size=21, color="black"))
        
        fig.add_hline(y=(low_sample_rate * 60 * 30 * 4), x0=0.05, x1=0.95, line_dash="dot", line_color="black", line_width=2.5, annotation_text='30 minutes', annotation_position="right",
                      annotation_font=dict(family="Times New Roman", size=21, color="black"))
        fig = update_plotly_layout(fig, y_axis_title=str_n_samples, x_axis_title= str_nth_ema, legend_orientation='v', legend_y_pos=0.9, legend_x_pos=0.1)
        fig.show()

    df_perf_interval = pd.DataFrame(data= list_df_perf_based_on_interval, columns= temp_perf_columns)
    df_perf_interval = clean_performance_df(df_perf_interval)
    perf_interval_col_numeric_list = [col for col in df_perf_interval.columns.tolist() if is_numeric_dtype(df_perf_interval[col])]
    df_perf_interval[perf_interval_col_numeric_list] = df_perf_interval[perf_interval_col_numeric_list].round(1)

    df_perf_interval.drop(columns=['# of Ps', 'Class Distr.'], inplace=True)
    print(df_to_full_latex_table(df_perf_interval, caption='Performance when most interval among consecutive probe starting times was 5 minutes and 10 minutes.', 
                                 label='perform_5_min_10_min_interval', round_upto_n_decimal=1))


visualize_the_number_of_samples_required()

## After including TILES data to predict SAPIENS state anxiety

In [ ]:
def predict_sapiens_anxiety_after_including_tiles_18_data():
    df_perf_after_incl_tiles = pd.read_excel(os.path.join(loc_chi_25_findings, 'Models_performance', 'SAPIENS_fine_tuned_layers_incl_tiles_pred_True_explore_each_k_foldFalse_incl_predicted_instance_True_splitted_into3__global_person_False_add_prev_days_True_first_few_windows_findings.xlsx'))
    df_perf_after_incl_tiles = clean_performance_df(df_perf_after_incl_tiles)
    df_perf_after_incl_tiles.drop(columns=[str_modalities, 'n_segments_of_a_day', 'Window', str_algorithm, 'global_n_minor', 'Closest n'], inplace=True)

    print(df_to_full_latex_table(df_perf_after_incl_tiles, caption='Performance after including predicted probabilities on TILES-18, along with SAPIENS predicted probabilities, to predict state anxiety of SAPIENS dataset.'))

predict_sapiens_anxiety_after_including_tiles_18_data()

# `Baseline Models`

## TILES

In [ ]:
loc_baseline_models_chi = '... removed the location a bit for privacy .../Documents/SA39/SAPIENS/Conference/CHI\'26/Findings/Models_performance/Baseline_models_performance/'

In [ ]:
import pandas as pd 
import os
import numpy as np

b_loc_fitbit_features = '... removed the location a bit for privacy .../Documents/SA39/Hridita/Data/Tiles18/fitbit/daily-summary/' # b: baseline :)

def get_fitbit_features():
    df_features_all_p = pd.DataFrame()

    for pid in os.listdir(b_loc_fitbit_features):
        if (pid in 'README.md') or ('DS_Store' in pid):
            continue
        df_single_p_data = pd.read_csv(os.path.join(b_loc_fitbit_features, pid))
        df_single_p_data[str_p_id] = np.repeat(pid, df_single_p_data.shape[0])
        df_features_all_p = pd.concat([df_features_all_p, df_single_p_data])
    
    df_features_all_p[str_date] = pd.to_datetime(df_features_all_p[str_timestamp]).dt.date
    df_features_all_p[str_p_id] = df_features_all_p[str_p_id].str.replace('.csv.gz', '')
    
    return df_features_all_p

def get_processed_fitbit_anxiety_data():
    b_df_fitbit_dataset = get_fitbit_features()
    b_df_fitbit_dataset[str_p_id + str_date] = b_df_fitbit_dataset[str_p_id].astype(str) + b_df_fitbit_dataset[str_date].astype(str)
    b_df_fitbit_dataset = b_df_fitbit_dataset[~b_df_fitbit_dataset.duplicated(subset=[str_p_id + str_date], keep='last')] # I found some duplicate items. Seems like they have exactly same values. I can not remember any description regarding the duplicate items on the TILES-18 dataset root paper (https://www.nature.com/articles/s41597-020-00655-3) or the ICASSP paper (https://ieeexplore.ieee.org/document/10096235) we are following to build the baseline model. Thus, I just simply dropped one.

    b_df_anxiety = get_TILES_18_cleand_anxiety_demo_df(crt_state_anxiety_classification_approach).get(str_anxiety).rename(columns={str_tiles_pid: str_p_id})
    b_df_fitbit_dataset = pd.merge(b_df_fitbit_dataset, b_df_anxiety[[str_p_id, str_date, str_category]], on=[str_p_id, str_date], how='inner')

    # to make sure that we used the exactly same set of days and participants as we used to train the meta-learner (which we found to have a good performance), we are dropping the other days and pids
    b_our_base_model_performance = pd.read_excel('... removed the location a bit for privacy .../Documents/SA39/SAPIENS/Daily Anxiety/on_the_same_day/multi_modal/splitted_into3/Tiles18_multi_modal_multi_modal/Performance/ResNet_18/predicted_proba_fine_tuned_ResNet.xlsx')
    b_our_base_model_performance[str_date] = pd.to_datetime(b_our_base_model_performance[str_timestamp], unit='s', utc=True).dt.tz_convert(pytz.timezone('America/Los_Angeles')).dt.date # Empirically, I found there are 2 time zones in anxiety data which are -7 and -8
    b_our_base_model_performance[str_p_id + str_date] = b_our_base_model_performance[str_p_id].astype(str) + b_our_base_model_performance[str_date].astype(str)
    b_df_fitbit_dataset[str_p_id + str_date] = b_df_fitbit_dataset[str_p_id].astype(str) + b_df_fitbit_dataset[str_date].astype(str)
    b_df_fitbit_dataset = b_df_fitbit_dataset[b_df_fitbit_dataset[str_p_id + str_date].isin(b_our_base_model_performance[str_p_id + str_date].tolist())]

    if  b_df_fitbit_dataset.shape[0] != b_our_base_model_performance.shape[0]:
        print('\n\nShape of the final dataframe: ', b_df_fitbit_dataset.shape, b_our_base_model_performance.shape)
        raise ValueError('There should be exactly same number of rows as there is an overlap of anxiety and Fitbit data across b_df_fitbit_dataset and b_our_base_model_performance')
    
    return b_df_fitbit_dataset

In [ ]:
columns_to_drop_for_features = [str_date, str_category, str_p_id]
list_fitbit_features = ['Cardio_caloriesOut', 'Cardio_max', 'Cardio_min', 'Cardio_minutes',
                        'Fat Burn_caloriesOut', 'Fat Burn_max', 'Fat Burn_min', 'Fat Burn_minutes', 
                        'NumberSteps', 'Out of Range_caloriesOut', 'Out of Range_max', 'Out of Range_min', 'Out of Range_minutes', 'Peak_max', 'Peak_min', 
                        'RestingHeartRate', 
                        'Sleep1Efficiency', 'Sleep1MinutesAwake', 'Sleep1MinutesStageDeep', 'Sleep1MinutesStageLight', 'Sleep1MinutesStageRem', 
                        'Sleep1MinutesStageWake', 'SleepMinutesAsleep', 'SleepMinutesInBed', 'SleepPerDay']

def get_impute_and_scale_feature_values(bool_all_features):
    b_df_fitbit_dataset =  get_processed_fitbit_anxiety_data()
    columns_with_missing_values = b_df_fitbit_dataset.columns[b_df_fitbit_dataset.isna().any()].tolist()

    for feature_col in columns_with_missing_values:
        if feature_col not in list_fitbit_features:
            continue
        
        mean_value = b_df_fitbit_dataset[feature_col].mean(skipna=True)
        b_df_fitbit_dataset[feature_col].fillna(mean_value, inplace=True)
    
    if b_df_fitbit_dataset[list_fitbit_features].isna().sum().sum() > 0:
        raise ValueError('After imputation of the selected features to explore, there should not be any NaN values.')
    
    if bool_all_features:
        selected_fitbit_features = list_fitbit_features.copy()
    else:
        selected_fitbit_features = ['RestingHeartRate']
    
    ##### Scaling
    scaler = StandardScaler()
    b_df_fitbit_dataset[selected_fitbit_features] = scaler.fit_transform(b_df_fitbit_dataset[selected_fitbit_features])

    return b_df_fitbit_dataset

def model_development_and_validation(bool_all_features):
    
    if len(list_fitbit_features) != 25:
        raise ValueError("The authors used 25 features of daily activities and physiology. So there should be 25 features!")
    
    if bool_all_features:
        selected_fitbit_features = list_fitbit_features.copy()
    else:
        selected_fitbit_features = ['RestingHeartRate']
        
    b_df_fitbit_dataset = get_impute_and_scale_feature_values(bool_all_features)
    list_predicted_class = []
    list_true_class = []

    for nth_p, pid in enumerate(sorted(b_df_fitbit_dataset[str_p_id].unique())):
        df_train = b_df_fitbit_dataset[b_df_fitbit_dataset[str_p_id] != pid].copy()
        df_test = b_df_fitbit_dataset[b_df_fitbit_dataset[str_p_id] ==  pid].copy()

        model = RandomForestClassifier(random_state= random_state)
        model = model.fit(X=df_train[selected_fitbit_features], y=df_train[str_category])
        single_p_predicted_class = model.predict(X= df_test[selected_fitbit_features])

        list_true_class.extend(df_test[str_category].tolist())
        list_predicted_class.extend(single_p_predicted_class)
    
        print(nth_p+1, clf_report(true_labels= list_true_class, predicted_val_list= list_predicted_class, bool_print=False))
    
    list_performances, list_columns = clf_report(true_labels= list_true_class, predicted_val_list= list_predicted_class, bool_print= False, bool_return_columns = True)
    list_performances.append(len(list_predicted_class)) ;list_columns.append('# of EMA')

    pd.DataFrame(data=[list_performances], columns= list_columns).to_excel(os.path.join(loc_baseline_models_chi, 'baseline_all_fitbit_features_' +str(bool_all_features)+ '_Tiles18_ICASSP_authors_paper_based_model.xlsx'), index=False)

model_development_and_validation(bool_all_features= True)

In [ ]:
from sklearn.dummy import DummyClassifier

def random_baselines_performance(random_model_strategy, loc_perf_file):
    df_meta_learners_data = pd.read_excel(loc_perf_file)
    
    temp_dataset = ''
    if str_daily_anxiety in loc_perf_file:
        temp_dataset  = str_tiles_18
    else:
        temp_dataset = str_sapiens

    list_actual_class_all_p = []
    list_predicted_class_all_p = []
    list_feature_cols = [str_segment+'1',  str_segment+'2', str_segment+'3']

    for pid in sorted(df_meta_learners_data[str_p_id].unique()):
        train_df = df_meta_learners_data[df_meta_learners_data[str_p_id] != pid].copy()
        test_df = df_meta_learners_data[df_meta_learners_data[str_p_id] == pid].copy()
        
        random_model = DummyClassifier(random_state= random_state, strategy= random_model_strategy).fit(train_df[list_feature_cols], train_df[str_actual_class])
        list_predicted_class_all_p.extend(random_model.predict(test_df[list_feature_cols]).tolist())
        list_actual_class_all_p.extend(test_df[str_actual_class].tolist())

    list_perform, list_columns = clf_report(true_labels= list_actual_class_all_p, predicted_val_list= list_predicted_class_all_p, bool_print=False, bool_return_columns = True)
    list_perform.append(len(list_predicted_class_all_p)); list_columns.append('# of EMA')

    pd.DataFrame(data=[list_perform], columns= list_columns).to_excel(os.path.join(loc_baseline_models_chi, 'baseline_random_'+random_model_strategy +'_'+ temp_dataset +'.xlsx'), index=False)

    print(list_perform)

for __ in ['uniform', 'most_frequent']:
    random_baselines_performance(random_model_strategy = __, loc_perf_file= '... removed the location a bit for privacy .../Documents/SA39/SAPIENS/Daily Anxiety/on_the_same_day/multi_modal/splitted_into3/Tiles18_multi_modal_multi_modal/Performance/ResNet_18/predicted_proba_fine_tuned_ResNet.xlsx') # uniform, most_frequent

In [ ]:
def publication_ready_baseline_models_tiles_18():
    df_base_line_tiles = pd.DataFrame()

    for base_file in sorted(os.listdir(loc_baseline_models_chi)):
        print(base_file)
        if ('baseline' in base_file) and (str_tiles_18 in base_file):
            temp_df_perf = pd.read_excel(os.path.join(loc_baseline_models_chi, base_file))
            temp_df_perf[str_filename] = np.repeat(base_file, temp_df_perf.shape[0])
            df_base_line_tiles = pd.concat([df_base_line_tiles, temp_df_perf])
        
    df_base_line_tiles.rename(columns={str_balanced_acc: 'BA', 'recall': str_sensitivity, str_speicificity.lower(): str_speicificity, 
                                      'precision': str_precision, 'F1_score': 'F1', 'accuracy': str_accuracy, 
                                       'window_folder': str_window, 'multi_mode_comb': str_modalities,
                                       'n_closest_instances': str_n_closest_sample, 'model_name': str_algorithm}, inplace=True)
    print(df_to_full_latex_table(df_base_line_tiles,))

publication_ready_baseline_models_tiles_18()

## SAPIENS

In [ ]:
for __ in ['uniform', 'most_frequent']:
    random_baselines_performance(random_model_strategy = __, loc_perf_file= '... removed the location a bit for privacy .../Documents/SA39/SAPIENS/State Anxiety/2.5_hours_1_vs_2_10_split/multi_modal/splitted_into3/SAPIENS_HeartRate_Light__accel___magnet_/Performance/ResNet_18/iter_130__a9dfbe4d-4076-48c7-a72b-342fe4c12514__chunk_test_5_val_2__fine_tuned_all_layers/predicted_proba_fine_tuned_ResNet.xlsx') # uniform, most_frequent



def publication_ready_baseline_models_SAPIENS():
    df_base_line_tiles = pd.DataFrame()

    for base_file in sorted(os.listdir(loc_baseline_models_chi)):
        if ('baseline' in base_file) and (str_sapiens in base_file):
            temp_df_perf = pd.read_excel(os.path.join(loc_baseline_models_chi, base_file))
            temp_df_perf[str_filename] = np.repeat(base_file, temp_df_perf.shape[0])
            df_base_line_tiles = pd.concat([df_base_line_tiles, temp_df_perf])
        
    df_base_line_tiles.rename(columns={str_balanced_acc: 'BA', 'recall': str_sensitivity, str_speicificity.lower(): str_speicificity, 
                                      'precision': str_precision, 'F1_score': 'F1', 'accuracy': str_accuracy, 
                                       'window_folder': str_window, 'multi_mode_comb': str_modalities,
                                       'n_closest_instances': str_n_closest_sample, 'model_name': str_algorithm}, inplace=True)
    print(df_to_full_latex_table(df_base_line_tiles,))

publication_ready_baseline_models_SAPIENS()

## Newest Model

# Probes per hour Viz

In [ ]:
import os
import glob
import math
from typing import Dict, List, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams["font.family"] = "Times New Roman"

def _read_df(pkl_path: str) -> pd.DataFrame:
    df = pd.read_pickle(pkl_path)
    if "T" not in df.columns:
        raise ValueError(f"'T' column not found in {pkl_path}")
    df = df.copy()
    df["T"] = pd.to_datetime(df["T"])
    if df["T"].isna().all():
        raise ValueError(f"Could not parse any timestamps in 'T' for {pkl_path}")
    return df.sort_values("T").reset_index(drop=True)

#  ['#FDE0D5', '#F5AE9D', '#df7863', '#CC4D38', '#ac2610', '#6d1002', '#FDE0D5', '#F5AE9D', '#778899', '#696969', '#2F4F4F']
# # # def _nearest_within_tolerance(expected: pd.Series, observed: pd.Series, tol_minutes: int) -> pd.Series:
# # #     """
# # #     Return a boolean Series (aligned with 'expected') that is True when there's at least
# # #     one observed timestamp within ±tol_minutes of each expected time.
# # #     Uses merge_asof with a dummy column to detect matches robustly.
# # #     """
# # #     left = pd.DataFrame({"T": expected.sort_values().values, "_idx": np.arange(len(expected))})
# # #     right = pd.DataFrame({"T": observed.sort_values().values})
# # #     right["_hit"] = 1  # dummy flag to indicate a match if merged
# # #     tol = pd.Timedelta(minutes=tol_minutes)

# # #     try:
# # #         merged = pd.merge_asof(left, right, on="T", direction="nearest", tolerance=tol)
# # #     except Exception as e:
# # #         print(e)
# # #         print(left['T'].tolist())
# # #         print(right['T'].tolist())
# # #     matched = merged["_hit"].notna()  # True when a right-side row matched within tolerance
# # #     out = pd.Series(False, index=np.arange(len(expected)))
# # #     out.loc[left["_idx"].values] = matched.values
# # #     return out


def has_any_within_forward_window(expected: pd.Series, observed: pd.Series, tol_minutes: int) -> pd.Series:
    # Ensure datetime and sorted (merge_asof requires sorted keys)
    exp = pd.to_datetime(expected).sort_values()
    obs = pd.to_datetime(observed).sort_values()

    left = pd.DataFrame({"T": exp.values, "_idx": np.arange(len(exp))})
    # Keep the observed timestamp in a separate column so we can compute the delta
    right = pd.DataFrame({
        "T": obs.values,          # key for merge_asof
        "T_obs": obs.values,      # preserved observed time
    })

    tol = pd.Timedelta(minutes=tol_minutes)

    merged = pd.merge_asof(left, right, on="T", direction="forward")

    # print(merged)

    matched = merged["T_obs"].notna() & ((merged["T_obs"] - merged["T"]) <= tol)

    out = pd.Series(False, index=np.arange(len(expected)))
    out.loc[left["_idx"].values] = matched.values
    return out




def analyze_probes(base_dir: str, tol_minutes: int = 1, show_table: bool = False) -> Tuple[pd.DataFrame, pd.DataFrame]:
    
    pids = [p for p in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, p))]
    if len(pids) == 0:
        raise FileNotFoundError(f"No participant folders found under: {base_dir}")

    hourly_records: List[Dict] = []

    for pid in [413, 414, 415, 416, 417, 418, 420, 421, 423, 424, 428, 429, 430, 431, 432, 433, 434, 436, 437, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 480, 481, 482, 483, 485, 486, 487, 488, 490, 491, 492, 493, 494, 497, 498]:
        pid = 'P'+str(pid)
        # if pid != 'P413':
        #     continue

        pid_dir = os.path.join(base_dir, pid)
        expected_path = os.path.join(pid_dir, "Smartwatch_SensorsStartingTimeDatum.pkl")
        if not os.path.exists(expected_path):
            continue

        try:
            df_expect = _read_df(expected_path)
        except Exception as e:
            print(f"[WARN] Skipping {pid} — could not read expected file: {e}")
            continue

        df_expect["_hour"] = df_expect["T"].dt.floor("H")

        sensor_pkls = [
            p for p in glob.glob(os.path.join(pid_dir, "Smartwatch_*Datum.pkl"))
            # if not p.endswith("Smartwatch_SensorsStartingTimeDatum.pkl")
        ]

        for pkl in sensor_pkls:
            if os.path.basename(pkl) not in [ 'Smartwatch_LinearAccelerationDatum.pkl', 'Smartwatch_SensorsStartingTimeDatum.pkl', 'Smartwatch_LightDatum.pkl', 'Smartwatch_HeartRateDatum.pkl', 'Smartwatch_GravityDatum.pkl', 'Smartwatch_MagnetometerDatum.pkl', 'Smartwatch_StepCountDatum.pkl']:
                continue

            print('\n\n File', pkl)
            sensor_name = os.path.basename(pkl).replace("Smartwatch_", "").replace("Datum.pkl", "").replace('Linear', '').replace('SensorsStartingTime', 'SensorsStart').replace('StepCount', 'Step').replace('Magnetometer', 'Magnet.').replace('Acceleration', 'Acc.')

            try:
                df_obs = _read_df(pkl)
                df_obs = df_obs.dropna(subset=["T"])
            except Exception as e:
                print(f"[WARN] {pid}/{sensor_name} — could not read: {e}")
                continue

            success_mask = has_any_within_forward_window(expected=df_expect["T"], observed=df_obs["T"], tol_minutes=tol_minutes)
            df_exp_copy = df_expect[["_hour"]].copy()
            df_exp_copy["success"] = success_mask.astype(int)

            per_hour = df_exp_copy.groupby("_hour")["success"].sum().reset_index()
            per_hour["PID"] = pid
            per_hour["Sensor"] = sensor_name
            per_hour.rename(columns={"_hour": "Hour", "success": "ProbesPerHour"}, inplace=True)
            # print(per_hour)
            hourly_records.append(per_hour)
        # break

    if len(hourly_records) == 0:
        raise RuntimeError("No hourly records were created. Check directory structure and file contents.")

    hourly_long = pd.concat(hourly_records, ignore_index=True)

    sensor_stats = hourly_long.groupby("Sensor")["ProbesPerHour"].agg(["mean", "std", "count", "sum"]).reset_index()
    sensor_stats["ci95"] = stats.t.ppf(0.975, sensor_stats["count"] - 1) * sensor_stats["std"] / np.sqrt(sensor_stats["count"])

    summary_ci = sensor_stats[["Sensor", "mean", "sum", "ci95", "count"]].rename(columns={"mean": "MeanProbesPerHour", "count": "N_pid_hours"})

    summary_ci["Sensor"] = summary_ci["Sensor"].astype(str)
    has_expected = "SensorsStart" in set(summary_ci["Sensor"])
    if has_expected:
        order = [s for s in summary_ci["Sensor"] if s != "SensorsStart"] + ["SensorsStart"]
        summary_ci = summary_ci.set_index("Sensor").loc[order].reset_index()

    if "SensorsStart" in summary_ci["Sensor"].values:
        summary_ci.loc[summary_ci["Sensor"] == "SensorsStart", "ci95"] = 0.0

    plt.figure(figsize=(9, 5))
    colors = [None] * len(summary_ci)  # default colors from Matplotlib

    bars = plt.bar( summary_ci["Sensor"], summary_ci["MeanProbesPerHour"], yerr=summary_ci["ci95"], color=['#C5D4DE'], capsize=5, alpha=0.8,)
    
    for __bar, __val in zip(bars, summary_ci["MeanProbesPerHour"]):
        plt.text(
            __bar.get_x() + __bar.get_width() / 2,
            __bar.get_height() /2,  
            f"{__val:.2f}",
            ha="center", va="bottom",
            fontsize=20,
            color="black"
        )



    if "SensorsStart" in summary_ci["Sensor"].values:
        idx = summary_ci.index[summary_ci["Sensor"] == "SensorsStart"][0]
        bars[idx].set_color("#CC4D38")  # or a hex like "#d62728"



    plt.ylabel("Mean # of probe starts per hour", fontsize=18)
    plt.title("")
    plt.yticks(fontsize=18)
    plt.xticks(fontsize=18, rotation=15, ha="right")
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.grid(False)
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)

    plt.ylim(0, 9)
    plt.show()

    print(summary_ci)


analyze_probes('... removed the location a bit for privacy .../Documents/SA39/SAPIENS/All Data Spring\'24/Data of Spring\'24', show_table=False)

In [ ]:
analyze_probes('... removed the location a bit for privacy .../Documents/SA39/SAPIENS/All Data Spring\'24/AppTestingSpring24/Processed Files/Problematic version', show_table=False)

# Operation Theater